# Success

# success 1 - job gain job loss by industry
The stacked chart uses Company file only, with these raw columns:
1. Company.year = the Y-axis (years 2020 to 2025)
2. Company.industry_name = the stacked bar groups / multiple colors
3. Company.metric_name = used to decide whether a row is job gain or job loss
4. Company.value = the numeric value used in the chart

It does NOT use:
- degree_count
- field_count
- startup_count
- closed_business_count

Formula / logic:
- job_gain = Company.value when Company.metric_name contains words like:
  "job gain", "jobs gained", "job creation", "jobs created", "employment gain", "hires", "hiring", "job opening", "job openings"

- job_loss = Company.value when Company.metric_name contains words like:
  "job loss", "jobs lost", "job destruction", "employment loss", "layoffs", "separations"

Grouping:
For each Year + Industry:
- total_job_gain = sum(job_gain)
- total_job_loss = sum(job_loss)

Chart display:
- left side = -total_job_loss
- right side = total_job_gain
- each color = one industry
- each row = one year

So the chart means:
A 2020–2025 job chart by year, stacked by industry color, with job loss on the left and job gain on the right, using raw Company.value data.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# SETTINGS OBJECT
# Change only this part later
# ============================================================

CFG = {
    "COMPANY_FILE": Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv",

    "OUTPUT_DIR": Path.home() / "Downloads" / "pre_formula_job_charts",

    "CHUNKSIZE": 100_000,

    "START_YEAR": 1970,
    "END_YEAR": 2025,

    # None = keep all industries
    # Example: 10 or 15 = show top industries, combine rest as Other
    "TOP_N": None,

    "COMPANY_COLS": [
        "year",
        "industry_name",
        "metric_name",
        "value"
    ],

    # Exclude rate / percent rows
    "RATE_WORDS": "rate|percent|percentage|pct|share|ratio",

    # Exclude startup / business close rows
    "STARTUP_COMPANY_WORDS": (
        "startup|start-up|business|firm|company|establishment|"
        "birth|death|closed|closure|exit|entry|formation"
    ),

    # Job gain metric words
    "JOB_GAIN_WORDS": (
        "job gain|job gains|jobs gained|job creation|jobs created|"
        "employment gain|employment gains|hire|hires|hiring|"
        "job opening|job openings"
    ),

    # Job loss metric words
    "JOB_LOSS_WORDS": (
        "job loss|job losses|jobs lost|job destruction|"
        "employment loss|employment losses|layoff|layoffs|"
        "separation|separations"
    ),

    "FIGSIZE": (16, 9),
    "DPI": 300,
    "COLOR_MAP": "tab20",

    "SAVE_FILE_NAME": "chart_{start}_{end}_stacked_industry_left_loss_right_gain_by_year.png"
}


# ============================================================
# VARIABLES FROM SETTINGS OBJECT
# Do not need to change below this part much
# ============================================================

COMPANY_FILE = CFG["COMPANY_FILE"]
OUTPUT_DIR = CFG["OUTPUT_DIR"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNKSIZE = CFG["CHUNKSIZE"]
START_YEAR = CFG["START_YEAR"]
END_YEAR = CFG["END_YEAR"]
TOP_N = CFG["TOP_N"]
COMPANY_COLS = CFG["COMPANY_COLS"]


# ============================================================
# CHECK FILE
# ============================================================

print("Checking Company file...")

if not COMPANY_FILE.exists():
    raise FileNotFoundError(f"Missing Company file: {COMPANY_FILE}")

print("Finished checking file.")
print("Company file:", COMPANY_FILE)


# ============================================================
# LOAD RAW COMPANY DATA
# ============================================================

print(f"\nLoading raw Company job data from {START_YEAR} to {END_YEAR}...")

parts = []
chunk_count = 0

matched_job_gain_metric_names = set()
matched_job_loss_metric_names = set()

for chunk in pd.read_csv(
    COMPANY_FILE,
    usecols=COMPANY_COLS,
    chunksize=CHUNKSIZE,
    low_memory=False
):
    chunk_count += 1
    print(f"Loading chunk {chunk_count}... rows: {len(chunk):,}")

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")

    chunk = chunk.dropna(subset=["year", "industry_name", "metric_name", "value"])
    chunk["year"] = chunk["year"].astype(int)

    chunk = chunk[
        (chunk["year"] >= START_YEAR) &
        (chunk["year"] <= END_YEAR)
    ].copy()

    if chunk.empty:
        continue

    metric = chunk["metric_name"].astype(str).str.lower()

    # Exclude rate / percent / share rows
    not_rate_mask = ~metric.str.contains(
        CFG["RATE_WORDS"],
        na=False,
        regex=True
    )

    # Exclude startup / business / company close rows
    not_startup_company_mask = ~metric.str.contains(
        CFG["STARTUP_COMPANY_WORDS"],
        na=False,
        regex=True
    )

    # JOB GAIN ONLY
    job_gain_mask = metric.str.contains(
        CFG["JOB_GAIN_WORDS"],
        na=False,
        regex=True
    ) & not_rate_mask & not_startup_company_mask

    # JOB LOSS ONLY
    job_loss_mask = metric.str.contains(
        CFG["JOB_LOSS_WORDS"],
        na=False,
        regex=True
    ) & not_rate_mask & not_startup_company_mask

    matched_job_gain_metric_names.update(
        chunk.loc[job_gain_mask, "metric_name"].dropna().unique()
    )

    matched_job_loss_metric_names.update(
        chunk.loc[job_loss_mask, "metric_name"].dropna().unique()
    )

    chunk["job_gain"] = np.where(job_gain_mask, chunk["value"], 0)
    chunk["job_loss"] = np.where(job_loss_mask, chunk["value"], 0)

    part = (
        chunk
        .groupby(["year", "industry_name"], dropna=False)[["job_gain", "job_loss"]]
        .sum()
        .reset_index()
    )

    parts.append(part)

print("Finished loading data.")


# ============================================================
# COMBINE DATA
# ============================================================

if not parts:
    raise ValueError(f"No Company job data found from {START_YEAR} to {END_YEAR}.")

chart_data = pd.concat(parts, ignore_index=True)

chart_data = (
    chart_data
    .groupby(["year", "industry_name"], dropna=False)[["job_gain", "job_loss"]]
    .sum()
    .reset_index()
)

chart_data["job_total"] = chart_data["job_gain"] + chart_data["job_loss"]
chart_data = chart_data[chart_data["job_total"] > 0].copy()

if chart_data.empty:
    raise ValueError(
        f"Data loaded, but no job gain/loss rows matched from {START_YEAR} to {END_YEAR}."
    )

print("\nMetric names used for job_gain:")
print(sorted(matched_job_gain_metric_names))

print("\nMetric names used for job_loss:")
print(sorted(matched_job_loss_metric_names))


# ============================================================
# OPTIONAL: KEEP TOP INDUSTRIES ONLY
# None = keep all industries
# ============================================================

if TOP_N is not None:
    top_industries = (
        chart_data.groupby("industry_name")["job_total"]
        .sum()
        .sort_values(ascending=False)
        .head(TOP_N)
        .index
    )

    chart_data["industry_name"] = np.where(
        chart_data["industry_name"].isin(top_industries),
        chart_data["industry_name"],
        "Other"
    )

    chart_data = (
        chart_data
        .groupby(["year", "industry_name"], dropna=False)[["job_gain", "job_loss"]]
        .sum()
        .reset_index()
    )


# ============================================================
# PIVOT FOR STACKED BAR
# ============================================================

gain_pivot = (
    chart_data
    .pivot_table(
        index="year",
        columns="industry_name",
        values="job_gain",
        aggfunc="sum",
        fill_value=0
    )
    .sort_index()
)

loss_pivot = (
    chart_data
    .pivot_table(
        index="year",
        columns="industry_name",
        values="job_loss",
        aggfunc="sum",
        fill_value=0
    )
    .sort_index()
)

# Make sure both have same industry columns
all_industries = sorted(set(gain_pivot.columns).union(set(loss_pivot.columns)))

gain_pivot = gain_pivot.reindex(columns=all_industries, fill_value=0)
loss_pivot = loss_pivot.reindex(columns=all_industries, fill_value=0)

print("\nYears used:")
print(gain_pivot.index.tolist())

print("\nIndustries used:")
print(all_industries)


# ============================================================
# PLOT DIVERGING STACKED HORIZONTAL BAR CHART
# Y = YEAR
# COLORS = INDUSTRY
# LEFT = JOB LOSS
# RIGHT = JOB GAIN
# JOB ONLY, NOT STARTUP / COMPANY CLOSE
# ============================================================

plt.figure(figsize=CFG["FIGSIZE"])

years = gain_pivot.index.astype(str)
y_pos = np.arange(len(years))

# One color per industry
colors = getattr(plt.cm, CFG["COLOR_MAP"])(
    np.linspace(0, 1, len(all_industries))
)

# RIGHT SIDE = job gain
right_base = np.zeros(len(gain_pivot))

for i, industry in enumerate(all_industries):
    values = gain_pivot[industry].values

    plt.barh(
        y_pos,
        values,
        left=right_base,
        label=industry,
        color=colors[i]
    )

    right_base += values


# LEFT SIDE = job loss
left_base = np.zeros(len(loss_pivot))

for i, industry in enumerate(all_industries):
    values = loss_pivot[industry].values

    plt.barh(
        y_pos,
        -values,
        left=-left_base,
        color=colors[i]
    )

    left_base += values


plt.axvline(0, color="black", linewidth=1)

plt.yticks(y_pos, years)

plt.title(
    f"{START_YEAR}-{END_YEAR} Stacked Industry Job Loss vs Job Gain by Year"
)

plt.xlabel("Job loss on left   |   Job gain on right")
plt.ylabel("Year")

plt.grid(axis="x", linestyle="--", alpha=0.4)

plt.legend(
    title="Industry",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()

save_path = OUTPUT_DIR / CFG["SAVE_FILE_NAME"].format(
    start=START_YEAR,
    end=END_YEAR
)

plt.savefig(
    save_path,
    dpi=CFG["DPI"],
    bbox_inches="tight"
)

plt.close()


# ============================================================
# DONE
# ============================================================

print("\nALL DONE.")
print("Saved chart here:")
print(save_path)

# Success 2 - Startup & death Company

FOR STARTUP + COMPANY DEATH CHART

File used:
Company file only

Columns used:
1. Company.year
   - Used for the chart year
   - Example: 2020, 2021, 2022, 2023, 2024, 2025

2. Company.industry_name
   - Used for the industry
   - This becomes the stacked bar colors
   - Example: Finance, Education Services, Manufacturing, Health Care, etc.

3. Company.metric_name
   - Used to decide what type of row it is
   - This column tells if the row is startup, business birth, company death, closure, etc.

4. Company.value
   - Used as the number/count
   - This is the value added into startup_count or death_count


Formula:

startup_count =
Company.value
WHERE Company.metric_name contains:
startup
start-up
birth
births
formation
formations
entry
entries
new business
new firm
new company
opening
openings


death_count =
Company.value
WHERE Company.metric_name contains:
death
deaths
closed
closure
closures
closing
closings
exit
exits
failure
failures


Group by:
Company.year
Company.industry_name


Chart meaning:

Y-axis = year

Color = industry_name

Left side = death_count
This means company death / closed business / business exit

Right side = startup_count
This means startup / new business / business birth / new company


Not used:
Degree file is not used
Job gain is not used
Job loss is not used
Degree count is not used
Field of study is not used
metric_code is not used in this version


Main formula summary:

startup_count = sum(value) for startup/birth/formation/entry metric_name

death_count = sum(value) for death/closed/closure/exit metric_name

Then group the result by year and industry_name

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# SETTINGS OBJECT
# Change only this part later
# ============================================================

CFG = {
    "COMPANY_FILE": Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv",

    "OUTPUT_DIR": Path.home() / "Downloads" / "pre_formula_startup_death_charts",

    "CHUNKSIZE": 100_000,

    "START_YEAR": 1970,
    "END_YEAR": 2025,

    # None = keep all industries
    # Example: 10 or 15 = show top industries, combine rest as Other
    "TOP_N": None,

    "COMPANY_COLS": [
        "year",
        "industry_name",
        "metric_name",
        "value"
    ],

    # Exclude rate / percent rows
    "RATE_WORDS": "rate|percent|percentage|pct|share|ratio",

    # Exclude job rows
    # This chart is startup/death only, NOT job gain/loss
    "JOB_WORDS": (
        "job|jobs|employment|hire|hires|hiring|"
        "layoff|layoffs|separation|separations"
    ),

    # Startup / new business metric words
    "STARTUP_WORDS": (
        "startup|start-up|birth|births|formation|formations|"
        "entry|entries|new business|new firm|new company|"
        "opening|openings"
    ),

    # Company death / closed business metric words
    "DEATH_WORDS": (
        "death|deaths|closed|closure|closures|closing|closings|"
        "exit|exits|failure|failures"
    ),

    "FIGSIZE": (16, 9),
    "DPI": 300,
    "COLOR_MAP": "tab20",

    "SAVE_FILE_NAME": "chart_{start}_{end}_stacked_industry_left_death_right_startup_by_year.png",

    # How many rows to print in the check table
    "CHECK_TABLE_ROWS": 50
}


# ============================================================
# VARIABLES FROM SETTINGS OBJECT
# Do not need to change below this part much
# ============================================================

COMPANY_FILE = CFG["COMPANY_FILE"]
OUTPUT_DIR = CFG["OUTPUT_DIR"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNKSIZE = CFG["CHUNKSIZE"]
START_YEAR = CFG["START_YEAR"]
END_YEAR = CFG["END_YEAR"]
TOP_N = CFG["TOP_N"]
COMPANY_COLS = CFG["COMPANY_COLS"]


# ============================================================
# CHECK FILE
# ============================================================

print("Checking Company file...")

if not COMPANY_FILE.exists():
    raise FileNotFoundError(f"Missing Company file: {COMPANY_FILE}")

print("Finished checking file.")
print("Company file:", COMPANY_FILE)


# ============================================================
# LOAD RAW COMPANY STARTUP / DEATH DATA
# ============================================================

print(f"\nLoading raw Company startup/death data from {START_YEAR} to {END_YEAR}...")

parts = []
chunk_count = 0

matched_startup_metric_names = set()
matched_death_metric_names = set()

for chunk in pd.read_csv(
    COMPANY_FILE,
    usecols=COMPANY_COLS,
    chunksize=CHUNKSIZE,
    low_memory=False
):
    chunk_count += 1
    print(f"Loading chunk {chunk_count}... rows: {len(chunk):,}")

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")

    chunk = chunk.dropna(subset=["year", "industry_name", "metric_name", "value"])
    chunk["year"] = chunk["year"].astype(int)

    chunk = chunk[
        (chunk["year"] >= START_YEAR) &
        (chunk["year"] <= END_YEAR)
    ].copy()

    if chunk.empty:
        continue

    metric = chunk["metric_name"].astype(str).str.lower()

    # Exclude rate / percent / share rows
    not_rate_mask = ~metric.str.contains(
        CFG["RATE_WORDS"],
        na=False,
        regex=True
    )

    # Exclude job rows
    not_job_mask = ~metric.str.contains(
        CFG["JOB_WORDS"],
        na=False,
        regex=True
    )

    # STARTUP / NEW BUSINESS / NEW COMPANY
    startup_mask = metric.str.contains(
        CFG["STARTUP_WORDS"],
        na=False,
        regex=True
    ) & not_rate_mask & not_job_mask

    # COMPANY DEATH / CLOSED BUSINESS
    death_mask = metric.str.contains(
        CFG["DEATH_WORDS"],
        na=False,
        regex=True
    ) & not_rate_mask & not_job_mask

    matched_startup_metric_names.update(
        chunk.loc[startup_mask, "metric_name"].dropna().unique()
    )

    matched_death_metric_names.update(
        chunk.loc[death_mask, "metric_name"].dropna().unique()
    )

    chunk["startup_count"] = np.where(startup_mask, chunk["value"], 0)
    chunk["death_count"] = np.where(death_mask, chunk["value"], 0)

    part = (
        chunk
        .groupby(["year", "industry_name"], dropna=False)[["startup_count", "death_count"]]
        .sum()
        .reset_index()
    )

    parts.append(part)

print("Finished loading data.")


# ============================================================
# COMBINE DATA
# ============================================================

if not parts:
    raise ValueError(f"No Company startup/death data found from {START_YEAR} to {END_YEAR}.")

chart_data = pd.concat(parts, ignore_index=True)

chart_data = (
    chart_data
    .groupby(["year", "industry_name"], dropna=False)[["startup_count", "death_count"]]
    .sum()
    .reset_index()
)

chart_data["company_total"] = chart_data["startup_count"] + chart_data["death_count"]
chart_data = chart_data[chart_data["company_total"] > 0].copy()

if chart_data.empty:
    raise ValueError(
        f"Data loaded, but no startup/death rows matched from {START_YEAR} to {END_YEAR}."
    )

print("\nMetric names used for startup_count:")
print(sorted(matched_startup_metric_names))

print("\nMetric names used for death_count:")
print(sorted(matched_death_metric_names))


# ============================================================
# OPTIONAL: KEEP TOP INDUSTRIES ONLY
# None = keep all industries
# ============================================================

if TOP_N is not None:
    top_industries = (
        chart_data.groupby("industry_name")["company_total"]
        .sum()
        .sort_values(ascending=False)
        .head(TOP_N)
        .index
    )

    chart_data["industry_name"] = np.where(
        chart_data["industry_name"].isin(top_industries),
        chart_data["industry_name"],
        "Other"
    )

    chart_data = (
        chart_data
        .groupby(["year", "industry_name"], dropna=False)[["startup_count", "death_count"]]
        .sum()
        .reset_index()
    )


# ============================================================
# PIVOT FOR STACKED BAR
# ============================================================

startup_pivot = (
    chart_data
    .pivot_table(
        index="year",
        columns="industry_name",
        values="startup_count",
        aggfunc="sum",
        fill_value=0
    )
    .sort_index()
)

death_pivot = (
    chart_data
    .pivot_table(
        index="year",
        columns="industry_name",
        values="death_count",
        aggfunc="sum",
        fill_value=0
    )
    .sort_index()
)

# Make sure both have same industry columns
all_industries = sorted(set(startup_pivot.columns).union(set(death_pivot.columns)))

startup_pivot = startup_pivot.reindex(columns=all_industries, fill_value=0)
death_pivot = death_pivot.reindex(columns=all_industries, fill_value=0)

print("\nYears used:")
print(startup_pivot.index.tolist())

print("\nIndustries used:")
print(all_industries)


# ============================================================
# PLOT DIVERGING STACKED HORIZONTAL BAR CHART
# Y = YEAR
# COLORS = INDUSTRY
# LEFT = COMPANY DEATH / CLOSED BUSINESS
# RIGHT = STARTUP / NEW BUSINESS
# COMPANY STARTUP + DEATH ONLY
# NOT JOB GAIN / JOB LOSS
# ============================================================

plt.figure(figsize=CFG["FIGSIZE"])

years = startup_pivot.index.astype(str)
y_pos = np.arange(len(years))

# One color per industry
colors = getattr(plt.cm, CFG["COLOR_MAP"])(
    np.linspace(0, 1, len(all_industries))
)

# RIGHT SIDE = startup / new business
right_base = np.zeros(len(startup_pivot))

for i, industry in enumerate(all_industries):
    values = startup_pivot[industry].values

    plt.barh(
        y_pos,
        values,
        left=right_base,
        label=industry,
        color=colors[i]
    )

    right_base += values


# LEFT SIDE = death / closed business
left_base = np.zeros(len(death_pivot))

for i, industry in enumerate(all_industries):
    values = death_pivot[industry].values

    plt.barh(
        y_pos,
        -values,
        left=-left_base,
        color=colors[i]
    )

    left_base += values


plt.axvline(0, color="black", linewidth=1)

plt.yticks(y_pos, years)

plt.title(
    f"{START_YEAR}-{END_YEAR} Stacked Industry Company Death vs Startup by Year"
)

plt.xlabel("Company death / closed business on left   |   Startup / new business on right")
plt.ylabel("Year")

plt.grid(axis="x", linestyle="--", alpha=0.4)

plt.legend(
    title="Industry",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()

save_path = OUTPUT_DIR / CFG["SAVE_FILE_NAME"].format(
    start=START_YEAR,
    end=END_YEAR
)

plt.savefig(
    save_path,
    dpi=CFG["DPI"],
    bbox_inches="tight"
)

plt.close()


# ============================================================
# PRINT SMALL CHECK TABLE
# ============================================================

check_table = (
    chart_data
    .groupby(["year", "industry_name"], dropna=False)[["startup_count", "death_count"]]
    .sum()
    .reset_index()
    .sort_values(["year", "industry_name"])
)

print("\nPre-formula check table:")
print(check_table.head(CFG["CHECK_TABLE_ROWS"]))


# ============================================================
# DONE
# ============================================================

print("\nALL DONE.")
print("Saved chart here:")
print(save_path)

# Success 4 - degree Half
# stacked bar chart
| Code purpose        | Column name        | Meaning                                                  |
| ------------------- | ------------------ | -------------------------------------------------------- |
| Main year column    | `year`             | Groups data by year                                      |
| Number/value column | `degree_count`     | The degree amount/count being added                      |
| Category 1          | `major_name`       | Major name                                               |
| Category 2          | `field_group`      | Big field group                                          |
| Category 3          | `field_subgroup`   | Smaller field group                                      |
| Category 4          | `degree_group`     | Degree type/group                                        |
| Category 5          | `award_level_name` | Award level, like certificate, associate, bachelor, etc. |
| Category 6          | `cip_code`         | CIP code for the major/field                             |


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# ============================================================
# DEGREE COUNT BY YEAR + ALL CATEGORIES
# - NO TOP N
# - NO OTHER BUCKET
# - REMOVE UNKNOWN / BLANK / NAN
# - PRINT TABLE
# - SAVE FULL CSV
# - SAVE IMAGE
# ============================================================

CFG = {
    "DEGREE_FILE": Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv",
    "OUTPUT_DIR": Path.home() / "Downloads" / "degree_count_all_table_and_image",

    "START_YEAR": 1970,
    "END_YEAR": 2025,   # 2025 will be 0 if file stops at 2024

    "MAIN_COLUMN": "year",
    "VALUE_COLUMN": "degree_count",

    "OTHER_COLUMNS": [
        "major_name",
        "field_group",
        "field_subgroup",
        "degree_group",
        "award_level_name",
        "cip_code",
    ],

    "NEEDED_COLUMNS": [
        "year",
        "major_name",
        "field_group",
        "field_subgroup",
        "degree_count",
        "degree_group",
        "award_level_name",
        "cip_code",
    ],

    "CHUNKSIZE": 100_000,

    # print
    "PRINT_ALL_ROWS": True,   # True = print full table
    "PRINT_MAX_ROWS": None,   # None = unlimited

    # chart
    "FIGSIZE": (20, 10),
    "DPI": 200,
    "SHOW_LEGEND": True,      # can be too large for big columns
    "LEGEND_FONT_SIZE": 8,
}

CFG["OUTPUT_DIR"].mkdir(parents=True, exist_ok=True)

try:
    from IPython.display import display
except ImportError:
    display = print

# ============================================================
# PANDAS DISPLAY
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 300)

if CFG["PRINT_ALL_ROWS"]:
    pd.set_option("display.max_rows", CFG["PRINT_MAX_ROWS"])
else:
    pd.set_option("display.max_rows", 100)

# ============================================================
# LOAD DATA
# ============================================================

print("Reading degree file...")

all_chunks = []

for i, chunk in enumerate(
    pd.read_csv(
        CFG["DEGREE_FILE"],
        usecols=CFG["NEEDED_COLUMNS"],
        chunksize=CFG["CHUNKSIZE"],
        low_memory=False
    ),
    start=1
):
    print(f"Reading chunk {i}...")

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["degree_count"] = pd.to_numeric(chunk["degree_count"], errors="coerce").fillna(0)

    chunk = chunk[
        (chunk["year"] >= CFG["START_YEAR"]) &
        (chunk["year"] <= CFG["END_YEAR"])
    ].copy()

    if chunk.empty:
        continue

    chunk["year"] = chunk["year"].astype(int)
    all_chunks.append(chunk)

if not all_chunks:
    raise ValueError("No data found in selected year range.")

df = pd.concat(all_chunks, ignore_index=True)

print("Finished loading.")
print("Rows found:", len(df))
print("Year range found:", df["year"].min(), "-", df["year"].max())

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_col(series):
    """
    Convert to string, strip spaces, and standardize blanks/nan/none.
    """
    s = series.astype(str).str.strip()
    s = s.replace({
        "": "Unknown",
        "nan": "Unknown",
        "NaN": "Unknown",
        "None": "Unknown",
        "NULL": "Unknown",
        "null": "Unknown"
    })
    return s.fillna("Unknown")

def comma_format(x, pos):
    return f"{int(x):,}"

def make_table_and_image(col_name):
    print("\n" + "=" * 100)
    print(f"TABLE + IMAGE FOR: {col_name}")
    print("=" * 100)

    temp = df[["year", "degree_count", col_name]].copy()

    # clean text
    temp[col_name] = clean_col(temp[col_name])

    # remove Unknown / blank / nan
    temp = temp[
        ~temp[col_name].astype(str).str.strip().str.lower().isin(
            ["unknown", "nan", "none", ""]
        )
    ].copy()

    if temp.empty:
        print(f"No data left for {col_name} after removing Unknown/blank.")
        return

    # group
    grouped = (
        temp
        .groupby(["year", col_name], as_index=False)["degree_count"]
        .sum()
    )

    # pivot table for output
    table = grouped.pivot_table(
        index=col_name,
        columns="year",
        values="degree_count",
        aggfunc="sum",
        fill_value=0
    )

    years = list(range(CFG["START_YEAR"], CFG["END_YEAR"] + 1))

    for y in years:
        if y not in table.columns:
            table[y] = 0

    table = table[years]
    table[f"total_{CFG['START_YEAR']}_{CFG['END_YEAR']}"] = table[years].sum(axis=1)

    # sort biggest total first
    table = table.sort_values(
        by=f"total_{CFG['START_YEAR']}_{CFG['END_YEAR']}",
        ascending=False
    )

    table_reset = table.reset_index()

    # --------------------------------------------------------
    # PRINT TABLE
    # --------------------------------------------------------
    display(table_reset)

    # save csv
    csv_path = CFG["OUTPUT_DIR"] / f"{col_name}_degree_count_table_ALL.csv"
    table_reset.to_csv(csv_path, index=False)
    print("CSV saved:", csv_path)

    # --------------------------------------------------------
    # IMAGE
    # --------------------------------------------------------
    image_table = table.drop(columns=[f"total_{CFG['START_YEAR']}_{CFG['END_YEAR']}"])

    # transpose so x-axis = year
    plot_df = image_table.T

    fig, ax = plt.subplots(figsize=CFG["FIGSIZE"])

    plot_df.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        width=0.75
    )

    ax.set_title(f"Degree Count by Year and {col_name}", fontsize=20)
    ax.set_xlabel("Year", fontsize=14)
    ax.set_ylabel("Degree Count", fontsize=14)

    # remove scientific notation like 1e7
    ax.ticklabel_format(style="plain", axis="y", useOffset=False)
    ax.yaxis.set_major_formatter(FuncFormatter(comma_format))

    plt.xticks(rotation=45)

    # total on top of each year bar
    totals = plot_df.sum(axis=1)
    max_total = totals.max() if len(totals) > 0 else 0

    for x, total in enumerate(totals):
        if total > 0:
            ax.text(
                x,
                total + (max_total * 0.01 if max_total > 0 else 0),
                f"{int(total):,}",
                ha="center",
                va="bottom",
                fontsize=10,
                fontweight="bold"
            )

    # legend
    if CFG["SHOW_LEGEND"]:
        ax.legend(
            title=col_name,
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            fontsize=CFG["LEGEND_FONT_SIZE"]
        )
        plt.subplots_adjust(left=0.08, right=0.72, top=0.90, bottom=0.15)
    else:
        if ax.get_legend() is not None:
            ax.get_legend().remove()
        plt.subplots_adjust(left=0.08, right=0.95, top=0.90, bottom=0.15)

    image_path = CFG["OUTPUT_DIR"] / f"{col_name}_degree_count_image_ALL.png"
    plt.savefig(image_path, dpi=CFG["DPI"], bbox_inches="tight")
    plt.show()
    plt.close()

    print("Image saved:", image_path)

# ============================================================
# RUN
# ============================================================

for col in CFG["OTHER_COLUMNS"]:
    make_table_and_image(col)

print("\nDone.")
print("Saved in:", CFG["OUTPUT_DIR"])

# Export csv file

# 1

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# SETTINGS OBJECT
# Change only this part later
# ============================================================

CFG = {
    "COMPANY_FILE": Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv",

    "OUTPUT_DIR": Path.home() / "Downloads" / "pre_formula_job_csv",

    "CHUNKSIZE": 100_000,

    "START_YEAR": 1970,
    "END_YEAR": 2025,

    # None = keep all industries
    # Example: 10 or 15 = show top industries, combine rest as Other
    "TOP_N": None,

    "COMPANY_COLS": [
        "year",
        "industry_name",
        "metric_name",
        "value"
    ],

    # Exclude rate / percent rows
    "RATE_WORDS": "rate|percent|percentage|pct|share|ratio",

    # Exclude startup / business close rows
    "STARTUP_COMPANY_WORDS": (
        "startup|start-up|business|firm|company|establishment|"
        "birth|death|closed|closure|exit|entry|formation"
    ),

    # Job gain metric words
    "JOB_GAIN_WORDS": (
        "job gain|job gains|jobs gained|job creation|jobs created|"
        "employment gain|employment gains|hire|hires|hiring|"
        "job opening|job openings"
    ),

    # Job loss metric words
    "JOB_LOSS_WORDS": (
        "job loss|job losses|jobs lost|job destruction|"
        "employment loss|employment losses|layoff|layoffs|"
        "separation|separations"
    ),

    "SAVE_FILE_NAME": "job_gain_loss_data_{start}_{end}.csv"
}


# ============================================================
# VARIABLES FROM SETTINGS OBJECT
# ============================================================

COMPANY_FILE = CFG["COMPANY_FILE"]
OUTPUT_DIR = CFG["OUTPUT_DIR"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNKSIZE = CFG["CHUNKSIZE"]
START_YEAR = CFG["START_YEAR"]
END_YEAR = CFG["END_YEAR"]
TOP_N = CFG["TOP_N"]
COMPANY_COLS = CFG["COMPANY_COLS"]


# ============================================================
# CHECK FILE
# ============================================================

print("Checking Company file...")

if not COMPANY_FILE.exists():
    raise FileNotFoundError(f"Missing Company file: {COMPANY_FILE}")

print("Finished checking file.")
print("Company file:", COMPANY_FILE)


# ============================================================
# LOAD RAW COMPANY DATA
# ============================================================

print(f"\nLoading raw Company job data from {START_YEAR} to {END_YEAR}...")

parts = []
chunk_count = 0

matched_job_gain_metric_names = set()
matched_job_loss_metric_names = set()

for chunk in pd.read_csv(
    COMPANY_FILE,
    usecols=COMPANY_COLS,
    chunksize=CHUNKSIZE,
    low_memory=False
):
    chunk_count += 1
    print(f"Loading chunk {chunk_count}... rows: {len(chunk):,}")

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")

    chunk = chunk.dropna(subset=["year", "industry_name", "metric_name", "value"])
    chunk["year"] = chunk["year"].astype(int)

    chunk = chunk[
        (chunk["year"] >= START_YEAR) &
        (chunk["year"] <= END_YEAR)
    ].copy()

    if chunk.empty:
        continue

    metric = chunk["metric_name"].astype(str).str.lower()

    # Exclude rate / percent / share rows
    not_rate_mask = ~metric.str.contains(
        CFG["RATE_WORDS"],
        na=False,
        regex=True
    )

    # Exclude startup / business / company close rows
    not_startup_company_mask = ~metric.str.contains(
        CFG["STARTUP_COMPANY_WORDS"],
        na=False,
        regex=True
    )

    # Job gain only
    job_gain_mask = metric.str.contains(
        CFG["JOB_GAIN_WORDS"],
        na=False,
        regex=True
    ) & not_rate_mask & not_startup_company_mask

    # Job loss only
    job_loss_mask = metric.str.contains(
        CFG["JOB_LOSS_WORDS"],
        na=False,
        regex=True
    ) & not_rate_mask & not_startup_company_mask

    matched_job_gain_metric_names.update(
        chunk.loc[job_gain_mask, "metric_name"].dropna().unique()
    )

    matched_job_loss_metric_names.update(
        chunk.loc[job_loss_mask, "metric_name"].dropna().unique()
    )

    chunk["job_gain"] = np.where(job_gain_mask, chunk["value"], 0)
    chunk["job_loss"] = np.where(job_loss_mask, chunk["value"], 0)

    part = (
        chunk
        .groupby(["year", "industry_name"], dropna=False)[["job_gain", "job_loss"]]
        .sum()
        .reset_index()
    )

    parts.append(part)

print("Finished loading data.")


# ============================================================
# COMBINE DATA
# ============================================================

if not parts:
    raise ValueError(f"No Company job data found from {START_YEAR} to {END_YEAR}.")

csv_data = pd.concat(parts, ignore_index=True)

csv_data = (
    csv_data
    .groupby(["year", "industry_name"], dropna=False)[["job_gain", "job_loss"]]
    .sum()
    .reset_index()
)

csv_data["job_total"] = csv_data["job_gain"] + csv_data["job_loss"]
csv_data = csv_data[csv_data["job_total"] > 0].copy()

if csv_data.empty:
    raise ValueError(
        f"Data loaded, but no job gain/loss rows matched from {START_YEAR} to {END_YEAR}."
    )


# ============================================================
# OPTIONAL: KEEP TOP INDUSTRIES ONLY
# None = keep all industries
# ============================================================

if TOP_N is not None:
    top_industries = (
        csv_data.groupby("industry_name")["job_total"]
        .sum()
        .sort_values(ascending=False)
        .head(TOP_N)
        .index
    )

    csv_data["industry_name"] = np.where(
        csv_data["industry_name"].isin(top_industries),
        csv_data["industry_name"],
        "Other"
    )

    csv_data = (
        csv_data
        .groupby(["year", "industry_name"], dropna=False)[["job_gain", "job_loss"]]
        .sum()
        .reset_index()
    )

    csv_data["job_total"] = csv_data["job_gain"] + csv_data["job_loss"]


# ============================================================
# SORT DATA
# ============================================================

csv_data = csv_data.sort_values(
    by=["year", "industry_name"],
    ascending=[True, True]
).reset_index(drop=True)


# ============================================================
# SAVE CSV
# ============================================================

save_path = OUTPUT_DIR / CFG["SAVE_FILE_NAME"].format(
    start=START_YEAR,
    end=END_YEAR
)

csv_data.to_csv(save_path, index=False)


# ============================================================
# PRINT CHECK INFO
# ============================================================

print("\nMetric names used for job_gain:")
print(sorted(matched_job_gain_metric_names))

print("\nMetric names used for job_loss:")
print(sorted(matched_job_loss_metric_names))

print("\nCSV preview:")
print(csv_data.head(30).to_string(index=False))

print("\nRows saved:", len(csv_data))
print("Year range saved:", csv_data["year"].min(), "-", csv_data["year"].max())

print("\nALL DONE.")
print("Saved CSV here:")
print(save_path)

# 2

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# SETTINGS OBJECT
# Change only this part later
# ============================================================

CFG = {
    "COMPANY_FILE": Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv",

    "OUTPUT_DIR": Path.home() / "Downloads" / "pre_formula_startup_death_csv",

    "CHUNKSIZE": 100_000,

    "START_YEAR": 1970,
    "END_YEAR": 2025,

    # None = keep all industries
    # Example: 10 or 15 = show top industries, combine rest as Other
    "TOP_N": None,

    "COMPANY_COLS": [
        "year",
        "industry_name",
        "metric_name",
        "value"
    ],

    # Exclude rate / percent rows
    "RATE_WORDS": "rate|percent|percentage|pct|share|ratio",

    # Exclude job rows
    # This CSV is startup/death only, NOT job gain/loss
    "JOB_WORDS": (
        "job|jobs|employment|hire|hires|hiring|"
        "layoff|layoffs|separation|separations"
    ),

    # Startup / new business metric words
    "STARTUP_WORDS": (
        "startup|start-up|birth|births|formation|formations|"
        "entry|entries|new business|new firm|new company|"
        "opening|openings"
    ),

    # Company death / closed business metric words
    "DEATH_WORDS": (
        "death|deaths|closed|closure|closures|closing|closings|"
        "exit|exits|failure|failures"
    ),

    "SAVE_FILE_NAME": "startup_death_data_{start}_{end}.csv",

    # How many rows to print in preview
    "CHECK_TABLE_ROWS": 50
}


# ============================================================
# VARIABLES FROM SETTINGS OBJECT
# ============================================================

COMPANY_FILE = CFG["COMPANY_FILE"]
OUTPUT_DIR = CFG["OUTPUT_DIR"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNKSIZE = CFG["CHUNKSIZE"]
START_YEAR = CFG["START_YEAR"]
END_YEAR = CFG["END_YEAR"]
TOP_N = CFG["TOP_N"]
COMPANY_COLS = CFG["COMPANY_COLS"]


# ============================================================
# CHECK FILE
# ============================================================

print("Checking Company file...")

if not COMPANY_FILE.exists():
    raise FileNotFoundError(f"Missing Company file: {COMPANY_FILE}")

print("Finished checking file.")
print("Company file:", COMPANY_FILE)


# ============================================================
# LOAD RAW COMPANY STARTUP / DEATH DATA
# ============================================================

print(f"\nLoading raw Company startup/death data from {START_YEAR} to {END_YEAR}...")

parts = []
chunk_count = 0

matched_startup_metric_names = set()
matched_death_metric_names = set()

for chunk in pd.read_csv(
    COMPANY_FILE,
    usecols=COMPANY_COLS,
    chunksize=CHUNKSIZE,
    low_memory=False
):
    chunk_count += 1
    print(f"Loading chunk {chunk_count}... rows: {len(chunk):,}")

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")

    chunk = chunk.dropna(subset=["year", "industry_name", "metric_name", "value"])
    chunk["year"] = chunk["year"].astype(int)

    chunk = chunk[
        (chunk["year"] >= START_YEAR) &
        (chunk["year"] <= END_YEAR)
    ].copy()

    if chunk.empty:
        continue

    metric = chunk["metric_name"].astype(str).str.lower()

    # Exclude rate / percent / share rows
    not_rate_mask = ~metric.str.contains(
        CFG["RATE_WORDS"],
        na=False,
        regex=True
    )

    # Exclude job rows
    not_job_mask = ~metric.str.contains(
        CFG["JOB_WORDS"],
        na=False,
        regex=True
    )

    # Startup / new business / new company
    startup_mask = metric.str.contains(
        CFG["STARTUP_WORDS"],
        na=False,
        regex=True
    ) & not_rate_mask & not_job_mask

    # Company death / closed business
    death_mask = metric.str.contains(
        CFG["DEATH_WORDS"],
        na=False,
        regex=True
    ) & not_rate_mask & not_job_mask

    matched_startup_metric_names.update(
        chunk.loc[startup_mask, "metric_name"].dropna().unique()
    )

    matched_death_metric_names.update(
        chunk.loc[death_mask, "metric_name"].dropna().unique()
    )

    chunk["startup_count"] = np.where(startup_mask, chunk["value"], 0)
    chunk["death_count"] = np.where(death_mask, chunk["value"], 0)

    part = (
        chunk
        .groupby(["year", "industry_name"], dropna=False)[["startup_count", "death_count"]]
        .sum()
        .reset_index()
    )

    parts.append(part)

print("Finished loading data.")


# ============================================================
# COMBINE DATA
# ============================================================

if not parts:
    raise ValueError(f"No Company startup/death data found from {START_YEAR} to {END_YEAR}.")

csv_data = pd.concat(parts, ignore_index=True)

csv_data = (
    csv_data
    .groupby(["year", "industry_name"], dropna=False)[["startup_count", "death_count"]]
    .sum()
    .reset_index()
)

csv_data["company_total"] = csv_data["startup_count"] + csv_data["death_count"]
csv_data = csv_data[csv_data["company_total"] > 0].copy()

if csv_data.empty:
    raise ValueError(
        f"Data loaded, but no startup/death rows matched from {START_YEAR} to {END_YEAR}."
    )


# ============================================================
# OPTIONAL: KEEP TOP INDUSTRIES ONLY
# None = keep all industries
# ============================================================

if TOP_N is not None:
    top_industries = (
        csv_data.groupby("industry_name")["company_total"]
        .sum()
        .sort_values(ascending=False)
        .head(TOP_N)
        .index
    )

    csv_data["industry_name"] = np.where(
        csv_data["industry_name"].isin(top_industries),
        csv_data["industry_name"],
        "Other"
    )

    csv_data = (
        csv_data
        .groupby(["year", "industry_name"], dropna=False)[["startup_count", "death_count"]]
        .sum()
        .reset_index()
    )

    csv_data["company_total"] = csv_data["startup_count"] + csv_data["death_count"]


# ============================================================
# SORT DATA
# ============================================================

csv_data = csv_data.sort_values(
    by=["year", "industry_name"],
    ascending=[True, True]
).reset_index(drop=True)


# ============================================================
# SAVE CSV
# ============================================================

save_path = OUTPUT_DIR / CFG["SAVE_FILE_NAME"].format(
    start=START_YEAR,
    end=END_YEAR
)

csv_data.to_csv(save_path, index=False)


# ============================================================
# PRINT CHECK INFO
# ============================================================

print("\nMetric names used for startup_count:")
print(sorted(matched_startup_metric_names))

print("\nMetric names used for death_count:")
print(sorted(matched_death_metric_names))

print("\nCSV preview:")
print(csv_data.head(CFG["CHECK_TABLE_ROWS"]).to_string(index=False))

print("\nRows saved:", len(csv_data))
print("Year range saved:", csv_data["year"].min(), "-", csv_data["year"].max())

print("\nALL DONE.")
print("Saved CSV here:")
print(save_path)

# 4

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# DEGREE COUNT BY YEAR + ALL CATEGORIES
# - NO TOP N
# - NO OTHER BUCKET
# - REMOVE UNKNOWN / BLANK / NAN
# - SAVE CSV ONLY
# - NO CHART
# ============================================================

CFG = {
    "DEGREE_FILE": Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv",
    "OUTPUT_DIR": Path.home() / "Downloads" / "degree_count_all_csv",

    "START_YEAR": 1970,
    "END_YEAR": 2025,

    "MAIN_COLUMN": "year",
    "VALUE_COLUMN": "degree_count",

    "OTHER_COLUMNS": [
        "major_name",
        "field_group",
        "field_subgroup",
        "degree_group",
        "award_level_name",
        "cip_code",
    ],

    "NEEDED_COLUMNS": [
        "year",
        "major_name",
        "field_group",
        "field_subgroup",
        "degree_count",
        "degree_group",
        "award_level_name",
        "cip_code",
    ],

    "CHUNKSIZE": 100_000,

    "PRINT_PREVIEW_ROWS": 30,
}

CFG["OUTPUT_DIR"].mkdir(parents=True, exist_ok=True)


# ============================================================
# CHECK FILE
# ============================================================

print("Checking Degree file...")

if not CFG["DEGREE_FILE"].exists():
    raise FileNotFoundError(f"Missing Degree file: {CFG['DEGREE_FILE']}")

print("Finished checking file.")
print("Degree file:", CFG["DEGREE_FILE"])


# ============================================================
# HELPER FUNCTION
# ============================================================

def clean_col(series):
    s = series.astype(str).str.strip()

    s = s.replace({
        "": "Unknown",
        "nan": "Unknown",
        "NaN": "Unknown",
        "None": "Unknown",
        "NULL": "Unknown",
        "null": "Unknown",
    })

    return s.fillna("Unknown")


# ============================================================
# LOAD + GROUP DATA BY CHUNK
# ============================================================

print(f"\nReading degree file from {CFG['START_YEAR']} to {CFG['END_YEAR']}...")

parts_by_column = {
    col: [] for col in CFG["OTHER_COLUMNS"]
}

chunk_count = 0
total_rows_used = 0

for chunk in pd.read_csv(
    CFG["DEGREE_FILE"],
    usecols=CFG["NEEDED_COLUMNS"],
    chunksize=CFG["CHUNKSIZE"],
    low_memory=False
):
    chunk_count += 1
    print(f"Reading chunk {chunk_count}... rows: {len(chunk):,}")

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["degree_count"] = pd.to_numeric(chunk["degree_count"], errors="coerce").fillna(0)

    chunk = chunk[
        (chunk["year"] >= CFG["START_YEAR"]) &
        (chunk["year"] <= CFG["END_YEAR"])
    ].copy()

    if chunk.empty:
        continue

    chunk["year"] = chunk["year"].astype(int)
    total_rows_used += len(chunk)

    for col_name in CFG["OTHER_COLUMNS"]:
        temp = chunk[["year", "degree_count", col_name]].copy()

        temp[col_name] = clean_col(temp[col_name])

        temp = temp[
            ~temp[col_name].astype(str).str.strip().str.lower().isin(
                ["unknown", "nan", "none", ""]
            )
        ].copy()

        if temp.empty:
            continue

        grouped = (
            temp
            .groupby(["year", col_name], as_index=False)["degree_count"]
            .sum()
        )

        parts_by_column[col_name].append(grouped)

print("\nFinished loading.")
print("Rows used:", total_rows_used)


# ============================================================
# SAVE CSV FOR EACH COLUMN
# ============================================================

years = list(range(CFG["START_YEAR"], CFG["END_YEAR"] + 1))

for col_name in CFG["OTHER_COLUMNS"]:
    print("\n" + "=" * 100)
    print(f"SAVING CSV FOR: {col_name}")
    print("=" * 100)

    if not parts_by_column[col_name]:
        print(f"No data found for {col_name}.")
        continue

    combined = pd.concat(parts_by_column[col_name], ignore_index=True)

    combined = (
        combined
        .groupby(["year", col_name], as_index=False)["degree_count"]
        .sum()
    )

    table = combined.pivot_table(
        index=col_name,
        columns="year",
        values="degree_count",
        aggfunc="sum",
        fill_value=0
    )

    for y in years:
        if y not in table.columns:
            table[y] = 0

    table = table[years]

    total_col_name = f"total_{CFG['START_YEAR']}_{CFG['END_YEAR']}"
    table[total_col_name] = table[years].sum(axis=1)

    table = table.sort_values(
        by=total_col_name,
        ascending=False
    )

    table_reset = table.reset_index()

    csv_path = CFG["OUTPUT_DIR"] / f"{col_name}_degree_count_table_ALL.csv"
    table_reset.to_csv(csv_path, index=False)

    print("CSV saved:", csv_path)
    print("Rows saved:", len(table_reset))
    print("\nPreview:")
    print(table_reset.head(CFG["PRINT_PREVIEW_ROWS"]).to_string(index=False))


# ============================================================
# DONE
# ============================================================

print("\nALL DONE.")
print("Saved CSV files in:")
print(CFG["OUTPUT_DIR"])

# 4.2

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# DEGREE RAW DATA CSV ONLY
# - NO IMAGE
# - NO PIVOT TABLE
# - NO TOP N
# - NO OTHER BUCKET
# - NO FAKE ZERO YEARS
# - SAVE RAW CSV
# - ALSO SAVE RAW CSV FOR EACH CATEGORY COLUMN
# ============================================================

CFG = {
    "DEGREE_FILE": Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv",
    "OUTPUT_DIR": Path.home() / "Downloads" / "degree_count_raw_csv_only",

    "START_YEAR": 1970,
    "END_YEAR": 2025,

    "MAIN_COLUMN": "year",
    "VALUE_COLUMN": "degree_count",

    "OTHER_COLUMNS": [
        "major_name",
        "field_group",
        "field_subgroup",
        "degree_group",
        "award_level_name",
        "cip_code",
    ],

    "NEEDED_COLUMNS": [
        "year",
        "major_name",
        "field_group",
        "field_subgroup",
        "degree_count",
        "degree_group",
        "award_level_name",
        "cip_code",
    ],

    "CHUNKSIZE": 100_000,

    # True = remove Unknown / blank / nan from each category output
    "REMOVE_UNKNOWN": True,

    # print sample table
    "PRINT_SAMPLE_ROWS": 30,
}

CFG["OUTPUT_DIR"].mkdir(parents=True, exist_ok=True)

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 300)
pd.set_option("display.max_rows", 100)

# ============================================================
# HELPER
# ============================================================

def clean_col(series):
    """
    Convert to string, strip spaces, and standardize blanks/nan/none.
    """
    s = series.astype(str).str.strip()

    s = s.replace({
        "": "Unknown",
        "nan": "Unknown",
        "NaN": "Unknown",
        "None": "Unknown",
        "NULL": "Unknown",
        "null": "Unknown",
    })

    return s.fillna("Unknown")


def is_good_value(series):
    """
    Keep values that are not unknown/blank/nan/none.
    """
    return ~series.astype(str).str.strip().str.lower().isin(
        ["unknown", "nan", "none", "", "null"]
    )

# ============================================================
# OUTPUT FILES
# ============================================================

all_raw_path = CFG["OUTPUT_DIR"] / "degree_RAW_all_columns.csv"

category_paths = {
    col: CFG["OUTPUT_DIR"] / f"{col}_degree_RAW.csv"
    for col in CFG["OTHER_COLUMNS"]
}

year_summary_path = CFG["OUTPUT_DIR"] / "degree_RAW_year_summary.csv"

# delete old files first so it does not append to old result
for path in [all_raw_path, year_summary_path] + list(category_paths.values()):
    if path.exists():
        path.unlink()

# ============================================================
# READ + SAVE RAW DATA
# ============================================================

print("Reading degree file...")

total_rows_saved = 0
year_summary_parts = []

first_all_file = True
first_category_file = {col: True for col in CFG["OTHER_COLUMNS"]}

for i, chunk in enumerate(
    pd.read_csv(
        CFG["DEGREE_FILE"],
        usecols=CFG["NEEDED_COLUMNS"],
        chunksize=CFG["CHUNKSIZE"],
        low_memory=False
    ),
    start=1
):
    print(f"Reading chunk {i}...")

    # clean numeric columns
    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["degree_count"] = pd.to_numeric(chunk["degree_count"], errors="coerce").fillna(0)

    # filter year range
    chunk = chunk[
        (chunk["year"] >= CFG["START_YEAR"]) &
        (chunk["year"] <= CFG["END_YEAR"])
    ].copy()

    if chunk.empty:
        continue

    chunk["year"] = chunk["year"].astype(int)

    # --------------------------------------------------------
    # SAVE ONE RAW CSV WITH ALL COLUMNS
    # --------------------------------------------------------
    chunk.to_csv(
        all_raw_path,
        mode="w" if first_all_file else "a",
        header=first_all_file,
        index=False
    )
    first_all_file = False

    total_rows_saved += len(chunk)

    # --------------------------------------------------------
    # SAVE RAW CSV FOR EACH CATEGORY COLUMN
    # each file = year + category + degree_count
    # --------------------------------------------------------
    for col in CFG["OTHER_COLUMNS"]:
        temp = chunk[["year", col, "degree_count"]].copy()

        temp[col] = clean_col(temp[col])

        if CFG["REMOVE_UNKNOWN"]:
            temp = temp[is_good_value(temp[col])].copy()

        if temp.empty:
            continue

        temp.to_csv(
            category_paths[col],
            mode="w" if first_category_file[col] else "a",
            header=first_category_file[col],
            index=False
        )

        first_category_file[col] = False

    # --------------------------------------------------------
    # YEAR SUMMARY FOR CHECKING REAL DATA
    # --------------------------------------------------------
    year_part = (
        chunk.groupby("year", as_index=False)
             .agg(
                 raw_rows=("year", "size"),
                 degree_total=("degree_count", "sum")
             )
    )

    year_summary_parts.append(year_part)

# ============================================================
# CHECK RESULT
# ============================================================

if total_rows_saved == 0:
    raise ValueError("No data found in selected year range.")

print("\nFinished loading and saving.")
print("Raw rows saved:", total_rows_saved)
print("Main raw CSV saved:", all_raw_path)

# ============================================================
# SAVE YEAR SUMMARY
# ============================================================

year_summary = (
    pd.concat(year_summary_parts, ignore_index=True)
      .groupby("year", as_index=False)
      .agg(
          raw_rows=("raw_rows", "sum"),
          degree_total=("degree_total", "sum")
      )
      .sort_values("year")
)

year_summary.to_csv(year_summary_path, index=False)

print("\nREAL DATA BY YEAR:")
display(year_summary)

print("\nYear summary saved:", year_summary_path)

# ============================================================
# PRINT SAMPLE FROM MAIN RAW FILE
# ============================================================

print("\nRAW DATA SAMPLE:")
sample_df = pd.read_csv(all_raw_path, nrows=CFG["PRINT_SAMPLE_ROWS"])
display(sample_df)

# ============================================================
# PRINT CATEGORY FILE PATHS
# ============================================================

print("\nCategory raw CSV files saved:")

for col, path in category_paths.items():
    if path.exists():
        print(f"{col}: {path}")
    else:
        print(f"{col}: No file saved because no data after filtering.")

print("\nDone.")
print("Saved in:", CFG["OUTPUT_DIR"])

# 4.3

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# DEGREE RAW DATA CSV ONLY
# - SEARCH 1970–2025
# - AUTO DETECT REAL USABLE YEAR RANGE
# - IF DEGREE_COUNT IS 0, DO NOT USE THAT YEAR
# - SAVE RAW CSV
# - SAVE YEAR RANGE SUMMARY
# - NO IMAGE
# ============================================================

CFG = {
    "DEGREE_FILE": Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv",
    "OUTPUT_DIR": Path.home() / "Downloads" / "degree_raw_auto_real_year_range",

    # Keep your full search range
    "START_YEAR": 1970,
    "END_YEAR": 2025,

    "VALUE_COLUMN": "degree_count",

    "OTHER_COLUMNS": [
        "major_name",
        "field_group",
        "field_subgroup",
        "degree_group",
        "award_level_name",
        "cip_code",
    ],

    "NEEDED_COLUMNS": [
        "year",
        "major_name",
        "field_group",
        "field_subgroup",
        "degree_count",
        "degree_group",
        "award_level_name",
        "cip_code",
    ],

    "CHUNKSIZE": 100_000,

    "REMOVE_UNKNOWN": True,
    "PRINT_SAMPLE_ROWS": 30,
}

CFG["OUTPUT_DIR"].mkdir(parents=True, exist_ok=True)

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 300)
pd.set_option("display.max_rows", 100)

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_col(series):
    s = series.astype(str).str.strip()

    s = s.replace({
        "": "Unknown",
        "nan": "Unknown",
        "NaN": "Unknown",
        "None": "Unknown",
        "NULL": "Unknown",
        "null": "Unknown",
    })

    return s.fillna("Unknown")


def is_good_value(series):
    return ~series.astype(str).str.strip().str.lower().isin(
        ["unknown", "nan", "none", "", "null"]
    )

# ============================================================
# OUTPUT FILES
# ============================================================

main_raw_path = CFG["OUTPUT_DIR"] / "degree_RAW_auto_real_year_range.csv"
year_summary_path = CFG["OUTPUT_DIR"] / "degree_year_summary_auto_real_year_range.csv"
column_range_path = CFG["OUTPUT_DIR"] / "degree_column_year_range_summary.csv"

category_paths = {
    col: CFG["OUTPUT_DIR"] / f"{col}_RAW_auto_real_year_range.csv"
    for col in CFG["OTHER_COLUMNS"]
}

# remove old files
for path in [main_raw_path, year_summary_path, column_range_path] + list(category_paths.values()):
    if path.exists():
        path.unlink()

# ============================================================
# READ DATA FROM 1970–2025
# ============================================================

print("Reading degree file from", CFG["START_YEAR"], "to", CFG["END_YEAR"], "...")

all_chunks = []

for i, chunk in enumerate(
    pd.read_csv(
        CFG["DEGREE_FILE"],
        usecols=CFG["NEEDED_COLUMNS"],
        chunksize=CFG["CHUNKSIZE"],
        low_memory=False
    ),
    start=1
):
    print(f"Reading chunk {i}...")

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["degree_count"] = pd.to_numeric(chunk["degree_count"], errors="coerce").fillna(0)

    chunk = chunk[
        (chunk["year"] >= CFG["START_YEAR"]) &
        (chunk["year"] <= CFG["END_YEAR"])
    ].copy()

    if chunk.empty:
        continue

    chunk["year"] = chunk["year"].astype(int)

    for col in CFG["OTHER_COLUMNS"]:
        chunk[col] = clean_col(chunk[col])

    all_chunks.append(chunk)

if not all_chunks:
    raise ValueError("No rows found in selected year range.")

df = pd.concat(all_chunks, ignore_index=True)

print("\nFinished loading.")
print("Rows found from selected range:", len(df))
print("Raw year range found:", df["year"].min(), "-", df["year"].max())

# ============================================================
# YEAR SUMMARY BEFORE FILTER
# ============================================================

year_summary_all = (
    df.groupby("year", as_index=False)
      .agg(
          raw_rows=("year", "size"),
          degree_total=("degree_count", "sum")
      )
      .sort_values("year")
)

print("\nALL YEARS CHECK:")
display(year_summary_all)

# ============================================================
# AUTO DETECT REAL USABLE YEAR RANGE
# ============================================================

nonzero_years = year_summary_all.loc[
    year_summary_all["degree_total"] > 0,
    "year"
]

if nonzero_years.empty:
    raise ValueError("Rows exist, but degree_count is 0 for every selected year.")

REAL_START_YEAR = int(nonzero_years.min())
REAL_END_YEAR = int(nonzero_years.max())

print("\nREAL USABLE DEGREE_COUNT YEAR RANGE:")
print(f"{REAL_START_YEAR} - {REAL_END_YEAR}")

# ============================================================
# FILTER TO REAL USABLE YEARS + NONZERO DEGREE_COUNT
# ============================================================

df_real = df[
    (df["year"] >= REAL_START_YEAR) &
    (df["year"] <= REAL_END_YEAR) &
    (df["degree_count"] > 0)
].copy()

# remove unknown rows from all category columns
if CFG["REMOVE_UNKNOWN"]:
    for col in CFG["OTHER_COLUMNS"]:
        df_real = df_real[is_good_value(df_real[col])].copy()

if df_real.empty:
    raise ValueError("No usable rows left after filtering.")

# ============================================================
# SAVE MAIN RAW CSV
# ============================================================

df_real.to_csv(main_raw_path, index=False)

print("\nMAIN RAW CSV SAVED:")
print(main_raw_path)

print("\nRAW DATA SAMPLE:")
display(df_real.head(CFG["PRINT_SAMPLE_ROWS"]))

# ============================================================
# SAVE REAL YEAR SUMMARY
# ============================================================

year_summary_real = (
    df_real.groupby("year", as_index=False)
           .agg(
               raw_rows=("year", "size"),
               degree_total=("degree_count", "sum")
           )
           .sort_values("year")
)

year_summary_real.to_csv(year_summary_path, index=False)

print("\nREAL NONZERO YEAR SUMMARY:")
display(year_summary_real)

print("\nYear summary saved:")
print(year_summary_path)

# ============================================================
# SAVE ONE RAW CSV FOR EACH COLUMN
# ============================================================

column_range_rows = []

for col in CFG["OTHER_COLUMNS"]:
    print("\n" + "=" * 100)
    print("COLUMN:", col)
    print("=" * 100)

    temp = df_real[["year", col, "degree_count"]].copy()

    if CFG["REMOVE_UNKNOWN"]:
        temp = temp[is_good_value(temp[col])].copy()

    if temp.empty:
        print("No usable data for:", col)

        column_range_rows.append({
            "column_name": col,
            "search_range": f"{CFG['START_YEAR']}-{CFG['END_YEAR']}",
            "usable_range": "No usable degree_count data",
            "first_usable_year": None,
            "last_usable_year": None,
            "raw_rows": 0,
            "degree_total": 0,
        })

        continue

    col_year_summary = (
        temp.groupby("year", as_index=False)
            .agg(
                raw_rows=("year", "size"),
                degree_total=("degree_count", "sum")
            )
            .sort_values("year")
    )

    col_nonzero_years = col_year_summary.loc[
        col_year_summary["degree_total"] > 0,
        "year"
    ]

    col_start_year = int(col_nonzero_years.min())
    col_end_year = int(col_nonzero_years.max())

    usable_range = f"{col_start_year}-{col_end_year}"

    print("Search range:", f"{CFG['START_YEAR']}-{CFG['END_YEAR']}")
    print("Usable range:", usable_range)
    print("Rows:", len(temp))
    print("Degree total:", int(temp["degree_count"].sum()))

    # save category raw csv
    category_path = category_paths[col]
    temp.to_csv(category_path, index=False)

    print("CSV saved:", category_path)

    column_range_rows.append({
        "column_name": col,
        "search_range": f"{CFG['START_YEAR']}-{CFG['END_YEAR']}",
        "usable_range": usable_range,
        "first_usable_year": col_start_year,
        "last_usable_year": col_end_year,
        "raw_rows": len(temp),
        "degree_total": int(temp["degree_count"].sum()),
    })

# ============================================================
# SAVE COLUMN YEAR RANGE SUMMARY
# ============================================================

column_range_df = pd.DataFrame(column_range_rows)

column_range_df.to_csv(column_range_path, index=False)

print("\nCOLUMN YEAR RANGE SUMMARY:")
display(column_range_df)

print("\nColumn range summary saved:")
print(column_range_path)

print("\nDone.")
print("Saved in:", CFG["OUTPUT_DIR"])

# 4.4

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# DEGREE RAW DATA CSV ONLY
# - SEARCH FULL RANGE 1970–2025
# - AUTO DETECT REAL USABLE DEGREE_COUNT RANGE
# - REMOVE degree_count = 0
# - REMOVE UNKNOWN / BLANK / NAN
# - SAVE RAW CSV ONLY
# - NO IMAGE
# - FILE NAME INCLUDES SEARCH RANGE + USABLE RANGE
# ============================================================

CFG = {
    "DEGREE_FILE": Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv",
    "OUTPUT_DIR": Path.home() / "Downloads" / "degree_raw_csv_with_year_range_in_filename",

    # Search this full range
    "START_YEAR": 1970,
    "END_YEAR": 2025,

    "VALUE_COLUMN": "degree_count",

    "OTHER_COLUMNS": [
        "major_name",
        "field_group",
        "field_subgroup",
        "degree_group",
        "award_level_name",
        "cip_code",
    ],

    "NEEDED_COLUMNS": [
        "year",
        "major_name",
        "field_group",
        "field_subgroup",
        "degree_count",
        "degree_group",
        "award_level_name",
        "cip_code",
    ],

    "CHUNKSIZE": 100_000,

    # remove Unknown / blank / nan text
    "REMOVE_UNKNOWN": True,

    # print sample rows
    "PRINT_SAMPLE_ROWS": 30,
}

CFG["OUTPUT_DIR"].mkdir(parents=True, exist_ok=True)

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 300)
pd.set_option("display.max_rows", 100)

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_col(series):
    """
    Convert to string, strip spaces, and standardize blanks/nan/none.
    """
    s = series.astype(str).str.strip()

    s = s.replace({
        "": "Unknown",
        "nan": "Unknown",
        "NaN": "Unknown",
        "None": "Unknown",
        "NULL": "Unknown",
        "null": "Unknown",
    })

    return s.fillna("Unknown")


def is_good_value(series):
    """
    Keep values that are not unknown/blank/nan/none.
    """
    return ~series.astype(str).str.strip().str.lower().isin(
        ["unknown", "nan", "none", "", "null"]
    )


def safe_name(text):
    """
    Make safe file name text.
    """
    return str(text).replace("/", "_").replace("\\", "_").replace(" ", "_")


# ============================================================
# FIRST PASS:
# CHECK ALL YEARS 1970–2025 AND FIND REAL USABLE RANGE
# ============================================================

print("First pass: checking real usable year range...")
print("Search range:", CFG["START_YEAR"], "-", CFG["END_YEAR"])

year_summary_parts = []
total_rows_found = 0

for i, chunk in enumerate(
    pd.read_csv(
        CFG["DEGREE_FILE"],
        usecols=CFG["NEEDED_COLUMNS"],
        chunksize=CFG["CHUNKSIZE"],
        low_memory=False
    ),
    start=1
):
    print(f"Checking chunk {i}...")

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["degree_count"] = pd.to_numeric(chunk["degree_count"], errors="coerce").fillna(0)

    chunk = chunk[
        (chunk["year"] >= CFG["START_YEAR"]) &
        (chunk["year"] <= CFG["END_YEAR"])
    ].copy()

    if chunk.empty:
        continue

    chunk["year"] = chunk["year"].astype(int)

    total_rows_found += len(chunk)

    year_part = (
        chunk.groupby("year", as_index=False)
             .agg(
                 raw_rows=("year", "size"),
                 degree_total=("degree_count", "sum")
             )
    )

    year_summary_parts.append(year_part)

if total_rows_found == 0:
    raise ValueError("No rows found in selected search range.")

year_summary_all = (
    pd.concat(year_summary_parts, ignore_index=True)
      .groupby("year", as_index=False)
      .agg(
          raw_rows=("raw_rows", "sum"),
          degree_total=("degree_total", "sum")
      )
      .sort_values("year")
)

print("\nALL YEARS CHECK:")
display(year_summary_all)

nonzero_years = year_summary_all.loc[
    year_summary_all["degree_total"] > 0,
    "year"
]

if nonzero_years.empty:
    raise ValueError("Rows exist, but degree_count is 0 for every selected year.")

REAL_START_YEAR = int(nonzero_years.min())
REAL_END_YEAR = int(nonzero_years.max())

print("\nREAL USABLE DEGREE_COUNT RANGE:")
print(f"{REAL_START_YEAR}-{REAL_END_YEAR}")

# ============================================================
# FILE NAME TAGS
# ============================================================

SEARCH_TAG = f"search_{CFG['START_YEAR']}-{CFG['END_YEAR']}"
USABLE_TAG = f"usable_{REAL_START_YEAR}-{REAL_END_YEAR}"
FILE_TAG = f"{SEARCH_TAG}_{USABLE_TAG}"

# ============================================================
# OUTPUT FILES WITH YEAR RANGE IN FILE NAME
# ============================================================

main_raw_path = CFG["OUTPUT_DIR"] / f"degree_RAW_{FILE_TAG}.csv"

all_year_summary_path = CFG["OUTPUT_DIR"] / f"degree_ALL_year_summary_{FILE_TAG}.csv"

usable_year_summary_path = CFG["OUTPUT_DIR"] / f"degree_USABLE_year_summary_{FILE_TAG}.csv"

column_range_path = CFG["OUTPUT_DIR"] / f"degree_column_year_range_summary_{FILE_TAG}.csv"

category_paths = {
    col: CFG["OUTPUT_DIR"] / f"{safe_name(col)}_RAW_{FILE_TAG}.csv"
    for col in CFG["OTHER_COLUMNS"]
}

# remove old files with same names
for path in [main_raw_path, all_year_summary_path, usable_year_summary_path, column_range_path] + list(category_paths.values()):
    if path.exists():
        path.unlink()

# save all-year summary
year_summary_all.to_csv(all_year_summary_path, index=False)

print("\nAll-year summary saved:")
print(all_year_summary_path)

# ============================================================
# SECOND PASS:
# SAVE RAW NONZERO DATA ONLY
# ============================================================

print("\nSecond pass: saving raw usable data...")

first_main_file = True
first_category_file = {col: True for col in CFG["OTHER_COLUMNS"]}

total_rows_read_usable_range = 0
total_rows_saved = 0

usable_year_summary_parts = []
column_summary_rows = []

# for per-column summaries
column_year_summary_parts = {col: [] for col in CFG["OTHER_COLUMNS"]}

for i, chunk in enumerate(
    pd.read_csv(
        CFG["DEGREE_FILE"],
        usecols=CFG["NEEDED_COLUMNS"],
        chunksize=CFG["CHUNKSIZE"],
        low_memory=False
    ),
    start=1
):
    print(f"Saving chunk {i}...")

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["degree_count"] = pd.to_numeric(chunk["degree_count"], errors="coerce").fillna(0)

    # use only real usable range
    chunk = chunk[
        (chunk["year"] >= REAL_START_YEAR) &
        (chunk["year"] <= REAL_END_YEAR)
    ].copy()

    if chunk.empty:
        continue

    chunk["year"] = chunk["year"].astype(int)

    total_rows_read_usable_range += len(chunk)

    # remove zero count rows
    chunk = chunk[chunk["degree_count"] > 0].copy()

    if chunk.empty:
        continue

    # clean category text columns
    for col in CFG["OTHER_COLUMNS"]:
        chunk[col] = clean_col(chunk[col])

    # remove rows where any category column is Unknown / blank / nan
    if CFG["REMOVE_UNKNOWN"]:
        for col in CFG["OTHER_COLUMNS"]:
            chunk = chunk[is_good_value(chunk[col])].copy()

    if chunk.empty:
        continue

    total_rows_saved += len(chunk)

    # --------------------------------------------------------
    # SAVE MAIN RAW CSV
    # --------------------------------------------------------

    chunk.to_csv(
        main_raw_path,
        mode="w" if first_main_file else "a",
        header=first_main_file,
        index=False
    )

    first_main_file = False

    # --------------------------------------------------------
    # SAVE USABLE YEAR SUMMARY PART
    # --------------------------------------------------------

    usable_year_part = (
        chunk.groupby("year", as_index=False)
             .agg(
                 raw_rows=("year", "size"),
                 degree_total=("degree_count", "sum")
             )
    )

    usable_year_summary_parts.append(usable_year_part)

    # --------------------------------------------------------
    # SAVE RAW CSV FOR EACH CATEGORY COLUMN
    # --------------------------------------------------------

    for col in CFG["OTHER_COLUMNS"]:
        temp = chunk[["year", col, "degree_count"]].copy()

        if CFG["REMOVE_UNKNOWN"]:
            temp = temp[is_good_value(temp[col])].copy()

        if temp.empty:
            continue

        temp.to_csv(
            category_paths[col],
            mode="w" if first_category_file[col] else "a",
            header=first_category_file[col],
            index=False
        )

        first_category_file[col] = False

        col_year_part = (
            temp.groupby("year", as_index=False)
                .agg(
                    raw_rows=("year", "size"),
                    degree_total=("degree_count", "sum")
                )
        )

        column_year_summary_parts[col].append(col_year_part)

# ============================================================
# CHECK SAVED RESULT
# ============================================================

if total_rows_saved == 0:
    raise ValueError("No usable nonzero degree_count rows saved.")

print("\nFinished saving.")
print("Rows read in usable range:", total_rows_read_usable_range)
print("Rows saved with degree_count > 0:", total_rows_saved)

# ============================================================
# SAVE USABLE YEAR SUMMARY
# ============================================================

usable_year_summary = (
    pd.concat(usable_year_summary_parts, ignore_index=True)
      .groupby("year", as_index=False)
      .agg(
          raw_rows=("raw_rows", "sum"),
          degree_total=("degree_total", "sum")
      )
      .sort_values("year")
)

usable_year_summary.to_csv(usable_year_summary_path, index=False)

print("\nUSABLE YEAR SUMMARY:")
display(usable_year_summary)

print("\nUsable year summary saved:")
print(usable_year_summary_path)

# ============================================================
# MAKE COLUMN RANGE SUMMARY
# ============================================================

for col in CFG["OTHER_COLUMNS"]:
    if not column_year_summary_parts[col]:
        column_summary_rows.append({
            "column_name": col,
            "search_range": f"{CFG['START_YEAR']}-{CFG['END_YEAR']}",
            "usable_range": "No usable degree_count data",
            "first_usable_year": None,
            "last_usable_year": None,
            "raw_rows": 0,
            "degree_total": 0,
            "file_saved": "No file saved"
        })
        continue

    col_summary = (
        pd.concat(column_year_summary_parts[col], ignore_index=True)
          .groupby("year", as_index=False)
          .agg(
              raw_rows=("raw_rows", "sum"),
              degree_total=("degree_total", "sum")
          )
          .sort_values("year")
    )

    col_nonzero_years = col_summary.loc[
        col_summary["degree_total"] > 0,
        "year"
    ]

    col_start = int(col_nonzero_years.min())
    col_end = int(col_nonzero_years.max())

    column_summary_rows.append({
        "column_name": col,
        "search_range": f"{CFG['START_YEAR']}-{CFG['END_YEAR']}",
        "usable_range": f"{col_start}-{col_end}",
        "first_usable_year": col_start,
        "last_usable_year": col_end,
        "raw_rows": int(col_summary["raw_rows"].sum()),
        "degree_total": int(col_summary["degree_total"].sum()),
        "file_saved": str(category_paths[col])
    })

column_range_df = pd.DataFrame(column_summary_rows)

column_range_df.to_csv(column_range_path, index=False)

print("\nCOLUMN YEAR RANGE SUMMARY:")
display(column_range_df)

print("\nColumn range summary saved:")
print(column_range_path)

# ============================================================
# PRINT RAW SAMPLE
# ============================================================

print("\nRAW SAVED DATA SAMPLE:")
sample_df = pd.read_csv(main_raw_path, nrows=CFG["PRINT_SAMPLE_ROWS"])
display(sample_df)

# ============================================================
# PRINT FILES SAVED
# ============================================================

print("\nFILES SAVED:")
print("Main raw CSV:")
print(main_raw_path)

print("\nAll-year summary CSV:")
print(all_year_summary_path)

print("\nUsable-year summary CSV:")
print(usable_year_summary_path)

print("\nColumn range summary CSV:")
print(column_range_path)

print("\nCategory raw CSV files:")

for col, path in category_paths.items():
    if path.exists():
        print(f"{col}: {path}")
    else:
        print(f"{col}: No file saved")

print("\nDone.")
print("Saved in:", CFG["OUTPUT_DIR"])

# Read column

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# READ ALL COLUMN NAMES IN TABLE
# Column names only, not full data
# ============================================================

BASE_DIR = Path.home() / "Downloads" / "MyProject_1"

FILES = [
    BASE_DIR / "degree_raw_csv_with_year_range_in_filename" / "award_level_name_RAW_search_1970-2025_usable_2008-2024.csv",
    BASE_DIR / "degree_raw_csv_with_year_range_in_filename" / "cip_code_RAW_search_1970-2025_usable_2008-2024.csv",
    BASE_DIR / "degree_raw_csv_with_year_range_in_filename" / "degree_ALL_year_summary_search_1970-2025_usable_2008-2024.csv",
    BASE_DIR / "degree_raw_csv_with_year_range_in_filename" / "degree_column_year_range_summary_search_1970-2025_usable_2008-2024-UserNameUnknow.csv",
    BASE_DIR / "degree_raw_csv_with_year_range_in_filename" / "degree_group_RAW_search_1970-2025_usable_2008-2024.csv",
    BASE_DIR / "degree_raw_csv_with_year_range_in_filename" / "degree_RAW_search_1970-2025_usable_2008-2024.csv",
    BASE_DIR / "degree_raw_csv_with_year_range_in_filename" / "degree_USABLE_year_summary_search_1970-2025_usable_2008-2024.csv",
    BASE_DIR / "degree_raw_csv_with_year_range_in_filename" / "field_group_RAW_search_1970-2025_usable_2008-2024.csv",
    BASE_DIR / "degree_raw_csv_with_year_range_in_filename" / "field_subgroup_RAW_search_1970-2025_usable_2008-2024.csv",
    BASE_DIR / "degree_raw_csv_with_year_range_in_filename" / "major_name_RAW_search_1970-2025_usable_2008-2024.csv",

    BASE_DIR / "pre_formula_job_csv" / "job_gain_loss_data_1970_2025.csv",
    BASE_DIR / "pre_formula_startup_death_csv" / "startup_death_data_1970_2025.csv",
]

rows = []

for file_path in FILES:
    print(f"Reading columns only: {file_path.name}")

    if not file_path.exists():
        rows.append({
            "file": file_path.name,
            "folder": file_path.parent.name,
            "column_number": "",
            "column_name": "FILE NOT FOUND"
        })
        continue

    try:
        # Read header only
        df = pd.read_csv(file_path, nrows=0)

        for number, column in enumerate(df.columns, start=1):
            rows.append({
                "file": file_path.name,
                "folder": file_path.parent.name,
                "column_number": number,
                "column_name": column
            })

    except Exception as e:
        rows.append({
            "file": file_path.name,
            "folder": file_path.parent.name,
            "column_number": "",
            "column_name": f"ERROR: {e}"
        })

# Create table
column_table = pd.DataFrame(rows)

# Print full table, no ...
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

print("\nALL COLUMNS TABLE:")
print(column_table.to_string(index=False))


print("Done yes")

# Html Path
MyProject_1/
  degree_raw_csv_with_year_range_in_filename/
  pre_formula_job_csv/
  pre_formula_startup_death_csv/

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import time

# ============================================================
# MYPROJECT_1: PRINT TABLE + SAVE PNG CHARTS
# BY YEAR RANGE
#
# Example:
# Information industry from 2020-2024
#
# Company:
#   year + industry_name + metric_name + value
#
# Degree:
#   year + field_group / field_subgroup / major_name / etc. + degree_count
# ============================================================

# ============================================================
# CHANGE THIS PATH IF NEEDED
# ============================================================

BASE_DIR = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/MyProject_1"

OUTPUT_DIR = BASE_DIR / "saved_python_pic_and_tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# SETTINGS YOU CAN CHANGE
# ============================================================

START_YEAR = 2020
END_YEAR = 2024

# Example: print/chart only Information industry
# Use [] if you want ALL industries
TARGET_INDUSTRIES = ["Information"]

# Degree category filter.
# Use [] if you want ALL degree fields/categories
# Example:
# TARGET_DEGREE_VALUES = ["Computer and Information Sciences and Support Services"]
TARGET_DEGREE_VALUES = []

# Save top N only when showing ALL categories.
# Use None if you want all categories.
TOP_N = None

CHUNKSIZE = 100_000

# ============================================================
# YOUR FILES
# ============================================================

FILES = [
    BASE_DIR / "degree_raw_csv_with_year_range_in_filename" / "award_level_name_RAW_search_1970-2025_usable_2008-2024.csv",
    BASE_DIR / "degree_raw_csv_with_year_range_in_filename" / "cip_code_RAW_search_1970-2025_usable_2008-2024.csv",
    BASE_DIR / "degree_raw_csv_with_year_range_in_filename" / "degree_ALL_year_summary_search_1970-2025_usable_2008-2024.csv",
    BASE_DIR / "degree_raw_csv_with_year_range_in_filename" / "degree_column_year_range_summary_search_1970-2025_usable_2008-2024-UserNameUnknow.csv",
    BASE_DIR / "degree_raw_csv_with_year_range_in_filename" / "degree_group_RAW_search_1970-2025_usable_2008-2024.csv",
    BASE_DIR / "degree_raw_csv_with_year_range_in_filename" / "degree_RAW_search_1970-2025_usable_2008-2024.csv",
    BASE_DIR / "degree_raw_csv_with_year_range_in_filename" / "degree_USABLE_year_summary_search_1970-2025_usable_2008-2024.csv",
    BASE_DIR / "degree_raw_csv_with_year_range_in_filename" / "field_group_RAW_search_1970-2025_usable_2008-2024.csv",
    BASE_DIR / "degree_raw_csv_with_year_range_in_filename" / "field_subgroup_RAW_search_1970-2025_usable_2008-2024.csv",
    BASE_DIR / "degree_raw_csv_with_year_range_in_filename" / "major_name_RAW_search_1970-2025_usable_2008-2024.csv",

    BASE_DIR / "pre_formula_job_csv" / "job_gain_loss_data_1970_2025.csv",
    BASE_DIR / "pre_formula_startup_death_csv" / "startup_death_data_1970_2025.csv",
]

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def fmt_number(x, pos):
    try:
        return f"{int(x):,}"
    except Exception:
        return str(x)


def clean_text(s):
    return (
        s.astype(str)
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "NaN": pd.NA,
            "None": pd.NA,
            "UNKNOWN": pd.NA,
            "Unknown": pd.NA,
            "unknown": pd.NA,
        })
    )


def safe_name(text):
    text = str(text)
    bad = ['/', '\\', ':', '*', '?', '"', '<', '>', '|', ' ']
    for b in bad:
        text = text.replace(b, "_")
    return text[:120]


def get_columns(file_path):
    return pd.read_csv(file_path, nrows=0).columns.tolist()


def detect_file_type_and_columns(file_path):
    cols = get_columns(file_path)

    # Company job/startup/death files
    if "industry_name" in cols and "value" in cols and "year" in cols:
        return {
            "file_type": "company",
            "year_col": "year",
            "category_col": "industry_name",
            "value_col": "value",
            "metric_col": "metric_name" if "metric_name" in cols else None,
        }

    # Degree raw files
    degree_category_candidates = [
        "award_level_name",
        "cip_code",
        "degree_group",
        "field_group",
        "field_subgroup",
        "major_name",
    ]

    if "year" in cols and "degree_count" in cols:
        for c in degree_category_candidates:
            if c in cols:
                return {
                    "file_type": "degree",
                    "year_col": "year",
                    "category_col": c,
                    "value_col": "degree_count",
                    "metric_col": None,
                }

    return {
        "file_type": "skip",
        "year_col": None,
        "category_col": None,
        "value_col": None,
        "metric_col": None,
    }


def apply_text_filter(df, col, targets):
    if not targets:
        return df

    mask = False
    text = df[col].astype(str).str.lower()

    for t in targets:
        mask = mask | text.str.contains(str(t).lower(), na=False)

    return df[mask]


def apply_top_n(df, category_col, value_col, top_n):
    if top_n is None:
        return df

    totals = (
        df.groupby(category_col, as_index=False)[value_col]
        .sum()
        .sort_values(value_col, ascending=False)
    )

    keep = totals.head(top_n)[category_col].tolist()

    df = df.copy()
    df[category_col] = df[category_col].where(df[category_col].isin(keep), "Other")

    group_cols = ["year", category_col]
    if "metric_name" in df.columns:
        group_cols.append("metric_name")

    return df.groupby(group_cols, as_index=False)[value_col].sum()


def print_total_table(df, file_type, category_col, value_col, file_name):
    print("\n" + "=" * 100)
    print("TABLE FOR:", file_name)
    print("TYPE:", file_type)
    print("YEAR RANGE:", START_YEAR, "-", END_YEAR)
    print("CATEGORY COLUMN:", category_col)
    print("VALUE COLUMN:", value_col)

    if df.empty:
        print("No data found.")
        return

    if "metric_name" in df.columns:
        yearly_table = (
            df.groupby(["year", category_col, "metric_name"], as_index=False)[value_col]
            .sum()
            .sort_values(["year", category_col, "metric_name"])
        )

        pivot = yearly_table.pivot_table(
            index=["year", category_col],
            columns="metric_name",
            values=value_col,
            aggfunc="sum",
            fill_value=0
        ).reset_index()

        print("\nYEARLY TABLE:")
        print(pivot.to_string(index=False))

        total_table = (
            df.groupby([category_col, "metric_name"], as_index=False)[value_col]
            .sum()
            .sort_values(value_col, ascending=False)
        )

        print("\nTOTAL TABLE FOR WHOLE RANGE:")
        print(total_table.to_string(index=False))

        grand_total = df[value_col].sum()
        print("\nGRAND TOTAL:", f"{grand_total:,.0f}")

    else:
        yearly_table = (
            df.groupby(["year", category_col], as_index=False)[value_col]
            .sum()
            .sort_values(["year", category_col])
        )

        print("\nYEARLY TABLE:")
        print(yearly_table.to_string(index=False))

        total_table = (
            df.groupby(category_col, as_index=False)[value_col]
            .sum()
            .sort_values(value_col, ascending=False)
        )

        print("\nTOTAL TABLE FOR WHOLE RANGE:")
        print(total_table.to_string(index=False))

        grand_total = df[value_col].sum()
        print("\nGRAND TOTAL:", f"{grand_total:,.0f}")


def save_chart(df, file_type, category_col, value_col, file_path):
    if df.empty:
        print("No chart saved because data is empty.")
        return

    file_stem = safe_name(file_path.stem)
    category_safe = safe_name(category_col)

    # If metric_name exists, chart by metric_name.
    # Example: Information industry, job gain vs job loss.
    if "metric_name" in df.columns:
        chart_df = df.groupby(["year", "metric_name"], as_index=False)[value_col].sum()

        pivot = chart_df.pivot_table(
            index="year",
            columns="metric_name",
            values=value_col,
            aggfunc="sum",
            fill_value=0
        ).sort_index()

        csv_path = OUTPUT_DIR / f"{file_stem}_{START_YEAR}_{END_YEAR}_table.csv"
        png_path = OUTPUT_DIR / f"{file_stem}_{START_YEAR}_{END_YEAR}_chart.png"

        pivot.to_csv(csv_path)

        ax = pivot.plot(
            kind="bar",
            stacked=False,
            figsize=(14, 7),
            width=0.85
        )

        ax.set_title(f"{file_type.title()} by Year: {file_path.name}")
        ax.set_xlabel("Year")
        ax.set_ylabel(value_col)
        ax.yaxis.set_major_formatter(FuncFormatter(fmt_number))

        plt.xticks(rotation=0)
        plt.legend(title="metric_name", bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        plt.savefig(png_path, dpi=200, bbox_inches="tight")
        plt.close()

        print("\nSaved PNG:", png_path)
        print("Saved CSV:", csv_path)
        return

    # If no metric_name, chart by category.
    chart_df = df.groupby(["year", category_col], as_index=False)[value_col].sum()

    chart_df = apply_top_n(
        chart_df,
        category_col=category_col,
        value_col=value_col,
        top_n=TOP_N
    )

    pivot = chart_df.pivot_table(
        index="year",
        columns=category_col,
        values=value_col,
        aggfunc="sum",
        fill_value=0
    ).sort_index()

    csv_path = OUTPUT_DIR / f"{file_stem}_{category_safe}_{START_YEAR}_{END_YEAR}_table.csv"
    png_path = OUTPUT_DIR / f"{file_stem}_{category_safe}_{START_YEAR}_{END_YEAR}_chart.png"

    pivot.to_csv(csv_path)

    ax = pivot.plot(
        kind="bar",
        stacked=True,
        figsize=(16, 8),
        width=0.85
    )

    ax.set_title(f"{file_type.title()} by Year and {category_col}: {file_path.name}")
    ax.set_xlabel("Year")
    ax.set_ylabel(value_col)
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_number))

    plt.xticks(rotation=0)
    plt.legend(title=category_col, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(png_path, dpi=200, bbox_inches="tight")
    plt.close()

    print("\nSaved PNG:", png_path)
    print("Saved CSV:", csv_path)


def load_file_by_year_range(file_path):
    info = detect_file_type_and_columns(file_path)

    file_type = info["file_type"]
    year_col = info["year_col"]
    category_col = info["category_col"]
    value_col = info["value_col"]
    metric_col = info["metric_col"]

    if file_type == "skip":
        print("\nSKIP:", file_path.name)
        print("Reason: no usable chart columns found.")
        print("Columns:", get_columns(file_path))
        return None, info

    usecols = [year_col, category_col, value_col]
    if metric_col:
        usecols.append(metric_col)

    parts = []
    total_rows = 0

    print("\n" + "#" * 100)
    print("LOADING:", file_path.name)
    print("TYPE:", file_type)
    print("USECOLS:", usecols)

    start_time = time.time()

    for chunk_number, chunk in enumerate(
        pd.read_csv(file_path, usecols=usecols, chunksize=CHUNKSIZE),
        start=1
    ):
        print(f"Reading chunk {chunk_number}...")

        chunk[year_col] = pd.to_numeric(chunk[year_col], errors="coerce")
        chunk[value_col] = pd.to_numeric(chunk[value_col], errors="coerce").fillna(0)

        chunk = chunk[
            (chunk[year_col] >= START_YEAR)
            & (chunk[year_col] <= END_YEAR)
        ]

        chunk[category_col] = clean_text(chunk[category_col])
        chunk = chunk.dropna(subset=[year_col, category_col])

        if metric_col:
            chunk[metric_col] = clean_text(chunk[metric_col])
            chunk = chunk.dropna(subset=[metric_col])
            chunk = chunk.rename(columns={metric_col: "metric_name"})

        if file_type == "company":
            chunk = apply_text_filter(chunk, category_col, TARGET_INDUSTRIES)

        if file_type == "degree":
            chunk = apply_text_filter(chunk, category_col, TARGET_DEGREE_VALUES)

        if chunk.empty:
            continue

        group_cols = [year_col, category_col]
        if "metric_name" in chunk.columns:
            group_cols.append("metric_name")

        grouped = (
            chunk.groupby(group_cols, as_index=False)[value_col]
            .sum()
        )

        parts.append(grouped)
        total_rows += len(chunk)

    elapsed = time.time() - start_time

    if not parts:
        print("No rows found for this file in selected range/filter.")
        return pd.DataFrame(), info

    df = pd.concat(parts, ignore_index=True)

    group_cols = [year_col, category_col]
    if "metric_name" in df.columns:
        group_cols.append("metric_name")

    df = (
        df.groupby(group_cols, as_index=False)[value_col]
        .sum()
        .rename(columns={year_col: "year"})
    )

    print("Rows used:", f"{total_rows:,}")
    print("Finished in:", f"{elapsed:.2f} seconds")

    return df, info


# ============================================================
# RUN ALL FILES
# ============================================================

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("YEAR RANGE:", START_YEAR, "-", END_YEAR)
print("TARGET_INDUSTRIES:", TARGET_INDUSTRIES if TARGET_INDUSTRIES else "ALL")
print("TARGET_DEGREE_VALUES:", TARGET_DEGREE_VALUES if TARGET_DEGREE_VALUES else "ALL")

all_summary_rows = []

for file_path in FILES:
    if not file_path.exists():
        print("\nMISSING FILE:", file_path)
        continue

    df, info = load_file_by_year_range(file_path)

    if df is None:
        continue

    file_type = info["file_type"]
    category_col = info["category_col"]
    value_col = info["value_col"]

    if df.empty:
        continue

    print_total_table(
        df=df,
        file_type=file_type,
        category_col=category_col,
        value_col=value_col,
        file_name=file_path.name
    )

    save_chart(
        df=df,
        file_type=file_type,
        category_col=category_col,
        value_col=value_col,
        file_path=file_path
    )

    # Summary total row
    total_value = df[value_col].sum()

    all_summary_rows.append({
        "file": file_path.name,
        "type": file_type,
        "category_column": category_col,
        "value_column": value_col,
        "start_year": START_YEAR,
        "end_year": END_YEAR,
        "total_count": total_value,
    })

# ============================================================
# FINAL SUMMARY TABLE
# ============================================================

summary_df = pd.DataFrame(all_summary_rows)

print("\n" + "=" * 100)
print("FINAL SUMMARY TABLE")

if summary_df.empty:
    print("No data found.")
else:
    summary_df["total_count"] = summary_df["total_count"].round(0).astype("int64")
    print(summary_df.to_string(index=False))

    summary_csv = OUTPUT_DIR / f"FINAL_SUMMARY_{START_YEAR}_{END_YEAR}.csv"
    summary_df.to_csv(summary_csv, index=False)
    print("\nSaved final summary CSV:", summary_csv)

print("\nDONE.")
print("All PNG pictures and CSV tables saved in:")
print(OUTPUT_DIR)

# Too many degree

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import time

# ============================================================
# DEGREE FIELD_GROUP ONLY
# PRINT TABLE + SAVE CSV + SAVE PNG
#
# File columns:
# year, field_group, degree_count
# ============================================================

# ============================================================
# PATH SETTINGS
# ============================================================

BASE_DIR = Path.home() / "Downloads/MyProject_1"

OUTPUT_DIR = BASE_DIR / "saved_python_pic_and_tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIELD_GROUP_FILE = (
    BASE_DIR
    / "degree_raw_csv_with_year_range_in_filename"
    / "field_group_RAW_search_1970-2025_usable_2008-2024.csv"
)

# ============================================================
# CHANGE THESE SETTINGS
# ============================================================

START_YEAR = 2020
END_YEAR = 2024

CHUNKSIZE = 100_000

# Use [] for ALL field groups
# Example:
# TARGET_FIELD_GROUP_VALUES = ["Computer and Information Sciences and Support Services"]
TARGET_FIELD_GROUP_VALUES = []

YEAR_COL = "year"
CATEGORY_COL = "field_group"
VALUE_COL = "degree_count"

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def fmt_number(x, pos):
    try:
        return f"{int(x):,}"
    except Exception:
        return str(x)


def clean_text(s):
    return (
        s.astype(str)
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "NaN": pd.NA,
            "None": pd.NA,
            "UNKNOWN": pd.NA,
            "Unknown": pd.NA,
            "unknown": pd.NA,
        })
    )


def apply_text_filter(df, col, targets):
    if not targets:
        return df

    mask = pd.Series(False, index=df.index)
    text = df[col].astype(str).str.lower()

    for t in targets:
        mask = mask | text.str.contains(str(t).lower(), na=False)

    return df[mask]


def print_table(title, df):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    if df.empty:
        print("No data found.")
    else:
        print(df.to_string(index=False))


# ============================================================
# CHECK FILE
# ============================================================

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("FIELD_GROUP_FILE:", FIELD_GROUP_FILE)
print("YEAR RANGE:", START_YEAR, "-", END_YEAR)
print("TARGET_FIELD_GROUP_VALUES:", TARGET_FIELD_GROUP_VALUES if TARGET_FIELD_GROUP_VALUES else "ALL")

if not FIELD_GROUP_FILE.exists():
    raise FileNotFoundError(f"Missing file: {FIELD_GROUP_FILE}")

# ============================================================
# LOAD ONLY FIELD_GROUP FILE
# ============================================================

parts = []
total_rows_used = 0
start_time = time.time()

print("\nLoading ONLY degree field_group file...")

for chunk_number, chunk in enumerate(
    pd.read_csv(
        FIELD_GROUP_FILE,
        usecols=[YEAR_COL, CATEGORY_COL, VALUE_COL],
        chunksize=CHUNKSIZE
    ),
    start=1
):
    print(f"Reading chunk {chunk_number}...")

    chunk[YEAR_COL] = pd.to_numeric(chunk[YEAR_COL], errors="coerce")
    chunk[VALUE_COL] = pd.to_numeric(chunk[VALUE_COL], errors="coerce").fillna(0)

    chunk = chunk[
        (chunk[YEAR_COL] >= START_YEAR)
        & (chunk[YEAR_COL] <= END_YEAR)
    ]

    chunk[CATEGORY_COL] = clean_text(chunk[CATEGORY_COL])
    chunk = chunk.dropna(subset=[YEAR_COL, CATEGORY_COL])

    chunk = apply_text_filter(
        chunk,
        CATEGORY_COL,
        TARGET_FIELD_GROUP_VALUES
    )

    if chunk.empty:
        continue

    grouped = (
        chunk.groupby([YEAR_COL, CATEGORY_COL], as_index=False)[VALUE_COL]
        .sum()
    )

    parts.append(grouped)
    total_rows_used += len(chunk)

elapsed = time.time() - start_time

# ============================================================
# IF NO DATA
# ============================================================

if not parts:
    print("\nNo data found.")
    print("DONE.")

else:
    # ========================================================
    # COMBINE CHUNKS
    # ========================================================

    df = pd.concat(parts, ignore_index=True)

    df = (
        df.groupby([YEAR_COL, CATEGORY_COL], as_index=False)[VALUE_COL]
        .sum()
        .sort_values([YEAR_COL, CATEGORY_COL])
    )

    print("\nFinished loading.")
    print("Rows used:", f"{total_rows_used:,}")
    print("Finished in:", f"{elapsed:.2f} seconds")

    # ========================================================
    # YEARLY TABLE
    # ========================================================

    yearly_table = df.copy()

    print_table(
        title=f"YEARLY DEGREE FIELD_GROUP TABLE {START_YEAR}-{END_YEAR}",
        df=yearly_table
    )

    # ========================================================
    # TOTAL TABLE
    # ========================================================

    total_table = (
        df.groupby(CATEGORY_COL, as_index=False)[VALUE_COL]
        .sum()
        .sort_values(VALUE_COL, ascending=False)
    )

    print_table(
        title=f"TOTAL DEGREE FIELD_GROUP TABLE {START_YEAR}-{END_YEAR}",
        df=total_table
    )

    grand_total = df[VALUE_COL].sum()

    print("\n" + "=" * 100)
    print("GRAND TOTAL:", f"{grand_total:,.0f}")

    # ========================================================
    # PIVOT TABLE FOR CHART
    # ========================================================

    pivot = df.pivot_table(
        index=YEAR_COL,
        columns=CATEGORY_COL,
        values=VALUE_COL,
        aggfunc="sum",
        fill_value=0
    ).sort_index()

    all_years = list(range(START_YEAR, END_YEAR + 1))
    pivot = pivot.reindex(all_years, fill_value=0)

    # ========================================================
    # SAVE CSV FILES
    # ========================================================

    yearly_csv = OUTPUT_DIR / f"degree_field_group_YEARLY_{START_YEAR}_{END_YEAR}.csv"
    total_csv = OUTPUT_DIR / f"degree_field_group_TOTAL_{START_YEAR}_{END_YEAR}.csv"
    pivot_csv = OUTPUT_DIR / f"degree_field_group_PIVOT_{START_YEAR}_{END_YEAR}.csv"

    yearly_table.to_csv(yearly_csv, index=False)
    total_table.to_csv(total_csv, index=False)
    pivot.to_csv(pivot_csv)

    print("\nSaved yearly CSV:", yearly_csv)
    print("Saved total CSV:", total_csv)
    print("Saved pivot CSV:", pivot_csv)

    # ========================================================
    # SAVE STACKED BAR PNG
    # ========================================================

    png_path = OUTPUT_DIR / f"degree_field_group_STACKED_{START_YEAR}_{END_YEAR}.png"

    ax = pivot.plot(
        kind="bar",
        stacked=True,
        figsize=(16, 8),
        width=0.85
    )

    ax.set_title(f"Degree Count by Year and Field Group ({START_YEAR}-{END_YEAR})")
    ax.set_xlabel("Year")
    ax.set_ylabel("Degree Count")
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_number))

    plt.xticks(rotation=0)
    plt.legend(title="field_group", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(png_path, dpi=200, bbox_inches="tight")
    plt.close()

    print("Saved PNG:", png_path)

    # ========================================================
    # DONE
    # ========================================================

    print("\nDONE.")
    print("All files saved in:")
    print(OUTPUT_DIR)

# Show top only and only 2008-2024

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter
import textwrap
import time

# ============================================================
# DEGREE FIELD_GROUP ONLY
# PRINT TABLE + SAVE CSV + SAVE BETTER PNG CHARTS
#
# File columns:
# year, field_group, degree_count
# ============================================================

# ============================================================
# PATH SETTINGS
# ============================================================

BASE_DIR = Path.home() / "Downloads/MyProject_1"

OUTPUT_DIR = BASE_DIR / "saved_python_pic_and_tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIELD_GROUP_FILE = (
    BASE_DIR
    / "degree_raw_csv_with_year_range_in_filename"
    / "field_group_RAW_search_1970-2025_usable_2008-2024.csv"
)

# ============================================================
# CHANGE THESE SETTINGS
# ============================================================

START_YEAR = 2008
END_YEAR = 2024

CHUNKSIZE = 100_000

# Use [] for ALL field groups
# Example:
# TARGET_FIELD_GROUP_VALUES = ["Computer and Information Sciences and Support Services"]
TARGET_FIELD_GROUP_VALUES = []

YEAR_COL = "year"
CATEGORY_COL = "field_group"
VALUE_COL = "degree_count"

# Chart setting:
# Use 12 or 15 for readable chart.
# Use None if you really want ALL categories in the stacked chart.
TOP_N_FOR_CHART = 12

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def fmt_number(x, pos):
    try:
        return f"{int(x):,}"
    except Exception:
        return str(x)


def clean_text(s):
    return (
        s.astype(str)
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "NaN": pd.NA,
            "None": pd.NA,
            "UNKNOWN": pd.NA,
            "Unknown": pd.NA,
            "unknown": pd.NA,
        })
    )


def apply_text_filter(df, col, targets):
    if not targets:
        return df

    mask = pd.Series(False, index=df.index)
    text = df[col].astype(str).str.lower()

    for t in targets:
        mask = mask | text.str.contains(str(t).lower(), na=False)

    return df[mask]


def print_table(title, df):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    if df.empty:
        print("No data found.")
    else:
        print(df.to_string(index=False))


def wrap_labels(labels, width=34):
    return [
        "\n".join(textwrap.wrap(str(label), width=width))
        for label in labels
    ]


def make_chart_pivot_for_picture(full_pivot, top_n):
    """
    This keeps CSV full, but makes the image easier to read.

    If top_n is None:
        chart uses all categories.

    If top_n is a number:
        chart uses Top N categories by total,
        and combines the rest into OTHER FIELD GROUPS.
    """

    category_totals = full_pivot.sum(axis=0).sort_values(ascending=False)

    if top_n is None:
        chart_pivot = full_pivot[category_totals.index]
        return chart_pivot

    top_categories = category_totals.head(top_n).index.tolist()

    chart_pivot = full_pivot[top_categories].copy()

    other_categories = [c for c in full_pivot.columns if c not in top_categories]

    if other_categories:
        chart_pivot["OTHER FIELD GROUPS"] = full_pivot[other_categories].sum(axis=1)

    return chart_pivot


# ============================================================
# CHECK FILE
# ============================================================

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("FIELD_GROUP_FILE:", FIELD_GROUP_FILE)
print("YEAR RANGE:", START_YEAR, "-", END_YEAR)
print("TARGET_FIELD_GROUP_VALUES:", TARGET_FIELD_GROUP_VALUES if TARGET_FIELD_GROUP_VALUES else "ALL")
print("TOP_N_FOR_CHART:", TOP_N_FOR_CHART if TOP_N_FOR_CHART is not None else "ALL")

if not FIELD_GROUP_FILE.exists():
    raise FileNotFoundError(f"Missing file: {FIELD_GROUP_FILE}")

# ============================================================
# LOAD ONLY FIELD_GROUP FILE
# ============================================================

parts = []
total_rows_used = 0
start_time = time.time()

print("\nLoading ONLY degree field_group file...")

for chunk_number, chunk in enumerate(
    pd.read_csv(
        FIELD_GROUP_FILE,
        usecols=[YEAR_COL, CATEGORY_COL, VALUE_COL],
        chunksize=CHUNKSIZE
    ),
    start=1
):
    print(f"Reading chunk {chunk_number}...")

    chunk[YEAR_COL] = pd.to_numeric(chunk[YEAR_COL], errors="coerce")
    chunk[VALUE_COL] = pd.to_numeric(chunk[VALUE_COL], errors="coerce").fillna(0)

    chunk = chunk[
        (chunk[YEAR_COL] >= START_YEAR)
        & (chunk[YEAR_COL] <= END_YEAR)
    ]

    chunk[CATEGORY_COL] = clean_text(chunk[CATEGORY_COL])
    chunk = chunk.dropna(subset=[YEAR_COL, CATEGORY_COL])

    chunk = apply_text_filter(
        chunk,
        CATEGORY_COL,
        TARGET_FIELD_GROUP_VALUES
    )

    if chunk.empty:
        continue

    grouped = (
        chunk.groupby([YEAR_COL, CATEGORY_COL], as_index=False)[VALUE_COL]
        .sum()
    )

    parts.append(grouped)
    total_rows_used += len(chunk)

elapsed = time.time() - start_time

# ============================================================
# IF NO DATA
# ============================================================

if not parts:
    print("\nNo data found.")
    print("DONE.")

else:
    # ========================================================
    # COMBINE CHUNKS
    # ========================================================

    df = pd.concat(parts, ignore_index=True)

    df = (
        df.groupby([YEAR_COL, CATEGORY_COL], as_index=False)[VALUE_COL]
        .sum()
        .sort_values([YEAR_COL, CATEGORY_COL])
    )

    print("\nFinished loading.")
    print("Rows used:", f"{total_rows_used:,}")
    print("Finished in:", f"{elapsed:.2f} seconds")

    # ========================================================
    # YEARLY TABLE
    # ========================================================

    yearly_table = df.copy()

    print_table(
        title=f"YEARLY DEGREE FIELD_GROUP TABLE {START_YEAR}-{END_YEAR}",
        df=yearly_table
    )

    # ========================================================
    # TOTAL TABLE
    # ========================================================

    total_table = (
        df.groupby(CATEGORY_COL, as_index=False)[VALUE_COL]
        .sum()
        .sort_values(VALUE_COL, ascending=False)
    )

    print_table(
        title=f"TOTAL DEGREE FIELD_GROUP TABLE {START_YEAR}-{END_YEAR}",
        df=total_table
    )

    grand_total = df[VALUE_COL].sum()

    print("\n" + "=" * 100)
    print("GRAND TOTAL:", f"{grand_total:,.0f}")

    # ========================================================
    # FULL PIVOT TABLE FOR CSV
    # ========================================================

    pivot = df.pivot_table(
        index=YEAR_COL,
        columns=CATEGORY_COL,
        values=VALUE_COL,
        aggfunc="sum",
        fill_value=0
    ).sort_index()

    all_years = list(range(START_YEAR, END_YEAR + 1))
    pivot = pivot.reindex(all_years, fill_value=0)

    # Sort columns by total count, biggest first
    pivot = pivot[pivot.sum(axis=0).sort_values(ascending=False).index]

    # ========================================================
    # SAVE FULL CSV FILES
    # ========================================================

    yearly_csv = OUTPUT_DIR / f"degree_field_group_YEARLY_FULL_{START_YEAR}_{END_YEAR}.csv"
    total_csv = OUTPUT_DIR / f"degree_field_group_TOTAL_FULL_{START_YEAR}_{END_YEAR}.csv"
    pivot_csv = OUTPUT_DIR / f"degree_field_group_PIVOT_FULL_{START_YEAR}_{END_YEAR}.csv"

    yearly_table.to_csv(yearly_csv, index=False)
    total_table.to_csv(total_csv, index=False)
    pivot.to_csv(pivot_csv)

    print("\nSaved yearly CSV:", yearly_csv)
    print("Saved total CSV:", total_csv)
    print("Saved pivot CSV:", pivot_csv)

    # ========================================================
    # MAKE CLEAN CHART DATA
    # ========================================================

    chart_pivot = make_chart_pivot_for_picture(
        full_pivot=pivot,
        top_n=TOP_N_FOR_CHART
    )

    chart_total_csv = OUTPUT_DIR / f"degree_field_group_CHART_DATA_{START_YEAR}_{END_YEAR}.csv"
    chart_pivot.to_csv(chart_total_csv)

    print("Saved chart data CSV:", chart_total_csv)

    # ========================================================
    # CHART 1: BETTER STACKED BAR
    # ========================================================

    stacked_png = OUTPUT_DIR / f"degree_field_group_BETTER_STACKED_{START_YEAR}_{END_YEAR}.png"

    ax = chart_pivot.plot(
        kind="bar",
        stacked=True,
        figsize=(18, 10),
        width=0.78,
        colormap="tab20"
    )

    ax.set_title(
        f"Degree Count by Year and Field Group ({START_YEAR}-{END_YEAR})",
        fontsize=16,
        pad=15
    )
    ax.set_xlabel("Year", fontsize=12)
    ax.set_ylabel("Degree Count", fontsize=12)
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_number))

    plt.xticks(rotation=0)

    handles, labels = ax.get_legend_handles_labels()
    labels = wrap_labels(labels, width=38)

    ax.legend(
        handles,
        labels,
        title="Field Group",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=8,
        title_fontsize=10,
        frameon=True
    )

    ax.grid(axis="y", alpha=0.25)

    plt.tight_layout()
    plt.savefig(stacked_png, dpi=220, bbox_inches="tight")
    plt.close()

    print("Saved better stacked PNG:", stacked_png)

    # ========================================================
    # CHART 2: 100% STACKED BAR
    # Easier to compare percent/share by year
    # ========================================================

    percent_png = OUTPUT_DIR / f"degree_field_group_PERCENT_STACKED_{START_YEAR}_{END_YEAR}.png"

    percent_pivot = chart_pivot.div(chart_pivot.sum(axis=1), axis=0).fillna(0)

    ax = percent_pivot.plot(
        kind="bar",
        stacked=True,
        figsize=(18, 10),
        width=0.78,
        colormap="tab20"
    )

    ax.set_title(
        f"Degree Field Group Share by Year ({START_YEAR}-{END_YEAR})",
        fontsize=16,
        pad=15
    )
    ax.set_xlabel("Year", fontsize=12)
    ax.set_ylabel("Percent of Degree Count", fontsize=12)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))

    plt.xticks(rotation=0)

    handles, labels = ax.get_legend_handles_labels()
    labels = wrap_labels(labels, width=38)

    ax.legend(
        handles,
        labels,
        title="Field Group",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=8,
        title_fontsize=10,
        frameon=True
    )

    ax.grid(axis="y", alpha=0.25)

    plt.tight_layout()
    plt.savefig(percent_png, dpi=220, bbox_inches="tight")
    plt.close()

    print("Saved percent stacked PNG:", percent_png)

    # ========================================================
    # CHART 3: TOTAL HORIZONTAL BAR
    # This is usually the cleanest chart for many categories
    # ========================================================

    total_bar_png = OUTPUT_DIR / f"degree_field_group_TOTAL_BAR_{START_YEAR}_{END_YEAR}.png"

    chart_totals = chart_pivot.sum(axis=0).sort_values(ascending=True)

    fig_height = max(7, len(chart_totals) * 0.55)

    ax = chart_totals.plot(
        kind="barh",
        figsize=(14, fig_height),
        colormap="tab20"
    )

    ax.set_title(
        f"Total Degree Count by Field Group ({START_YEAR}-{END_YEAR})",
        fontsize=16,
        pad=15
    )
    ax.set_xlabel("Degree Count", fontsize=12)
    ax.set_ylabel("Field Group", fontsize=12)
    ax.xaxis.set_major_formatter(FuncFormatter(fmt_number))

    wrapped_y_labels = wrap_labels(chart_totals.index, width=45)
    ax.set_yticklabels(wrapped_y_labels)

    ax.grid(axis="x", alpha=0.25)

    for i, value in enumerate(chart_totals.values):
        ax.text(
            value,
            i,
            f" {value:,.0f}",
            va="center",
            fontsize=8
        )

    plt.tight_layout()
    plt.savefig(total_bar_png, dpi=220, bbox_inches="tight")
    plt.close()

    print("Saved total bar PNG:", total_bar_png)

    # ========================================================
    # DONE
    # ========================================================

    print("\nDONE.")
    print("All files saved in:")
    print(OUTPUT_DIR)

# Show 2010-2024

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter
import textwrap
import time

# ============================================================
# DEGREE FIELD_GROUP ONLY
# PRINT TABLE + SAVE CSV + SAVE BETTER PNG CHARTS
#
# File columns:
# year, field_group, degree_count
#
# IMPORTANT:
# TOP_N_FOR_CHART = 12 means chart shows exactly Top 12.
# This version DOES NOT add "OTHER FIELD GROUPS".
# ============================================================

# ============================================================
# PATH SETTINGS
# ============================================================

BASE_DIR = Path.home() / "Downloads/MyProject_1"

OUTPUT_DIR = BASE_DIR / "saved_python_pic_and_tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIELD_GROUP_FILE = (
    BASE_DIR
    / "degree_raw_csv_with_year_range_in_filename"
    / "field_group_RAW_search_1970-2025_usable_2008-2024.csv"
)

# ============================================================
# CHANGE THESE SETTINGS
# ============================================================

START_YEAR = 2010
END_YEAR = 2024

CHUNKSIZE = 100_000

# Use [] for ALL field groups
# Example:
# TARGET_FIELD_GROUP_VALUES = ["Computer and Information Sciences and Support Services"]
TARGET_FIELD_GROUP_VALUES = []

YEAR_COL = "year"
CATEGORY_COL = "field_group"
VALUE_COL = "degree_count"

# Chart setting:
# Use 12 or 15 for readable chart.
# Use None if you really want ALL categories in the stacked chart.
TOP_N_FOR_CHART = 12

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def fmt_number(x, pos):
    try:
        return f"{int(x):,}"
    except Exception:
        return str(x)


def clean_text(s):
    return (
        s.astype(str)
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "NaN": pd.NA,
            "None": pd.NA,
            "UNKNOWN": pd.NA,
            "Unknown": pd.NA,
            "unknown": pd.NA,
        })
    )


def apply_text_filter(df, col, targets):
    if not targets:
        return df

    mask = pd.Series(False, index=df.index)
    text = df[col].astype(str).str.lower()

    for t in targets:
        mask = mask | text.str.contains(str(t).lower(), na=False)

    return df[mask]


def print_table(title, df):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    if df.empty:
        print("No data found.")
    else:
        print(df.to_string(index=False))


def wrap_labels(labels, width=34):
    return [
        "\n".join(textwrap.wrap(str(label), width=width))
        for label in labels
    ]


def make_chart_pivot_for_picture(full_pivot, top_n):
    """
    This keeps the CSV full, but makes the image easier to read.

    If top_n is None:
        chart uses all categories.

    If top_n is a number:
        chart uses ONLY Top N categories by total degree_count.
        This version DOES NOT create OTHER FIELD GROUPS.
    """

    category_totals = full_pivot.sum(axis=0).sort_values(ascending=False)

    if top_n is None:
        chart_pivot = full_pivot[category_totals.index]
        return chart_pivot

    top_categories = category_totals.head(top_n).index.tolist()

    chart_pivot = full_pivot[top_categories].copy()

    return chart_pivot


# ============================================================
# CHECK FILE
# ============================================================

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("FIELD_GROUP_FILE:", FIELD_GROUP_FILE)
print("YEAR RANGE:", START_YEAR, "-", END_YEAR)
print("TARGET_FIELD_GROUP_VALUES:", TARGET_FIELD_GROUP_VALUES if TARGET_FIELD_GROUP_VALUES else "ALL")
print("TOP_N_FOR_CHART:", TOP_N_FOR_CHART if TOP_N_FOR_CHART is not None else "ALL")

if not FIELD_GROUP_FILE.exists():
    raise FileNotFoundError(f"Missing file: {FIELD_GROUP_FILE}")

# ============================================================
# LOAD ONLY FIELD_GROUP FILE
# ============================================================

parts = []
total_rows_used = 0
start_time = time.time()

print("\nLoading ONLY degree field_group file...")

for chunk_number, chunk in enumerate(
    pd.read_csv(
        FIELD_GROUP_FILE,
        usecols=[YEAR_COL, CATEGORY_COL, VALUE_COL],
        chunksize=CHUNKSIZE
    ),
    start=1
):
    print(f"Reading chunk {chunk_number}...")

    chunk[YEAR_COL] = pd.to_numeric(chunk[YEAR_COL], errors="coerce")
    chunk[VALUE_COL] = pd.to_numeric(chunk[VALUE_COL], errors="coerce").fillna(0)

    chunk = chunk[
        (chunk[YEAR_COL] >= START_YEAR)
        & (chunk[YEAR_COL] <= END_YEAR)
    ]

    chunk[CATEGORY_COL] = clean_text(chunk[CATEGORY_COL])
    chunk = chunk.dropna(subset=[YEAR_COL, CATEGORY_COL])

    chunk = apply_text_filter(
        chunk,
        CATEGORY_COL,
        TARGET_FIELD_GROUP_VALUES
    )

    if chunk.empty:
        continue

    grouped = (
        chunk.groupby([YEAR_COL, CATEGORY_COL], as_index=False)[VALUE_COL]
        .sum()
    )

    parts.append(grouped)
    total_rows_used += len(chunk)

elapsed = time.time() - start_time

# ============================================================
# IF NO DATA
# ============================================================

if not parts:
    print("\nNo data found.")
    print("DONE.")

else:
    # ========================================================
    # COMBINE CHUNKS
    # ========================================================

    df = pd.concat(parts, ignore_index=True)

    df = (
        df.groupby([YEAR_COL, CATEGORY_COL], as_index=False)[VALUE_COL]
        .sum()
        .sort_values([YEAR_COL, CATEGORY_COL])
    )

    print("\nFinished loading.")
    print("Rows used:", f"{total_rows_used:,}")
    print("Finished in:", f"{elapsed:.2f} seconds")

    # ========================================================
    # YEARLY TABLE
    # ========================================================

    yearly_table = df.copy()

    print_table(
        title=f"YEARLY DEGREE FIELD_GROUP TABLE {START_YEAR}-{END_YEAR}",
        df=yearly_table
    )

    # ========================================================
    # TOTAL TABLE
    # ========================================================

    total_table = (
        df.groupby(CATEGORY_COL, as_index=False)[VALUE_COL]
        .sum()
        .sort_values(VALUE_COL, ascending=False)
    )

    print_table(
        title=f"TOTAL DEGREE FIELD_GROUP TABLE {START_YEAR}-{END_YEAR}",
        df=total_table
    )

    grand_total = df[VALUE_COL].sum()

    print("\n" + "=" * 100)
    print("GRAND TOTAL:", f"{grand_total:,.0f}")

    # ========================================================
    # FULL PIVOT TABLE FOR CSV
    # ========================================================

    pivot = df.pivot_table(
        index=YEAR_COL,
        columns=CATEGORY_COL,
        values=VALUE_COL,
        aggfunc="sum",
        fill_value=0
    ).sort_index()

    all_years = list(range(START_YEAR, END_YEAR + 1))
    pivot = pivot.reindex(all_years, fill_value=0)

    # Sort columns by total count, biggest first
    pivot = pivot[pivot.sum(axis=0).sort_values(ascending=False).index]

    # ========================================================
    # SAVE FULL CSV FILES
    # ========================================================

    yearly_csv = OUTPUT_DIR / f"degree_field_group_YEARLY_FULL_{START_YEAR}_{END_YEAR}.csv"
    total_csv = OUTPUT_DIR / f"degree_field_group_TOTAL_FULL_{START_YEAR}_{END_YEAR}.csv"
    pivot_csv = OUTPUT_DIR / f"degree_field_group_PIVOT_FULL_{START_YEAR}_{END_YEAR}.csv"

    yearly_table.to_csv(yearly_csv, index=False)
    total_table.to_csv(total_csv, index=False)
    pivot.to_csv(pivot_csv)

    print("\nSaved yearly CSV:", yearly_csv)
    print("Saved total CSV:", total_csv)
    print("Saved pivot CSV:", pivot_csv)

    # ========================================================
    # MAKE CLEAN CHART DATA
    # NO OTHER FIELD GROUPS
    # ========================================================

    chart_pivot = make_chart_pivot_for_picture(
        full_pivot=pivot,
        top_n=TOP_N_FOR_CHART
    )

    chart_total_csv = OUTPUT_DIR / f"degree_field_group_CHART_DATA_TOP_{TOP_N_FOR_CHART}_{START_YEAR}_{END_YEAR}.csv"
    chart_pivot.to_csv(chart_total_csv)

    all_category_count = len(pivot.columns)
    chart_category_count = len(chart_pivot.columns)
    excluded_category_count = all_category_count - chart_category_count

    print("\n" + "=" * 100)
    print("CHART CATEGORY CHECK")
    print("=" * 100)
    print("Total field groups in full data:", all_category_count)
    print("Field groups used in chart:", chart_category_count)
    print("Field groups excluded from chart:", excluded_category_count)
    print("OTHER FIELD GROUPS added?: NO")
    print("Saved chart data CSV:", chart_total_csv)

    chart_category_table = (
        chart_pivot.sum(axis=0)
        .sort_values(ascending=False)
        .reset_index()
    )

    chart_category_table.columns = [CATEGORY_COL, VALUE_COL]

    print_table(
        title=f"TOP {chart_category_count} FIELD GROUPS USED IN CHART",
        df=chart_category_table
    )

    # ========================================================
    # CHART 1: BETTER STACKED BAR
    # ========================================================

    stacked_png = OUTPUT_DIR / f"degree_field_group_TOP_{chart_category_count}_STACKED_{START_YEAR}_{END_YEAR}.png"

    ax = chart_pivot.plot(
        kind="bar",
        stacked=True,
        figsize=(18, 10),
        width=0.78,
        colormap="tab20"
    )

    ax.set_title(
        f"Top {chart_category_count} Degree Field Groups by Year ({START_YEAR}-{END_YEAR})",
        fontsize=16,
        pad=15
    )
    ax.set_xlabel("Year", fontsize=12)
    ax.set_ylabel("Degree Count", fontsize=12)
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_number))

    plt.xticks(rotation=0)

    handles, labels = ax.get_legend_handles_labels()
    labels = wrap_labels(labels, width=38)

    ax.legend(
        handles,
        labels,
        title="Field Group",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=8,
        title_fontsize=10,
        frameon=True
    )

    ax.grid(axis="y", alpha=0.25)

    plt.tight_layout()
    plt.savefig(stacked_png, dpi=220, bbox_inches="tight")
    plt.close()

    print("Saved better stacked PNG:", stacked_png)

    # ========================================================
    # CHART 2: 100% STACKED BAR
    # Easier to compare percent/share by year
    # This percent is only among the Top N shown.
    # ========================================================

    percent_png = OUTPUT_DIR / f"degree_field_group_TOP_{chart_category_count}_PERCENT_STACKED_{START_YEAR}_{END_YEAR}.png"

    percent_pivot = chart_pivot.div(chart_pivot.sum(axis=1), axis=0).fillna(0)

    ax = percent_pivot.plot(
        kind="bar",
        stacked=True,
        figsize=(18, 10),
        width=0.78,
        colormap="tab20"
    )

    ax.set_title(
        f"Top {chart_category_count} Degree Field Group Share by Year ({START_YEAR}-{END_YEAR})",
        fontsize=16,
        pad=15
    )
    ax.set_xlabel("Year", fontsize=12)
    ax.set_ylabel("Percent of Top Field Groups Shown", fontsize=12)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))

    plt.xticks(rotation=0)

    handles, labels = ax.get_legend_handles_labels()
    labels = wrap_labels(labels, width=38)

    ax.legend(
        handles,
        labels,
        title="Field Group",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=8,
        title_fontsize=10,
        frameon=True
    )

    ax.grid(axis="y", alpha=0.25)

    plt.tight_layout()
    plt.savefig(percent_png, dpi=220, bbox_inches="tight")
    plt.close()

    print("Saved percent stacked PNG:", percent_png)

    # ========================================================
    # CHART 3: TOTAL HORIZONTAL BAR
    # This is usually the cleanest chart for many categories.
    # ========================================================

    total_bar_png = OUTPUT_DIR / f"degree_field_group_TOP_{chart_category_count}_TOTAL_BAR_{START_YEAR}_{END_YEAR}.png"

    chart_totals = chart_pivot.sum(axis=0).sort_values(ascending=True)

    fig_height = max(7, len(chart_totals) * 0.55)

    ax = chart_totals.plot(
        kind="barh",
        figsize=(14, fig_height),
        colormap="tab20"
    )

    ax.set_title(
        f"Top {chart_category_count} Total Degree Count by Field Group ({START_YEAR}-{END_YEAR})",
        fontsize=16,
        pad=15
    )
    ax.set_xlabel("Degree Count", fontsize=12)
    ax.set_ylabel("Field Group", fontsize=12)
    ax.xaxis.set_major_formatter(FuncFormatter(fmt_number))

    wrapped_y_labels = wrap_labels(chart_totals.index, width=45)
    ax.set_yticklabels(wrapped_y_labels)

    ax.grid(axis="x", alpha=0.25)

    for i, value in enumerate(chart_totals.values):
        ax.text(
            value,
            i,
            f" {value:,.0f}",
            va="center",
            fontsize=8
        )

    plt.tight_layout()
    plt.savefig(total_bar_png, dpi=220, bbox_inches="tight")
    plt.close()

    print("Saved total bar PNG:", total_bar_png)

    # ========================================================
    # DONE
    # ========================================================

    print("\nDONE.")
    print("All files saved in:")
    print(OUTPUT_DIR)

# Degree number 2010-2024

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import textwrap
import time

# ============================================================
# DEGREE FIELD_GROUP ONLY
# PRINT TABLE + SAVE CSV + SAVE BETTER PNG CHARTS
#
# File columns:
# year, field_group, degree_count
#
# IMPORTANT:
# TOP_N_FOR_CHART = 12 means chart shows exactly Top 12.
# This version DOES NOT add "OTHER FIELD GROUPS".
# Also: NO percent chart.
# ============================================================

# ============================================================
# PATH SETTINGS
# ============================================================

BASE_DIR = Path.home() / "Downloads/MyProject_1"

OUTPUT_DIR = BASE_DIR / "saved_python_pic_and_tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIELD_GROUP_FILE = (
    BASE_DIR
    / "degree_raw_csv_with_year_range_in_filename"
    / "field_group_RAW_search_1970-2025_usable_2008-2024.csv"
)

# ============================================================
# CHANGE THESE SETTINGS
# ============================================================

START_YEAR = 2010
END_YEAR = 2024

CHUNKSIZE = 100_000

# Use [] for ALL field groups
# Example:
# TARGET_FIELD_GROUP_VALUES = ["Computer and Information Sciences and Support Services"]
TARGET_FIELD_GROUP_VALUES = []

YEAR_COL = "year"
CATEGORY_COL = "field_group"
VALUE_COL = "degree_count"

# Use 12 or 15 for readable chart
# Use None if you want ALL categories
TOP_N_FOR_CHART = 12

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def fmt_number(x, pos):
    try:
        return f"{int(x):,}"
    except Exception:
        return str(x)


def clean_text(s):
    return (
        s.astype(str)
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "NaN": pd.NA,
            "None": pd.NA,
            "UNKNOWN": pd.NA,
            "Unknown": pd.NA,
            "unknown": pd.NA,
        })
    )


def apply_text_filter(df, col, targets):
    if not targets:
        return df

    mask = pd.Series(False, index=df.index)
    text = df[col].astype(str).str.lower()

    for t in targets:
        mask = mask | text.str.contains(str(t).lower(), na=False)

    return df[mask]


def print_table(title, df):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    if df.empty:
        print("No data found.")
    else:
        print(df.to_string(index=False))


def wrap_labels(labels, width=34):
    return [
        "\n".join(textwrap.wrap(str(label), width=width))
        for label in labels
    ]


def make_chart_pivot_for_picture(full_pivot, top_n):
    """
    This keeps the CSV full, but makes the image easier to read.

    If top_n is None:
        chart uses all categories.

    If top_n is a number:
        chart uses ONLY Top N categories by total degree_count.
        This version DOES NOT create OTHER FIELD GROUPS.
    """

    category_totals = full_pivot.sum(axis=0).sort_values(ascending=False)

    if top_n is None:
        chart_pivot = full_pivot[category_totals.index]
        return chart_pivot

    top_categories = category_totals.head(top_n).index.tolist()

    chart_pivot = full_pivot[top_categories].copy()

    return chart_pivot


# ============================================================
# CHECK FILE
# ============================================================

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("FIELD_GROUP_FILE:", FIELD_GROUP_FILE)
print("YEAR RANGE:", START_YEAR, "-", END_YEAR)
print("TARGET_FIELD_GROUP_VALUES:", TARGET_FIELD_GROUP_VALUES if TARGET_FIELD_GROUP_VALUES else "ALL")
print("TOP_N_FOR_CHART:", TOP_N_FOR_CHART if TOP_N_FOR_CHART is not None else "ALL")

if not FIELD_GROUP_FILE.exists():
    raise FileNotFoundError(f"Missing file: {FIELD_GROUP_FILE}")

# ============================================================
# LOAD ONLY FIELD_GROUP FILE
# ============================================================

parts = []
total_rows_used = 0
start_time = time.time()

print("\nLoading ONLY degree field_group file...")

for chunk_number, chunk in enumerate(
    pd.read_csv(
        FIELD_GROUP_FILE,
        usecols=[YEAR_COL, CATEGORY_COL, VALUE_COL],
        chunksize=CHUNKSIZE
    ),
    start=1
):
    print(f"Reading chunk {chunk_number}...")

    chunk[YEAR_COL] = pd.to_numeric(chunk[YEAR_COL], errors="coerce")
    chunk[VALUE_COL] = pd.to_numeric(chunk[VALUE_COL], errors="coerce").fillna(0)

    chunk = chunk[
        (chunk[YEAR_COL] >= START_YEAR)
        & (chunk[YEAR_COL] <= END_YEAR)
    ]

    chunk[CATEGORY_COL] = clean_text(chunk[CATEGORY_COL])
    chunk = chunk.dropna(subset=[YEAR_COL, CATEGORY_COL])

    chunk = apply_text_filter(
        chunk,
        CATEGORY_COL,
        TARGET_FIELD_GROUP_VALUES
    )

    if chunk.empty:
        continue

    grouped = (
        chunk.groupby([YEAR_COL, CATEGORY_COL], as_index=False)[VALUE_COL]
        .sum()
    )

    parts.append(grouped)
    total_rows_used += len(chunk)

elapsed = time.time() - start_time

# ============================================================
# IF NO DATA
# ============================================================

if not parts:
    print("\nNo data found.")
    print("DONE.")

else:
    # ========================================================
    # COMBINE CHUNKS
    # ========================================================

    df = pd.concat(parts, ignore_index=True)

    df = (
        df.groupby([YEAR_COL, CATEGORY_COL], as_index=False)[VALUE_COL]
        .sum()
        .sort_values([YEAR_COL, CATEGORY_COL])
    )

    print("\nFinished loading.")
    print("Rows used:", f"{total_rows_used:,}")
    print("Finished in:", f"{elapsed:.2f} seconds")

    # ========================================================
    # YEARLY TABLE
    # ========================================================

    yearly_table = df.copy()

    print_table(
        title=f"YEARLY DEGREE FIELD_GROUP TABLE {START_YEAR}-{END_YEAR}",
        df=yearly_table
    )

    # ========================================================
    # TOTAL TABLE
    # ========================================================

    total_table = (
        df.groupby(CATEGORY_COL, as_index=False)[VALUE_COL]
        .sum()
        .sort_values(VALUE_COL, ascending=False)
    )

    print_table(
        title=f"TOTAL DEGREE FIELD_GROUP TABLE {START_YEAR}-{END_YEAR}",
        df=total_table
    )

    grand_total = df[VALUE_COL].sum()

    print("\n" + "=" * 100)
    print("GRAND TOTAL:", f"{grand_total:,.0f}")

    # ========================================================
    # FULL PIVOT TABLE FOR CSV
    # ========================================================

    pivot = df.pivot_table(
        index=YEAR_COL,
        columns=CATEGORY_COL,
        values=VALUE_COL,
        aggfunc="sum",
        fill_value=0
    ).sort_index()

    all_years = list(range(START_YEAR, END_YEAR + 1))
    pivot = pivot.reindex(all_years, fill_value=0)

    # Sort columns by total count, biggest first
    pivot = pivot[pivot.sum(axis=0).sort_values(ascending=False).index]

    # ========================================================
    # SAVE FULL CSV FILES
    # ========================================================

    yearly_csv = OUTPUT_DIR / f"degree_field_group_YEARLY_FULL_{START_YEAR}_{END_YEAR}.csv"
    total_csv = OUTPUT_DIR / f"degree_field_group_TOTAL_FULL_{START_YEAR}_{END_YEAR}.csv"
    pivot_csv = OUTPUT_DIR / f"degree_field_group_PIVOT_FULL_{START_YEAR}_{END_YEAR}.csv"

    yearly_table.to_csv(yearly_csv, index=False)
    total_table.to_csv(total_csv, index=False)
    pivot.to_csv(pivot_csv)

    print("\nSaved yearly CSV:", yearly_csv)
    print("Saved total CSV:", total_csv)
    print("Saved pivot CSV:", pivot_csv)

    # ========================================================
    # MAKE CLEAN CHART DATA
    # NO OTHER FIELD GROUPS
    # ========================================================

    chart_pivot = make_chart_pivot_for_picture(
        full_pivot=pivot,
        top_n=TOP_N_FOR_CHART
    )

    chart_total_csv = OUTPUT_DIR / f"degree_field_group_CHART_DATA_TOP_{TOP_N_FOR_CHART}_{START_YEAR}_{END_YEAR}.csv"
    chart_pivot.to_csv(chart_total_csv)

    all_category_count = len(pivot.columns)
    chart_category_count = len(chart_pivot.columns)
    excluded_category_count = all_category_count - chart_category_count

    print("\n" + "=" * 100)
    print("CHART CATEGORY CHECK")
    print("=" * 100)
    print("Total field groups in full data:", all_category_count)
    print("Field groups used in chart:", chart_category_count)
    print("Field groups excluded from chart:", excluded_category_count)
    print("OTHER FIELD GROUPS added?: NO")
    print("Saved chart data CSV:", chart_total_csv)

    chart_category_table = (
        chart_pivot.sum(axis=0)
        .sort_values(ascending=False)
        .reset_index()
    )

    chart_category_table.columns = [CATEGORY_COL, VALUE_COL]

    print_table(
        title=f"TOP {chart_category_count} FIELD GROUPS USED IN CHART",
        df=chart_category_table
    )

    # ========================================================
    # CHART 1: STACKED BAR (NUMBERS)
    # ========================================================

    stacked_png = OUTPUT_DIR / f"degree_field_group_TOP_{chart_category_count}_STACKED_{START_YEAR}_{END_YEAR}.png"

    ax = chart_pivot.plot(
        kind="bar",
        stacked=True,
        figsize=(18, 10),
        width=0.78,
        colormap="tab20"
    )

    ax.set_title(
        f"Top {chart_category_count} Degree Field Groups by Year ({START_YEAR}-{END_YEAR})",
        fontsize=16,
        pad=15
    )
    ax.set_xlabel("Year", fontsize=12)
    ax.set_ylabel("Degree Count", fontsize=12)
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_number))

    plt.xticks(rotation=0)

    handles, labels = ax.get_legend_handles_labels()
    labels = wrap_labels(labels, width=38)

    ax.legend(
        handles,
        labels,
        title="Field Group",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=8,
        title_fontsize=10,
        frameon=True
    )

    ax.grid(axis="y", alpha=0.25)

    plt.tight_layout()
    plt.savefig(stacked_png, dpi=220, bbox_inches="tight")
    plt.close()

    print("Saved stacked PNG:", stacked_png)

    # ========================================================
    # CHART 2: GROUPED BAR (NUMBERS)
    # Easier to compare actual values than percent
    # ========================================================

    grouped_png = OUTPUT_DIR / f"degree_field_group_TOP_{chart_category_count}_GROUPED_{START_YEAR}_{END_YEAR}.png"

    ax = chart_pivot.plot(
        kind="bar",
        stacked=False,
        figsize=(22, 10),
        width=0.85,
        colormap="tab20"
    )

    ax.set_title(
        f"Top {chart_category_count} Degree Field Groups by Year - Grouped ({START_YEAR}-{END_YEAR})",
        fontsize=16,
        pad=15
    )
    ax.set_xlabel("Year", fontsize=12)
    ax.set_ylabel("Degree Count", fontsize=12)
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_number))

    plt.xticks(rotation=0)

    handles, labels = ax.get_legend_handles_labels()
    labels = wrap_labels(labels, width=38)

    ax.legend(
        handles,
        labels,
        title="Field Group",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=8,
        title_fontsize=10,
        frameon=True
    )

    ax.grid(axis="y", alpha=0.25)

    plt.tight_layout()
    plt.savefig(grouped_png, dpi=220, bbox_inches="tight")
    plt.close()

    print("Saved grouped PNG:", grouped_png)

    # ========================================================
    # CHART 3: TOTAL HORIZONTAL BAR
    # ========================================================

    total_bar_png = OUTPUT_DIR / f"degree_field_group_TOP_{chart_category_count}_TOTAL_BAR_{START_YEAR}_{END_YEAR}.png"

    chart_totals = chart_pivot.sum(axis=0).sort_values(ascending=True)

    fig_height = max(7, len(chart_totals) * 0.55)

    ax = chart_totals.plot(
        kind="barh",
        figsize=(14, fig_height),
        colormap="tab20"
    )

    ax.set_title(
        f"Top {chart_category_count} Total Degree Count by Field Group ({START_YEAR}-{END_YEAR})",
        fontsize=16,
        pad=15
    )
    ax.set_xlabel("Degree Count", fontsize=12)
    ax.set_ylabel("Field Group", fontsize=12)
    ax.xaxis.set_major_formatter(FuncFormatter(fmt_number))

    wrapped_y_labels = wrap_labels(chart_totals.index, width=45)
    ax.set_yticklabels(wrapped_y_labels)

    ax.grid(axis="x", alpha=0.25)

    for i, value in enumerate(chart_totals.values):
        ax.text(
            value,
            i,
            f" {value:,.0f}",
            va="center",
            fontsize=8
        )

    plt.tight_layout()
    plt.savefig(total_bar_png, dpi=220, bbox_inches="tight")
    plt.close()

    print("Saved total bar PNG:", total_bar_png)

    # ========================================================
    # DONE
    # ========================================================

    print("\nDONE.")
    print("All files saved in:")
    print(OUTPUT_DIR)

# industry number 2010-2024 - too much

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# ============================================================
# COMPANY JOB GAIN / JOB LOSS BY INDUSTRY
# 2010-2024
#
# LEFT  = job loss
# RIGHT = job gain
# COLOR = industry
#
# Works with:
# 1) raw company file:
#    year, industry_name, metric_name, value
#
# 2) pre-formula job file:
#    year, connected_industry, job_count, job_loss_count
#    or
#    year, industry_name, job_gain, job_loss
# ============================================================


# ============================================================
# SETTINGS OBJECT
# Change only this part later
# ============================================================

CFG = {
    "BASE_DIR": Path.home() / "Downloads/MyProject_1",

    # Put exact file here if auto-find chooses wrong file.
    # Example:
    # "COMPANY_FILE": Path.home() / "Downloads/MyProject_1/pre_formula_job_csv/your_file.csv",
    "COMPANY_FILE": None,

    "OUTPUT_DIR_NAME": "saved_python_pic_and_tables",

    "CHUNKSIZE": 100_000,

    "START_YEAR": 2010,
    "END_YEAR": 2024,

    # Chart limit.
    # 12 = show top 12 industries only.
    # None = show all industries.
    "TOP_N_FOR_CHART": 12,

    # False = do NOT make Other bucket.
    # True = combine non-top industries into Other.
    "INCLUDE_OTHER": False,

    # Remove rows like Total / All industries.
    "EXCLUDE_TOTAL_INDUSTRY_ROWS": True,

    # Raw company format columns
    "RAW_COMPANY_COLS": [
        "year",
        "industry_name",
        "metric_name",
        "value"
    ],

    # Exclude rate / percent rows
    "RATE_WORDS": r"rate|percent|percentage|pct|share|ratio",

    # Job gain metric words
    "JOB_GAIN_WORDS": (
        r"gross job gains?|job gains?|jobs gained|job creation|jobs created|"
        r"employment gains?|hires?|hiring|job openings?"
    ),

    # Job loss metric words
    "JOB_LOSS_WORDS": (
        r"gross job losses?|job losses?|jobs lost|job destruction|"
        r"employment losses?|layoffs?|separations?"
    ),

    # Do not use net jobs as gain/loss
    "NET_WORDS": r"net",

    "FIGSIZE": (16, 9),
    "DPI": 300,
    "COLOR_MAP": "tab20",

    "SAVE_CHART_NAME": "company_job_gain_loss_by_industry_{start}_{end}_top{top}.png",
    "SAVE_FULL_TABLE_NAME": "company_job_gain_loss_by_year_industry_{start}_{end}_FULL.csv",
    "SAVE_CHART_TABLE_NAME": "company_job_gain_loss_by_year_industry_{start}_{end}_CHART.csv",
    "SAVE_SUMMARY_TABLE_NAME": "company_job_gain_loss_by_industry_summary_{start}_{end}.csv",
}


# ============================================================
# VARIABLES FROM SETTINGS OBJECT
# ============================================================

BASE_DIR = CFG["BASE_DIR"]
OUTPUT_DIR = BASE_DIR / CFG["OUTPUT_DIR_NAME"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMPANY_FILE = CFG["COMPANY_FILE"]

CHUNKSIZE = CFG["CHUNKSIZE"]
START_YEAR = CFG["START_YEAR"]
END_YEAR = CFG["END_YEAR"]
TOP_N_FOR_CHART = CFG["TOP_N_FOR_CHART"]
INCLUDE_OTHER = CFG["INCLUDE_OTHER"]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_name(x):
    return str(x).strip()


def norm_col(x):
    return str(x).strip().lower().replace(" ", "_")


def find_col(columns, possible_names):
    col_map = {norm_col(c): c for c in columns}

    for name in possible_names:
        key = norm_col(name)
        if key in col_map:
            return col_map[key]

    return None


def inspect_csv_file(path):
    """
    Detect whether file is raw company format or pre-formula job format.
    """
    try:
        columns = pd.read_csv(path, nrows=0, low_memory=False).columns.tolist()
    except Exception:
        return None

    year_col = find_col(columns, ["year"])

    raw_industry_col = find_col(columns, ["industry_name"])
    raw_metric_col = find_col(columns, ["metric_name"])
    raw_value_col = find_col(columns, ["value"])

    pre_industry_col = find_col(
        columns,
        [
            "industry_name",
            "connected_industry",
            "industry",
            "industry_group"
        ]
    )

    pre_gain_col = find_col(
        columns,
        [
            "job_gain",
            "job_gains",
            "job_gain_count",
            "jobs_gained",
            "job_count"
        ]
    )

    pre_loss_col = find_col(
        columns,
        [
            "job_loss",
            "job_losses",
            "job_loss_count",
            "jobs_lost"
        ]
    )

    if year_col and raw_industry_col and raw_metric_col and raw_value_col:
        return {
            "format": "raw_company",
            "year_col": year_col,
            "industry_col": raw_industry_col,
            "metric_col": raw_metric_col,
            "value_col": raw_value_col,
            "columns": columns
        }

    if year_col and pre_industry_col and pre_gain_col and pre_loss_col:
        return {
            "format": "pre_formula_job",
            "year_col": year_col,
            "industry_col": pre_industry_col,
            "gain_col": pre_gain_col,
            "loss_col": pre_loss_col,
            "columns": columns
        }

    return None


def score_company_file(path):
    """
    Give higher score to likely company/job CSV files.
    """
    info = inspect_csv_file(path)
    if info is None:
        return -999, None

    name = path.name.lower()
    folder = str(path.parent).lower()

    score = 0

    if info["format"] == "raw_company":
        score += 100

    if info["format"] == "pre_formula_job":
        score += 90

    if "pre_formula_job_csv" in folder:
        score += 60

    if "job" in name:
        score += 40

    if "company" in name:
        score += 30

    if "business" in name:
        score += 30

    if "startup" in name:
        score -= 20

    if "degree" in folder or "degree" in name:
        score -= 100

    return score, info


def auto_find_company_file(base_dir):
    """
    Auto-find best company/job CSV inside MyProject_1.
    """
    if not base_dir.exists():
        raise FileNotFoundError(f"Missing BASE_DIR: {base_dir}")

    csv_files = list(base_dir.rglob("*.csv"))

    if not csv_files:
        raise FileNotFoundError(f"No CSV files found inside: {base_dir}")

    scored = []

    for path in csv_files:
        score, info = score_company_file(path)
        if score > 0:
            scored.append((score, path, info))

    if not scored:
        print("\nCSV files found, but none matched company/job format:")
        for p in csv_files[:50]:
            print(p)

        raise ValueError(
            "No usable company/job CSV found. "
            "Set CFG['COMPANY_FILE'] to the exact CSV path."
        )

    scored = sorted(scored, key=lambda x: x[0], reverse=True)

    print("\nPossible company/job files found:")
    for score, path, info in scored[:10]:
        print(f"score={score:>3} | format={info['format']} | {path}")

    best_score, best_path, best_info = scored[0]

    print("\nUsing this company/job file:")
    print(best_path)
    print("Detected format:", best_info["format"])

    return best_path, best_info


def filter_industry_rows(df, industry_col):
    df[industry_col] = df[industry_col].astype(str).str.strip()

    df = df[
        df[industry_col].notna()
        & (df[industry_col] != "")
        & (~df[industry_col].str.lower().isin(["nan", "none", "unknown"]))
    ].copy()

    if CFG["EXCLUDE_TOTAL_INDUSTRY_ROWS"]:
        total_mask = df[industry_col].str.lower().str.contains(
            r"^total$|total,|all industries",
            na=False,
            regex=True
        )
        df = df[~total_mask].copy()

    return df


def fmt_abs_number(x, pos):
    try:
        return f"{abs(int(x)):,}"
    except Exception:
        return str(x)


# ============================================================
# FIND COMPANY FILE
# ============================================================

print("Checking BASE_DIR...")
print(BASE_DIR)

if COMPANY_FILE is None:
    COMPANY_FILE, FILE_INFO = auto_find_company_file(BASE_DIR)
else:
    COMPANY_FILE = Path(COMPANY_FILE)

    if not COMPANY_FILE.exists():
        raise FileNotFoundError(f"Missing Company file: {COMPANY_FILE}")

    FILE_INFO = inspect_csv_file(COMPANY_FILE)

    if FILE_INFO is None:
        raise ValueError(
            f"This file does not look like raw company or pre-formula job CSV:\n{COMPANY_FILE}"
        )

    print("\nUsing manually selected company file:")
    print(COMPANY_FILE)
    print("Detected format:", FILE_INFO["format"])


# ============================================================
# LOAD RAW COMPANY FORMAT
# year, industry_name, metric_name, value
# ============================================================

parts = []
matched_job_gain_metric_names = set()
matched_job_loss_metric_names = set()

print(f"\nLoading Company job data from {START_YEAR} to {END_YEAR}...")

if FILE_INFO["format"] == "raw_company":

    usecols = [
        FILE_INFO["year_col"],
        FILE_INFO["industry_col"],
        FILE_INFO["metric_col"],
        FILE_INFO["value_col"]
    ]

    chunk_count = 0

    for chunk in pd.read_csv(
        COMPANY_FILE,
        usecols=usecols,
        chunksize=CHUNKSIZE,
        low_memory=False
    ):
        chunk_count += 1
        print(f"Loading chunk {chunk_count}... rows: {len(chunk):,}")

        year_col = FILE_INFO["year_col"]
        industry_col = FILE_INFO["industry_col"]
        metric_col = FILE_INFO["metric_col"]
        value_col = FILE_INFO["value_col"]

        chunk[year_col] = pd.to_numeric(chunk[year_col], errors="coerce")
        chunk[value_col] = pd.to_numeric(chunk[value_col], errors="coerce")

        chunk = chunk.dropna(subset=[year_col, industry_col, metric_col, value_col])
        chunk[year_col] = chunk[year_col].astype(int)

        chunk = chunk[
            (chunk[year_col] >= START_YEAR)
            & (chunk[year_col] <= END_YEAR)
        ].copy()

        if chunk.empty:
            continue

        chunk = filter_industry_rows(chunk, industry_col)

        if chunk.empty:
            continue

        metric = chunk[metric_col].astype(str).str.lower()

        not_rate_mask = ~metric.str.contains(
            CFG["RATE_WORDS"],
            na=False,
            regex=True
        )

        not_net_mask = ~metric.str.contains(
            CFG["NET_WORDS"],
            na=False,
            regex=True
        )

        job_gain_mask = (
            metric.str.contains(CFG["JOB_GAIN_WORDS"], na=False, regex=True)
            & not_rate_mask
            & not_net_mask
        )

        job_loss_mask = (
            metric.str.contains(CFG["JOB_LOSS_WORDS"], na=False, regex=True)
            & not_rate_mask
            & not_net_mask
        )

        matched_job_gain_metric_names.update(
            chunk.loc[job_gain_mask, metric_col].dropna().unique()
        )

        matched_job_loss_metric_names.update(
            chunk.loc[job_loss_mask, metric_col].dropna().unique()
        )

        chunk["job_gain"] = np.where(job_gain_mask, chunk[value_col], 0)
        chunk["job_loss"] = np.where(job_loss_mask, chunk[value_col], 0)

        part = (
            chunk
            .groupby([year_col, industry_col], dropna=False)[["job_gain", "job_loss"]]
            .sum()
            .reset_index()
            .rename(columns={
                year_col: "year",
                industry_col: "industry_name"
            })
        )

        parts.append(part)


# ============================================================
# LOAD PRE-FORMULA JOB FORMAT
# year, connected_industry, job_count, job_loss_count
# ============================================================

elif FILE_INFO["format"] == "pre_formula_job":

    usecols = [
        FILE_INFO["year_col"],
        FILE_INFO["industry_col"],
        FILE_INFO["gain_col"],
        FILE_INFO["loss_col"]
    ]

    chunk_count = 0

    for chunk in pd.read_csv(
        COMPANY_FILE,
        usecols=usecols,
        chunksize=CHUNKSIZE,
        low_memory=False
    ):
        chunk_count += 1
        print(f"Loading chunk {chunk_count}... rows: {len(chunk):,}")

        year_col = FILE_INFO["year_col"]
        industry_col = FILE_INFO["industry_col"]
        gain_col = FILE_INFO["gain_col"]
        loss_col = FILE_INFO["loss_col"]

        chunk[year_col] = pd.to_numeric(chunk[year_col], errors="coerce")
        chunk[gain_col] = pd.to_numeric(chunk[gain_col], errors="coerce")
        chunk[loss_col] = pd.to_numeric(chunk[loss_col], errors="coerce")

        chunk = chunk.dropna(subset=[year_col, industry_col])
        chunk[year_col] = chunk[year_col].astype(int)

        chunk = chunk[
            (chunk[year_col] >= START_YEAR)
            & (chunk[year_col] <= END_YEAR)
        ].copy()

        if chunk.empty:
            continue

        chunk = filter_industry_rows(chunk, industry_col)

        if chunk.empty:
            continue

        chunk[gain_col] = chunk[gain_col].fillna(0)
        chunk[loss_col] = chunk[loss_col].fillna(0)

        # Important:
        # Some pre-formula files repeat the same industry job values
        # for multiple degree fields. This removes exact duplicate rows.
        chunk = chunk.drop_duplicates(
            subset=[year_col, industry_col, gain_col, loss_col]
        )

        part = (
            chunk
            .groupby([year_col, industry_col], dropna=False)[[gain_col, loss_col]]
            .sum()
            .reset_index()
            .rename(columns={
                year_col: "year",
                industry_col: "industry_name",
                gain_col: "job_gain",
                loss_col: "job_loss"
            })
        )

        parts.append(part)


print("\nFinished loading data.")


# ============================================================
# COMBINE DATA
# ============================================================

if not parts:
    raise ValueError(f"No Company job data found from {START_YEAR} to {END_YEAR}.")

chart_data_full = pd.concat(parts, ignore_index=True)

chart_data_full = (
    chart_data_full
    .groupby(["year", "industry_name"], dropna=False)[["job_gain", "job_loss"]]
    .sum()
    .reset_index()
)

chart_data_full["job_total"] = (
    chart_data_full["job_gain"].abs()
    + chart_data_full["job_loss"].abs()
)

chart_data_full = chart_data_full[chart_data_full["job_total"] > 0].copy()

if chart_data_full.empty:
    raise ValueError(
        f"Data loaded, but no job gain/loss rows matched from {START_YEAR} to {END_YEAR}."
    )


# ============================================================
# PRINT MATCHED RAW METRIC NAMES
# ============================================================

if FILE_INFO["format"] == "raw_company":
    print("\nMetric names used for job_gain:")
    for x in sorted(matched_job_gain_metric_names):
        print("-", x)

    print("\nMetric names used for job_loss:")
    for x in sorted(matched_job_loss_metric_names):
        print("-", x)


# ============================================================
# SAVE FULL TABLE
# ============================================================

full_table_path = OUTPUT_DIR / CFG["SAVE_FULL_TABLE_NAME"].format(
    start=START_YEAR,
    end=END_YEAR
)

chart_data_full.to_csv(full_table_path, index=False)

print("\nSaved FULL year + industry table:")
print(full_table_path)


# ============================================================
# INDUSTRY SUMMARY TABLE
# ============================================================

summary_table = (
    chart_data_full
    .groupby("industry_name", dropna=False)[["job_gain", "job_loss", "job_total"]]
    .sum()
    .reset_index()
    .sort_values("job_total", ascending=False)
)

summary_table_path = OUTPUT_DIR / CFG["SAVE_SUMMARY_TABLE_NAME"].format(
    start=START_YEAR,
    end=END_YEAR
)

summary_table.to_csv(summary_table_path, index=False)

print("\nIndustry summary table:")
print(summary_table.head(30).to_string(index=False))

print("\nSaved industry summary table:")
print(summary_table_path)


# ============================================================
# KEEP TOP INDUSTRIES FOR CHART
# ============================================================

chart_data = chart_data_full.copy()

if TOP_N_FOR_CHART is not None:
    top_industries = (
        chart_data
        .groupby("industry_name")["job_total"]
        .sum()
        .sort_values(ascending=False)
        .head(TOP_N_FOR_CHART)
        .index
        .tolist()
    )

    if INCLUDE_OTHER:
        chart_data["industry_name"] = np.where(
            chart_data["industry_name"].isin(top_industries),
            chart_data["industry_name"],
            "Other"
        )

        chart_data = (
            chart_data
            .groupby(["year", "industry_name"], dropna=False)[["job_gain", "job_loss"]]
            .sum()
            .reset_index()
        )

        chart_data["job_total"] = chart_data["job_gain"].abs() + chart_data["job_loss"].abs()

    else:
        chart_data = chart_data[
            chart_data["industry_name"].isin(top_industries)
        ].copy()


# ============================================================
# SAVE CHART TABLE
# ============================================================

chart_table_path = OUTPUT_DIR / CFG["SAVE_CHART_TABLE_NAME"].format(
    start=START_YEAR,
    end=END_YEAR
)

chart_data.to_csv(chart_table_path, index=False)

print("\nSaved CHART table:")
print(chart_table_path)


# ============================================================
# PIVOT FOR STACKED BAR
# ============================================================

industry_order = (
    chart_data
    .groupby("industry_name")["job_total"]
    .sum()
    .sort_values(ascending=False)
    .index
    .tolist()
)

all_years = list(range(START_YEAR, END_YEAR + 1))

gain_pivot = (
    chart_data
    .pivot_table(
        index="year",
        columns="industry_name",
        values="job_gain",
        aggfunc="sum",
        fill_value=0
    )
    .reindex(index=all_years, columns=industry_order, fill_value=0)
)

loss_pivot = (
    chart_data
    .pivot_table(
        index="year",
        columns="industry_name",
        values="job_loss",
        aggfunc="sum",
        fill_value=0
    )
    .reindex(index=all_years, columns=industry_order, fill_value=0)
)

print("\nYears on chart:")
print(all_years)

print("\nIndustries on chart:")
for industry in industry_order:
    print("-", industry)


# ============================================================
# YEAR TOTAL TABLE
# ============================================================

year_total_table = (
    chart_data_full
    .groupby("year", dropna=False)[["job_gain", "job_loss", "job_total"]]
    .sum()
    .reset_index()
    .sort_values("year")
)

print("\nYear total table:")
print(year_total_table.to_string(index=False))


# ============================================================
# PLOT DIVERGING STACKED HORIZONTAL BAR CHART
# ============================================================

plt.figure(figsize=CFG["FIGSIZE"])

years = gain_pivot.index.astype(str)
y_pos = np.arange(len(years))

colors = getattr(plt.cm, CFG["COLOR_MAP"])(
    np.linspace(0, 1, len(industry_order))
)

# RIGHT SIDE = job gain
right_base = np.zeros(len(gain_pivot))

for i, industry in enumerate(industry_order):
    values = gain_pivot[industry].values

    plt.barh(
        y_pos,
        values,
        left=right_base,
        label=industry,
        color=colors[i]
    )

    right_base += values


# LEFT SIDE = job loss
left_base = np.zeros(len(loss_pivot))

for i, industry in enumerate(industry_order):
    values = loss_pivot[industry].values

    plt.barh(
        y_pos,
        -values,
        left=-left_base,
        color=colors[i]
    )

    left_base += values


plt.axvline(0, color="black", linewidth=1)

plt.yticks(y_pos, years)

top_text = "ALL" if TOP_N_FOR_CHART is None else f"TOP {TOP_N_FOR_CHART}"

plt.title(
    f"{START_YEAR}-{END_YEAR} {top_text} Industries: Job Loss vs Job Gain by Year"
)

plt.xlabel("Job loss on left   |   Job gain on right")
plt.ylabel("Year")

plt.gca().xaxis.set_major_formatter(FuncFormatter(fmt_abs_number))

plt.grid(axis="x", linestyle="--", alpha=0.4)

max_side = max(left_base.max(), right_base.max())

if max_side > 0:
    plt.xlim(-max_side * 1.12, max_side * 1.12)

plt.legend(
    title="Industry",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()

top_name = "ALL" if TOP_N_FOR_CHART is None else str(TOP_N_FOR_CHART)

chart_path = OUTPUT_DIR / CFG["SAVE_CHART_NAME"].format(
    start=START_YEAR,
    end=END_YEAR,
    top=top_name
)

plt.savefig(
    chart_path,
    dpi=CFG["DPI"],
    bbox_inches="tight"
)

plt.close()


# ============================================================
# DONE
# ============================================================

print("\nALL DONE.")
print("Saved chart here:")
print(chart_path)

print("\nSaved tables here:")
print(full_table_path)
print(chart_table_path)
print(summary_table_path)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# ============================================================
# COMPANY JOB GAIN / JOB LOSS BY INDUSTRY
# 2010-2024
#
# BASE FOLDER:
# Downloads/MyProject_1
#
# CHART:
# Y axis = year
# Right side = job gain
# Left side = job loss
# Color = industry
#
# TABLES:
# 1) full year + industry table
# 2) chart top industry table
# 3) year total table
# 4) industry total table
# 5) note text file
#
# IMPORTANT NOTE:
# This removes aggregate industry rows:
# - Total private
# - Service-providing
# - Goods-producing
#
# Reason:
# These rows already include many smaller industries.
# If we keep them together with detailed industries, the chart double-counts.
#
# Job activity total = job gain + job loss
# Net job change = job gain - job loss
# ============================================================


# ============================================================
# SETTINGS OBJECT
# Change only this part later
# ============================================================

CFG = {
    "BASE_DIR": Path.home() / "Downloads/MyProject_1",

    # Put exact file here if auto-find chooses wrong file.
    # Example:
    # "COMPANY_FILE": Path.home() / "Downloads/MyProject_1/pre_formula_job_csv/your_file.csv",
    "COMPANY_FILE": None,

    "OUTPUT_DIR_NAME": "saved_python_pic_and_tables",

    "CHUNKSIZE": 100_000,

    "START_YEAR": 2010,
    "END_YEAR": 2024,

    # 12 = chart shows top 12 real industries
    # None = chart shows all real industries
    "TOP_N_FOR_CHART": 12,

    # False = do not add Other
    # True = combine non-top industries into Other
    "INCLUDE_OTHER": False,

    # Remove aggregate rows to avoid double-counting
    "REMOVE_AGGREGATE_INDUSTRIES": True,

    "AGGREGATE_INDUSTRIES_TO_REMOVE": [
        "Total private",
        "Service-providing",
        "Goods-producing",
    ],

    # Remove unknown / blank values
    "REMOVE_UNKNOWN": True,

    # Raw company format columns
    "RAW_COMPANY_COLS": [
        "year",
        "industry_name",
        "metric_name",
        "value"
    ],

    # Exclude rate / percent rows
    "RATE_WORDS": r"rate|percent|percentage|pct|share|ratio",

    # Do not use net jobs as gain/loss
    "NET_WORDS": r"net",

    # Job gain metric words
    "JOB_GAIN_WORDS": (
        r"gross job gains?|job gains?|jobs gained|job creation|jobs created|"
        r"employment gains?|hires?|hiring|job openings?"
    ),

    # Job loss metric words
    "JOB_LOSS_WORDS": (
        r"gross job losses?|job losses?|jobs lost|job destruction|"
        r"employment losses?|layoffs?|separations?"
    ),

    "FIGSIZE": (16, 9),
    "DPI": 300,
    "COLOR_MAP": "tab20",

    "SAVE_CHART_NAME": "company_job_gain_loss_by_industry_{start}_{end}_top{top}.png",

    "SAVE_FULL_TABLE_NAME": "company_job_gain_loss_by_year_industry_{start}_{end}_FULL.csv",

    "SAVE_CHART_TABLE_NAME": "company_job_gain_loss_by_year_industry_{start}_{end}_CHART_TOP{top}.csv",

    "SAVE_YEAR_TOTAL_TABLE_NAME": "company_job_gain_loss_YEAR_TOTAL_{start}_{end}.csv",

    "SAVE_INDUSTRY_TOTAL_TABLE_NAME": "company_job_gain_loss_INDUSTRY_TOTAL_{start}_{end}.csv",

    "SAVE_NOTE_NAME": "company_job_gain_loss_NOTE_{start}_{end}.txt",
}


# ============================================================
# VARIABLES FROM SETTINGS OBJECT
# ============================================================

BASE_DIR = CFG["BASE_DIR"]
OUTPUT_DIR = BASE_DIR / CFG["OUTPUT_DIR_NAME"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMPANY_FILE = CFG["COMPANY_FILE"]

CHUNKSIZE = CFG["CHUNKSIZE"]
START_YEAR = CFG["START_YEAR"]
END_YEAR = CFG["END_YEAR"]
TOP_N_FOR_CHART = CFG["TOP_N_FOR_CHART"]
INCLUDE_OTHER = CFG["INCLUDE_OTHER"]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def norm_col(x):
    return str(x).strip().lower().replace(" ", "_")


def find_col(columns, possible_names):
    col_map = {norm_col(c): c for c in columns}

    for name in possible_names:
        key = norm_col(name)
        if key in col_map:
            return col_map[key]

    return None


def fmt_number(x):
    try:
        return f"{int(round(x)):,}"
    except Exception:
        return str(x)


def fmt_abs_number_axis(x, pos):
    try:
        return f"{abs(int(x)):,}"
    except Exception:
        return str(x)


def inspect_csv_file(path):
    """
    Detect whether file is:
    1) raw company format:
       year, industry_name, metric_name, value

    2) pre-formula job format:
       year, connected_industry, job_count, job_loss_count
       or similar
    """

    try:
        columns = pd.read_csv(path, nrows=0, low_memory=False).columns.tolist()
    except Exception:
        return None

    year_col = find_col(columns, ["year"])

    raw_industry_col = find_col(columns, ["industry_name"])
    raw_metric_col = find_col(columns, ["metric_name"])
    raw_value_col = find_col(columns, ["value"])

    pre_industry_col = find_col(
        columns,
        [
            "industry_name",
            "connected_industry",
            "industry",
            "industry_group"
        ]
    )

    pre_gain_col = find_col(
        columns,
        [
            "job_gain",
            "job_gains",
            "job_gain_count",
            "jobs_gained",
            "job_count"
        ]
    )

    pre_loss_col = find_col(
        columns,
        [
            "job_loss",
            "job_losses",
            "job_loss_count",
            "jobs_lost"
        ]
    )

    if year_col and raw_industry_col and raw_metric_col and raw_value_col:
        return {
            "format": "raw_company",
            "year_col": year_col,
            "industry_col": raw_industry_col,
            "metric_col": raw_metric_col,
            "value_col": raw_value_col,
            "columns": columns
        }

    if year_col and pre_industry_col and pre_gain_col and pre_loss_col:
        return {
            "format": "pre_formula_job",
            "year_col": year_col,
            "industry_col": pre_industry_col,
            "gain_col": pre_gain_col,
            "loss_col": pre_loss_col,
            "columns": columns
        }

    return None


def score_company_file(path):
    """
    Auto-find best company/job CSV inside MyProject_1.
    """

    info = inspect_csv_file(path)

    if info is None:
        return -999, None

    name = path.name.lower()
    folder = str(path.parent).lower()

    score = 0

    if info["format"] == "raw_company":
        score += 100

    if info["format"] == "pre_formula_job":
        score += 90

    if "pre_formula_job_csv" in folder:
        score += 60

    if "job" in name:
        score += 40

    if "company" in name:
        score += 30

    if "business" in name:
        score += 30

    if "startup" in name:
        score -= 20

    if "degree" in folder or "degree" in name:
        score -= 100

    return score, info


def auto_find_company_file(base_dir):
    """
    Auto-find company/job CSV inside Downloads/MyProject_1.
    """

    if not base_dir.exists():
        raise FileNotFoundError(f"Missing BASE_DIR: {base_dir}")

    csv_files = list(base_dir.rglob("*.csv"))

    if not csv_files:
        raise FileNotFoundError(f"No CSV files found inside: {base_dir}")

    scored = []

    for path in csv_files:
        score, info = score_company_file(path)
        if score > 0:
            scored.append((score, path, info))

    if not scored:
        print("\nCSV files found, but none matched company/job format:")
        for p in csv_files[:50]:
            print(p)

        raise ValueError(
            "No usable company/job CSV found. "
            "Set CFG['COMPANY_FILE'] to the exact CSV path."
        )

    scored = sorted(scored, key=lambda x: x[0], reverse=True)

    print("\nPossible company/job files found:")
    for score, path, info in scored[:10]:
        print(f"score={score:>3} | format={info['format']} | {path}")

    best_score, best_path, best_info = scored[0]

    print("\nUsing this company/job file:")
    print(best_path)
    print("Detected format:", best_info["format"])

    return best_path, best_info


def clean_industry_rows(df, industry_col):
    """
    Remove blank, unknown, and aggregate industry rows.
    """

    df[industry_col] = df[industry_col].astype(str).str.strip()

    if CFG["REMOVE_UNKNOWN"]:
        df = df[
            df[industry_col].notna()
            & (df[industry_col] != "")
            & (~df[industry_col].str.lower().isin(["nan", "none", "unknown"]))
        ].copy()

    if CFG["REMOVE_AGGREGATE_INDUSTRIES"]:
        df = df[
            ~df[industry_col].isin(CFG["AGGREGATE_INDUSTRIES_TO_REMOVE"])
        ].copy()

    return df


# ============================================================
# FIND COMPANY FILE
# ============================================================

print("Checking BASE_DIR...")
print(BASE_DIR)

if COMPANY_FILE is None:
    COMPANY_FILE, FILE_INFO = auto_find_company_file(BASE_DIR)

else:
    COMPANY_FILE = Path(COMPANY_FILE)

    if not COMPANY_FILE.exists():
        raise FileNotFoundError(f"Missing Company file: {COMPANY_FILE}")

    FILE_INFO = inspect_csv_file(COMPANY_FILE)

    if FILE_INFO is None:
        raise ValueError(
            f"This file does not look like raw company or pre-formula job CSV:\n{COMPANY_FILE}"
        )

    print("\nUsing manually selected company file:")
    print(COMPANY_FILE)
    print("Detected format:", FILE_INFO["format"])


# ============================================================
# LOAD DATA
# ============================================================

parts = []

matched_job_gain_metric_names = set()
matched_job_loss_metric_names = set()

print(f"\nLoading Company job data from {START_YEAR} to {END_YEAR}...")


# ============================================================
# CASE 1: RAW COMPANY FORMAT
# year, industry_name, metric_name, value
# ============================================================

if FILE_INFO["format"] == "raw_company":

    year_col = FILE_INFO["year_col"]
    industry_col = FILE_INFO["industry_col"]
    metric_col = FILE_INFO["metric_col"]
    value_col = FILE_INFO["value_col"]

    usecols = [
        year_col,
        industry_col,
        metric_col,
        value_col
    ]

    chunk_count = 0

    for chunk in pd.read_csv(
        COMPANY_FILE,
        usecols=usecols,
        chunksize=CHUNKSIZE,
        low_memory=False
    ):
        chunk_count += 1
        print(f"Loading chunk {chunk_count}... rows: {len(chunk):,}")

        chunk[year_col] = pd.to_numeric(chunk[year_col], errors="coerce")
        chunk[value_col] = pd.to_numeric(chunk[value_col], errors="coerce")

        chunk = chunk.dropna(subset=[year_col, industry_col, metric_col, value_col])
        chunk[year_col] = chunk[year_col].astype(int)

        chunk = chunk[
            (chunk[year_col] >= START_YEAR)
            & (chunk[year_col] <= END_YEAR)
        ].copy()

        if chunk.empty:
            continue

        chunk = clean_industry_rows(chunk, industry_col)

        if chunk.empty:
            continue

        metric = chunk[metric_col].astype(str).str.lower()

        not_rate_mask = ~metric.str.contains(
            CFG["RATE_WORDS"],
            na=False,
            regex=True
        )

        not_net_mask = ~metric.str.contains(
            CFG["NET_WORDS"],
            na=False,
            regex=True
        )

        job_gain_mask = (
            metric.str.contains(CFG["JOB_GAIN_WORDS"], na=False, regex=True)
            & not_rate_mask
            & not_net_mask
        )

        job_loss_mask = (
            metric.str.contains(CFG["JOB_LOSS_WORDS"], na=False, regex=True)
            & not_rate_mask
            & not_net_mask
        )

        matched_job_gain_metric_names.update(
            chunk.loc[job_gain_mask, metric_col].dropna().unique()
        )

        matched_job_loss_metric_names.update(
            chunk.loc[job_loss_mask, metric_col].dropna().unique()
        )

        chunk["job_gain"] = np.where(job_gain_mask, chunk[value_col], 0)
        chunk["job_loss"] = np.where(job_loss_mask, chunk[value_col], 0)

        part = (
            chunk
            .groupby([year_col, industry_col], dropna=False)[["job_gain", "job_loss"]]
            .sum()
            .reset_index()
            .rename(columns={
                year_col: "year",
                industry_col: "industry_name"
            })
        )

        parts.append(part)


# ============================================================
# CASE 2: PRE-FORMULA JOB FORMAT
# year, connected_industry, job_count, job_loss_count
# ============================================================

elif FILE_INFO["format"] == "pre_formula_job":

    year_col = FILE_INFO["year_col"]
    industry_col = FILE_INFO["industry_col"]
    gain_col = FILE_INFO["gain_col"]
    loss_col = FILE_INFO["loss_col"]

    usecols = [
        year_col,
        industry_col,
        gain_col,
        loss_col
    ]

    chunk_count = 0

    for chunk in pd.read_csv(
        COMPANY_FILE,
        usecols=usecols,
        chunksize=CHUNKSIZE,
        low_memory=False
    ):
        chunk_count += 1
        print(f"Loading chunk {chunk_count}... rows: {len(chunk):,}")

        chunk[year_col] = pd.to_numeric(chunk[year_col], errors="coerce")
        chunk[gain_col] = pd.to_numeric(chunk[gain_col], errors="coerce")
        chunk[loss_col] = pd.to_numeric(chunk[loss_col], errors="coerce")

        chunk = chunk.dropna(subset=[year_col, industry_col])
        chunk[year_col] = chunk[year_col].astype(int)

        chunk = chunk[
            (chunk[year_col] >= START_YEAR)
            & (chunk[year_col] <= END_YEAR)
        ].copy()

        if chunk.empty:
            continue

        chunk = clean_industry_rows(chunk, industry_col)

        if chunk.empty:
            continue

        chunk[gain_col] = chunk[gain_col].fillna(0)
        chunk[loss_col] = chunk[loss_col].fillna(0)

        # Important:
        # Some pre-formula files repeat the same company job value
        # for multiple degree fields.
        # This removes exact duplicate job rows.
        chunk = chunk.drop_duplicates(
            subset=[year_col, industry_col, gain_col, loss_col]
        )

        part = (
            chunk
            .groupby([year_col, industry_col], dropna=False)[[gain_col, loss_col]]
            .sum()
            .reset_index()
            .rename(columns={
                year_col: "year",
                industry_col: "industry_name",
                gain_col: "job_gain",
                loss_col: "job_loss"
            })
        )

        parts.append(part)


print("\nFinished loading data.")


# ============================================================
# COMBINE DATA
# ============================================================

if not parts:
    raise ValueError(f"No Company job data found from {START_YEAR} to {END_YEAR}.")

full_table = pd.concat(parts, ignore_index=True)

full_table = (
    full_table
    .groupby(["year", "industry_name"], dropna=False)[["job_gain", "job_loss"]]
    .sum()
    .reset_index()
)

full_table["job_gain"] = full_table["job_gain"].fillna(0)
full_table["job_loss"] = full_table["job_loss"].fillna(0)

full_table["job_activity_total"] = (
    full_table["job_gain"].abs()
    + full_table["job_loss"].abs()
)

full_table["net_job_change"] = (
    full_table["job_gain"]
    - full_table["job_loss"]
)

full_table = full_table[full_table["job_activity_total"] > 0].copy()

if full_table.empty:
    raise ValueError(
        f"Data loaded, but no job gain/loss rows matched from {START_YEAR} to {END_YEAR}."
    )


# ============================================================
# PRINT RAW METRIC NAMES USED
# ============================================================

if FILE_INFO["format"] == "raw_company":

    print("\nMetric names used for job_gain:")
    for x in sorted(matched_job_gain_metric_names):
        print("-", x)

    print("\nMetric names used for job_loss:")
    for x in sorted(matched_job_loss_metric_names):
        print("-", x)


# ============================================================
# TOTAL TABLE 1: YEAR TOTAL
# Like degree total count by year
# ============================================================

year_total_table = (
    full_table
    .groupby("year", dropna=False)[["job_gain", "job_loss", "job_activity_total", "net_job_change"]]
    .sum()
    .reset_index()
    .sort_values("year")
)

year_total_table = year_total_table.rename(columns={
    "job_gain": "total_job_gain",
    "job_loss": "total_job_loss",
    "job_activity_total": "total_job_activity_gain_plus_loss",
    "net_job_change": "net_job_change_gain_minus_loss"
})


# ============================================================
# TOTAL TABLE 2: INDUSTRY TOTAL
# Like degree total count by category
# ============================================================

industry_total_table = (
    full_table
    .groupby("industry_name", dropna=False)[["job_gain", "job_loss", "job_activity_total", "net_job_change"]]
    .sum()
    .reset_index()
    .sort_values("job_activity_total", ascending=False)
)

industry_total_table = industry_total_table.rename(columns={
    "job_gain": "range_total_job_gain",
    "job_loss": "range_total_job_loss",
    "job_activity_total": "range_total_job_activity_gain_plus_loss",
    "net_job_change": "range_net_job_change_gain_minus_loss"
})


# ============================================================
# OVERALL TOTAL COUNT
# ============================================================

overall_total_job_gain = full_table["job_gain"].sum()
overall_total_job_loss = full_table["job_loss"].sum()
overall_job_activity_total = full_table["job_activity_total"].sum()
overall_net_job_change = full_table["net_job_change"].sum()

unique_industry_count = full_table["industry_name"].nunique()
year_count = full_table["year"].nunique()


# ============================================================
# PRINT TOTALS
# ============================================================

print("\n============================================================")
print("OVERALL TOTAL COUNT")
print("============================================================")
print("Year range:", START_YEAR, "-", END_YEAR)
print("Years found:", year_count)
print("Unique industries:", unique_industry_count)
print("Total job gain:", fmt_number(overall_total_job_gain))
print("Total job loss:", fmt_number(overall_total_job_loss))
print("Total job activity gain + loss:", fmt_number(overall_job_activity_total))
print("Net job change gain - loss:", fmt_number(overall_net_job_change))

print("\n============================================================")
print("YEAR TOTAL TABLE")
print("============================================================")
print(year_total_table.to_string(index=False))

print("\n============================================================")
print("INDUSTRY TOTAL TABLE")
print("============================================================")
print(industry_total_table.to_string(index=False))


# ============================================================
# SAVE FULL TABLES
# ============================================================

top_name = "ALL" if TOP_N_FOR_CHART is None else str(TOP_N_FOR_CHART)

full_table_path = OUTPUT_DIR / CFG["SAVE_FULL_TABLE_NAME"].format(
    start=START_YEAR,
    end=END_YEAR
)

year_total_path = OUTPUT_DIR / CFG["SAVE_YEAR_TOTAL_TABLE_NAME"].format(
    start=START_YEAR,
    end=END_YEAR
)

industry_total_path = OUTPUT_DIR / CFG["SAVE_INDUSTRY_TOTAL_TABLE_NAME"].format(
    start=START_YEAR,
    end=END_YEAR
)

full_table.to_csv(full_table_path, index=False)
year_total_table.to_csv(year_total_path, index=False)
industry_total_table.to_csv(industry_total_path, index=False)

print("\nSaved FULL table:")
print(full_table_path)

print("\nSaved YEAR TOTAL table:")
print(year_total_path)

print("\nSaved INDUSTRY TOTAL table:")
print(industry_total_path)


# ============================================================
# CHART DATA: TOP INDUSTRIES ONLY
# ============================================================

chart_data = full_table.copy()

if TOP_N_FOR_CHART is not None:

    top_industries = (
        chart_data
        .groupby("industry_name")["job_activity_total"]
        .sum()
        .sort_values(ascending=False)
        .head(TOP_N_FOR_CHART)
        .index
        .tolist()
    )

    if INCLUDE_OTHER:
        chart_data["industry_name"] = np.where(
            chart_data["industry_name"].isin(top_industries),
            chart_data["industry_name"],
            "Other"
        )

        chart_data = (
            chart_data
            .groupby(["year", "industry_name"], dropna=False)[["job_gain", "job_loss"]]
            .sum()
            .reset_index()
        )

        chart_data["job_activity_total"] = (
            chart_data["job_gain"].abs()
            + chart_data["job_loss"].abs()
        )

        chart_data["net_job_change"] = (
            chart_data["job_gain"]
            - chart_data["job_loss"]
        )

    else:
        chart_data = chart_data[
            chart_data["industry_name"].isin(top_industries)
        ].copy()


# ============================================================
# SAVE CHART TABLE
# ============================================================

chart_table_path = OUTPUT_DIR / CFG["SAVE_CHART_TABLE_NAME"].format(
    start=START_YEAR,
    end=END_YEAR,
    top=top_name
)

chart_data.to_csv(chart_table_path, index=False)

print("\nSaved CHART table:")
print(chart_table_path)


# ============================================================
# PIVOT FOR STACKED BAR
# ============================================================

industry_order = (
    chart_data
    .groupby("industry_name")["job_activity_total"]
    .sum()
    .sort_values(ascending=False)
    .index
    .tolist()
)

all_years = list(range(START_YEAR, END_YEAR + 1))

gain_pivot = (
    chart_data
    .pivot_table(
        index="year",
        columns="industry_name",
        values="job_gain",
        aggfunc="sum",
        fill_value=0
    )
    .reindex(index=all_years, columns=industry_order, fill_value=0)
)

loss_pivot = (
    chart_data
    .pivot_table(
        index="year",
        columns="industry_name",
        values="job_loss",
        aggfunc="sum",
        fill_value=0
    )
    .reindex(index=all_years, columns=industry_order, fill_value=0)
)

print("\nYears on chart:")
print(all_years)

print("\nIndustries on chart:")
for industry in industry_order:
    print("-", industry)


# ============================================================
# PLOT CHART
# ============================================================

plt.figure(figsize=CFG["FIGSIZE"])

years = gain_pivot.index.astype(str)
y_pos = np.arange(len(years))

colors = getattr(plt.cm, CFG["COLOR_MAP"])(
    np.linspace(0, 1, len(industry_order))
)

# RIGHT SIDE = job gain
right_base = np.zeros(len(gain_pivot))

for i, industry in enumerate(industry_order):
    values = gain_pivot[industry].values

    plt.barh(
        y_pos,
        values,
        left=right_base,
        label=industry,
        color=colors[i]
    )

    right_base += values


# LEFT SIDE = job loss
left_base = np.zeros(len(loss_pivot))

for i, industry in enumerate(industry_order):
    values = loss_pivot[industry].values

    plt.barh(
        y_pos,
        -values,
        left=-left_base,
        color=colors[i]
    )

    left_base += values


plt.axvline(0, color="black", linewidth=1)

plt.yticks(y_pos, years)

chart_title_top = "ALL" if TOP_N_FOR_CHART is None else f"TOP {TOP_N_FOR_CHART}"

plt.title(
    f"{START_YEAR}-{END_YEAR} {chart_title_top} Real Industries: Job Loss vs Job Gain by Year\n"
    f"All real-industry total: Gain {fmt_number(overall_total_job_gain)} | "
    f"Loss {fmt_number(overall_total_job_loss)} | "
    f"Net {fmt_number(overall_net_job_change)}"
)

plt.xlabel("Job loss on left   |   Job gain on right")
plt.ylabel("Year")

plt.gca().xaxis.set_major_formatter(FuncFormatter(fmt_abs_number_axis))

plt.grid(axis="x", linestyle="--", alpha=0.4)

max_side = max(left_base.max(), right_base.max())

if max_side > 0:
    plt.xlim(-max_side * 1.12, max_side * 1.12)

plt.legend(
    title="Industry",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

# Note under chart
note_text_short = (
    "Note: Total private, Service-providing, and Goods-producing were removed "
    "because they are aggregate rows and would double-count detailed industries. "
    "Job activity total = job gain + job loss. Net job change = job gain - job loss."
)

plt.figtext(
    0.01,
    0.01,
    note_text_short,
    ha="left",
    fontsize=8
)

plt.tight_layout(rect=[0, 0.04, 1, 1])

chart_path = OUTPUT_DIR / CFG["SAVE_CHART_NAME"].format(
    start=START_YEAR,
    end=END_YEAR,
    top=top_name
)

plt.savefig(
    chart_path,
    dpi=CFG["DPI"],
    bbox_inches="tight"
)

plt.close()


# ============================================================
# SAVE NOTE FILE
# ============================================================

note_text_long = f"""
COMPANY JOB GAIN / JOB LOSS NOTE
================================

Year range:
{START_YEAR}-{END_YEAR}

File used:
{COMPANY_FILE}

Chart meaning:
- Left side = job loss
- Right side = job gain
- Each color = one industry
- Each row = one year

Important cleaning note:
The chart removes these aggregate industry rows:
- Total private
- Service-providing
- Goods-producing

Reason:
These are not detailed single industries.
They already include many smaller industries.
If they are kept together with detailed industries, the chart double-counts job gain and job loss.

Total count meaning:
- Total job gain = sum of job gain values
- Total job loss = sum of job loss values
- Total job activity = job gain + job loss
- Net job change = job gain - job loss

Overall total count for {START_YEAR}-{END_YEAR}:
- Years found: {year_count}
- Unique real industries: {unique_industry_count}
- Total job gain: {fmt_number(overall_total_job_gain)}
- Total job loss: {fmt_number(overall_total_job_loss)}
- Total job activity gain + loss: {fmt_number(overall_job_activity_total)}
- Net job change gain - loss: {fmt_number(overall_net_job_change)}

Output files:
- Chart PNG: {chart_path}
- Full year + industry table: {full_table_path}
- Chart table: {chart_table_path}
- Year total table: {year_total_path}
- Industry total table: {industry_total_path}
"""

note_path = OUTPUT_DIR / CFG["SAVE_NOTE_NAME"].format(
    start=START_YEAR,
    end=END_YEAR
)

with open(note_path, "w", encoding="utf-8") as f:
    f.write(note_text_long.strip())


# ============================================================
# DONE
# ============================================================

print("\nALL DONE.")
print("\nSaved chart here:")
print(chart_path)

print("\nSaved tables here:")
print(full_table_path)
print(chart_table_path)
print(year_total_path)
print(industry_total_path)

print("\nSaved note here:")
print(note_path)

print("\nNOTE YOU CAN WRITE DOWN:")
print(note_text_long)

# This chart shows company job gain and job loss by industry from 2010 to 2024.
# Each row is one year. Job loss is shown on the left side, and job gain is shown on the right side.
# Each color represents one industry. The chart uses the top 12 real industries by total job activity.
# The total job activity means job gain plus job loss.
# Net job change means job gain minus job loss.
# Aggregate rows such as Total private, Service-providing, and Goods-producing were removed because they are summary rows and would double-count the detailed industries.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# ============================================================
# COMPANY JOB GAIN / JOB LOSS BY INDUSTRY
# 2010-2024
#
# BASE FOLDER:
# Downloads/MyProject_1
#
# CHART:
# Y axis = year
# Right side = job gain
# Left side = job loss
# Color = industry
#
# TABLES:
# 1) full year + industry table
# 2) chart top industry table
# 3) year total table
# 4) industry total table
#
# NOTE REMOVED:
# - No note under chart
# - No note text file saved
#
# Job activity total = job gain + job loss
# Net job change = job gain - job loss
# ============================================================


# ============================================================
# SETTINGS OBJECT
# Change only this part later
# ============================================================

CFG = {
    "BASE_DIR": Path.home() / "Downloads/MyProject_1",

    # Put exact file here if auto-find chooses wrong file.
    # Example:
    # "COMPANY_FILE": Path.home() / "Downloads/MyProject_1/pre_formula_job_csv/your_file.csv",
    "COMPANY_FILE": None,

    "OUTPUT_DIR_NAME": "saved_python_pic_and_tables",

    "CHUNKSIZE": 100_000,

    "START_YEAR": 2010,
    "END_YEAR": 2024,

    # 12 = chart shows top 12 real industries
    # None = chart shows all real industries
    "TOP_N_FOR_CHART": 12,

    # False = do not add Other
    # True = combine non-top industries into Other
    "INCLUDE_OTHER": False,

    # Remove aggregate rows to avoid double-counting
    "REMOVE_AGGREGATE_INDUSTRIES": True,

    "AGGREGATE_INDUSTRIES_TO_REMOVE": [
        "Total private",
        "Service-providing",
        "Goods-producing",
    ],

    # Remove unknown / blank values
    "REMOVE_UNKNOWN": True,

    # Exclude rate / percent rows
    "RATE_WORDS": r"rate|percent|percentage|pct|share|ratio",

    # Do not use net jobs as gain/loss
    "NET_WORDS": r"net",

    # Job gain metric words
    "JOB_GAIN_WORDS": (
        r"gross job gains?|job gains?|jobs gained|job creation|jobs created|"
        r"employment gains?|hires?|hiring|job openings?"
    ),

    # Job loss metric words
    "JOB_LOSS_WORDS": (
        r"gross job losses?|job losses?|jobs lost|job destruction|"
        r"employment losses?|layoffs?|separations?"
    ),

    "FIGSIZE": (16, 9),
    "DPI": 300,
    "COLOR_MAP": "tab20",

    "SAVE_CHART_NAME": "company_job_gain_loss_by_industry_{start}_{end}_top{top}.png",

    "SAVE_FULL_TABLE_NAME": "company_job_gain_loss_by_year_industry_{start}_{end}_FULL.csv",

    "SAVE_CHART_TABLE_NAME": "company_job_gain_loss_by_year_industry_{start}_{end}_CHART_TOP{top}.csv",

    "SAVE_YEAR_TOTAL_TABLE_NAME": "company_job_gain_loss_YEAR_TOTAL_{start}_{end}.csv",

    "SAVE_INDUSTRY_TOTAL_TABLE_NAME": "company_job_gain_loss_INDUSTRY_TOTAL_{start}_{end}.csv",
}


# ============================================================
# VARIABLES FROM SETTINGS OBJECT
# ============================================================

BASE_DIR = CFG["BASE_DIR"]
OUTPUT_DIR = BASE_DIR / CFG["OUTPUT_DIR_NAME"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMPANY_FILE = CFG["COMPANY_FILE"]

CHUNKSIZE = CFG["CHUNKSIZE"]
START_YEAR = CFG["START_YEAR"]
END_YEAR = CFG["END_YEAR"]
TOP_N_FOR_CHART = CFG["TOP_N_FOR_CHART"]
INCLUDE_OTHER = CFG["INCLUDE_OTHER"]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def norm_col(x):
    return str(x).strip().lower().replace(" ", "_")


def find_col(columns, possible_names):
    col_map = {norm_col(c): c for c in columns}

    for name in possible_names:
        key = norm_col(name)
        if key in col_map:
            return col_map[key]

    return None


def fmt_number(x):
    try:
        return f"{int(round(x)):,}"
    except Exception:
        return str(x)


def fmt_abs_number_axis(x, pos):
    try:
        return f"{abs(int(x)):,}"
    except Exception:
        return str(x)


def inspect_csv_file(path):
    """
    Detect whether file is:
    1) raw company format:
       year, industry_name, metric_name, value

    2) pre-formula job format:
       year, connected_industry, job_count, job_loss_count
       or similar
    """

    try:
        columns = pd.read_csv(path, nrows=0, low_memory=False).columns.tolist()
    except Exception:
        return None

    year_col = find_col(columns, ["year"])

    raw_industry_col = find_col(columns, ["industry_name"])
    raw_metric_col = find_col(columns, ["metric_name"])
    raw_value_col = find_col(columns, ["value"])

    pre_industry_col = find_col(
        columns,
        [
            "industry_name",
            "connected_industry",
            "industry",
            "industry_group"
        ]
    )

    pre_gain_col = find_col(
        columns,
        [
            "job_gain",
            "job_gains",
            "job_gain_count",
            "jobs_gained",
            "job_count"
        ]
    )

    pre_loss_col = find_col(
        columns,
        [
            "job_loss",
            "job_losses",
            "job_loss_count",
            "jobs_lost"
        ]
    )

    if year_col and raw_industry_col and raw_metric_col and raw_value_col:
        return {
            "format": "raw_company",
            "year_col": year_col,
            "industry_col": raw_industry_col,
            "metric_col": raw_metric_col,
            "value_col": raw_value_col,
            "columns": columns
        }

    if year_col and pre_industry_col and pre_gain_col and pre_loss_col:
        return {
            "format": "pre_formula_job",
            "year_col": year_col,
            "industry_col": pre_industry_col,
            "gain_col": pre_gain_col,
            "loss_col": pre_loss_col,
            "columns": columns
        }

    return None


def score_company_file(path):
    """
    Auto-find best company/job CSV inside MyProject_1.
    """

    info = inspect_csv_file(path)

    if info is None:
        return -999, None

    name = path.name.lower()
    folder = str(path.parent).lower()

    score = 0

    if info["format"] == "raw_company":
        score += 100

    if info["format"] == "pre_formula_job":
        score += 90

    if "pre_formula_job_csv" in folder:
        score += 60

    if "job" in name:
        score += 40

    if "company" in name:
        score += 30

    if "business" in name:
        score += 30

    if "startup" in name:
        score -= 20

    if "degree" in folder or "degree" in name:
        score -= 100

    return score, info


def auto_find_company_file(base_dir):
    """
    Auto-find company/job CSV inside Downloads/MyProject_1.
    """

    if not base_dir.exists():
        raise FileNotFoundError(f"Missing BASE_DIR: {base_dir}")

    csv_files = list(base_dir.rglob("*.csv"))

    if not csv_files:
        raise FileNotFoundError(f"No CSV files found inside: {base_dir}")

    scored = []

    for path in csv_files:
        score, info = score_company_file(path)
        if score > 0:
            scored.append((score, path, info))

    if not scored:
        print("\nCSV files found, but none matched company/job format:")

        for p in csv_files[:50]:
            print(p)

        raise ValueError(
            "No usable company/job CSV found. "
            "Set CFG['COMPANY_FILE'] to the exact CSV path."
        )

    scored = sorted(scored, key=lambda x: x[0], reverse=True)

    print("\nPossible company/job files found:")

    for score, path, info in scored[:10]:
        print(f"score={score:>3} | format={info['format']} | {path}")

    best_score, best_path, best_info = scored[0]

    print("\nUsing this company/job file:")
    print(best_path)
    print("Detected format:", best_info["format"])

    return best_path, best_info


def clean_industry_rows(df, industry_col):
    """
    Remove blank, unknown, and aggregate industry rows.
    """

    df[industry_col] = df[industry_col].astype(str).str.strip()

    if CFG["REMOVE_UNKNOWN"]:
        df = df[
            df[industry_col].notna()
            & (df[industry_col] != "")
            & (~df[industry_col].str.lower().isin(["nan", "none", "unknown"]))
        ].copy()

    if CFG["REMOVE_AGGREGATE_INDUSTRIES"]:
        df = df[
            ~df[industry_col].isin(CFG["AGGREGATE_INDUSTRIES_TO_REMOVE"])
        ].copy()

    return df


# ============================================================
# FIND COMPANY FILE
# ============================================================

print("Checking BASE_DIR...")
print(BASE_DIR)

if COMPANY_FILE is None:
    COMPANY_FILE, FILE_INFO = auto_find_company_file(BASE_DIR)

else:
    COMPANY_FILE = Path(COMPANY_FILE)

    if not COMPANY_FILE.exists():
        raise FileNotFoundError(f"Missing Company file: {COMPANY_FILE}")

    FILE_INFO = inspect_csv_file(COMPANY_FILE)

    if FILE_INFO is None:
        raise ValueError(
            f"This file does not look like raw company or pre-formula job CSV:\n{COMPANY_FILE}"
        )

    print("\nUsing manually selected company file:")
    print(COMPANY_FILE)
    print("Detected format:", FILE_INFO["format"])


# ============================================================
# LOAD DATA
# ============================================================

parts = []

matched_job_gain_metric_names = set()
matched_job_loss_metric_names = set()

print(f"\nLoading Company job data from {START_YEAR} to {END_YEAR}...")


# ============================================================
# CASE 1: RAW COMPANY FORMAT
# year, industry_name, metric_name, value
# ============================================================

if FILE_INFO["format"] == "raw_company":

    year_col = FILE_INFO["year_col"]
    industry_col = FILE_INFO["industry_col"]
    metric_col = FILE_INFO["metric_col"]
    value_col = FILE_INFO["value_col"]

    usecols = [
        year_col,
        industry_col,
        metric_col,
        value_col
    ]

    chunk_count = 0

    for chunk in pd.read_csv(
        COMPANY_FILE,
        usecols=usecols,
        chunksize=CHUNKSIZE,
        low_memory=False
    ):
        chunk_count += 1
        print(f"Loading chunk {chunk_count}... rows: {len(chunk):,}")

        chunk[year_col] = pd.to_numeric(chunk[year_col], errors="coerce")
        chunk[value_col] = pd.to_numeric(chunk[value_col], errors="coerce")

        chunk = chunk.dropna(subset=[year_col, industry_col, metric_col, value_col])
        chunk[year_col] = chunk[year_col].astype(int)

        chunk = chunk[
            (chunk[year_col] >= START_YEAR)
            & (chunk[year_col] <= END_YEAR)
        ].copy()

        if chunk.empty:
            continue

        chunk = clean_industry_rows(chunk, industry_col)

        if chunk.empty:
            continue

        metric = chunk[metric_col].astype(str).str.lower()

        not_rate_mask = ~metric.str.contains(
            CFG["RATE_WORDS"],
            na=False,
            regex=True
        )

        not_net_mask = ~metric.str.contains(
            CFG["NET_WORDS"],
            na=False,
            regex=True
        )

        job_gain_mask = (
            metric.str.contains(CFG["JOB_GAIN_WORDS"], na=False, regex=True)
            & not_rate_mask
            & not_net_mask
        )

        job_loss_mask = (
            metric.str.contains(CFG["JOB_LOSS_WORDS"], na=False, regex=True)
            & not_rate_mask
            & not_net_mask
        )

        matched_job_gain_metric_names.update(
            chunk.loc[job_gain_mask, metric_col].dropna().unique()
        )

        matched_job_loss_metric_names.update(
            chunk.loc[job_loss_mask, metric_col].dropna().unique()
        )

        chunk["job_gain"] = np.where(job_gain_mask, chunk[value_col], 0)
        chunk["job_loss"] = np.where(job_loss_mask, chunk[value_col], 0)

        part = (
            chunk
            .groupby([year_col, industry_col], dropna=False)[["job_gain", "job_loss"]]
            .sum()
            .reset_index()
            .rename(columns={
                year_col: "year",
                industry_col: "industry_name"
            })
        )

        parts.append(part)


# ============================================================
# CASE 2: PRE-FORMULA JOB FORMAT
# year, connected_industry, job_count, job_loss_count
# ============================================================

elif FILE_INFO["format"] == "pre_formula_job":

    year_col = FILE_INFO["year_col"]
    industry_col = FILE_INFO["industry_col"]
    gain_col = FILE_INFO["gain_col"]
    loss_col = FILE_INFO["loss_col"]

    usecols = [
        year_col,
        industry_col,
        gain_col,
        loss_col
    ]

    chunk_count = 0

    for chunk in pd.read_csv(
        COMPANY_FILE,
        usecols=usecols,
        chunksize=CHUNKSIZE,
        low_memory=False
    ):
        chunk_count += 1
        print(f"Loading chunk {chunk_count}... rows: {len(chunk):,}")

        chunk[year_col] = pd.to_numeric(chunk[year_col], errors="coerce")
        chunk[gain_col] = pd.to_numeric(chunk[gain_col], errors="coerce")
        chunk[loss_col] = pd.to_numeric(chunk[loss_col], errors="coerce")

        chunk = chunk.dropna(subset=[year_col, industry_col])
        chunk[year_col] = chunk[year_col].astype(int)

        chunk = chunk[
            (chunk[year_col] >= START_YEAR)
            & (chunk[year_col] <= END_YEAR)
        ].copy()

        if chunk.empty:
            continue

        chunk = clean_industry_rows(chunk, industry_col)

        if chunk.empty:
            continue

        chunk[gain_col] = chunk[gain_col].fillna(0)
        chunk[loss_col] = chunk[loss_col].fillna(0)

        # Important:
        # Some pre-formula files repeat the same company job value
        # for multiple degree fields.
        # This removes exact duplicate job rows.
        chunk = chunk.drop_duplicates(
            subset=[year_col, industry_col, gain_col, loss_col]
        )

        part = (
            chunk
            .groupby([year_col, industry_col], dropna=False)[[gain_col, loss_col]]
            .sum()
            .reset_index()
            .rename(columns={
                year_col: "year",
                industry_col: "industry_name",
                gain_col: "job_gain",
                loss_col: "job_loss"
            })
        )

        parts.append(part)


print("\nFinished loading data.")


# ============================================================
# COMBINE DATA
# ============================================================

if not parts:
    raise ValueError(f"No Company job data found from {START_YEAR} to {END_YEAR}.")

full_table = pd.concat(parts, ignore_index=True)

full_table = (
    full_table
    .groupby(["year", "industry_name"], dropna=False)[["job_gain", "job_loss"]]
    .sum()
    .reset_index()
)

full_table["job_gain"] = full_table["job_gain"].fillna(0)
full_table["job_loss"] = full_table["job_loss"].fillna(0)

full_table["job_activity_total"] = (
    full_table["job_gain"].abs()
    + full_table["job_loss"].abs()
)

full_table["net_job_change"] = (
    full_table["job_gain"]
    - full_table["job_loss"]
)

full_table = full_table[full_table["job_activity_total"] > 0].copy()

if full_table.empty:
    raise ValueError(
        f"Data loaded, but no job gain/loss rows matched from {START_YEAR} to {END_YEAR}."
    )


# ============================================================
# PRINT RAW METRIC NAMES USED
# ============================================================

if FILE_INFO["format"] == "raw_company":

    print("\nMetric names used for job_gain:")

    for x in sorted(matched_job_gain_metric_names):
        print("-", x)

    print("\nMetric names used for job_loss:")

    for x in sorted(matched_job_loss_metric_names):
        print("-", x)


# ============================================================
# TOTAL TABLE 1: YEAR TOTAL
# Like degree total count by year
# ============================================================

year_total_table = (
    full_table
    .groupby("year", dropna=False)[
        ["job_gain", "job_loss", "job_activity_total", "net_job_change"]
    ]
    .sum()
    .reset_index()
    .sort_values("year")
)

year_total_table = year_total_table.rename(columns={
    "job_gain": "total_job_gain",
    "job_loss": "total_job_loss",
    "job_activity_total": "total_job_activity_gain_plus_loss",
    "net_job_change": "net_job_change_gain_minus_loss"
})


# ============================================================
# TOTAL TABLE 2: INDUSTRY TOTAL
# Like degree total count by category
# ============================================================

industry_total_table = (
    full_table
    .groupby("industry_name", dropna=False)[
        ["job_gain", "job_loss", "job_activity_total", "net_job_change"]
    ]
    .sum()
    .reset_index()
    .sort_values("job_activity_total", ascending=False)
)

industry_total_table = industry_total_table.rename(columns={
    "job_gain": "range_total_job_gain",
    "job_loss": "range_total_job_loss",
    "job_activity_total": "range_total_job_activity_gain_plus_loss",
    "net_job_change": "range_net_job_change_gain_minus_loss"
})


# ============================================================
# OVERALL TOTAL COUNT
# ============================================================

overall_total_job_gain = full_table["job_gain"].sum()
overall_total_job_loss = full_table["job_loss"].sum()
overall_job_activity_total = full_table["job_activity_total"].sum()
overall_net_job_change = full_table["net_job_change"].sum()

unique_industry_count = full_table["industry_name"].nunique()
year_count = full_table["year"].nunique()


# ============================================================
# PRINT TOTALS
# ============================================================

print("\n============================================================")
print("OVERALL TOTAL COUNT")
print("============================================================")
print("Year range:", START_YEAR, "-", END_YEAR)
print("Years found:", year_count)
print("Unique industries:", unique_industry_count)
print("Total job gain:", fmt_number(overall_total_job_gain))
print("Total job loss:", fmt_number(overall_total_job_loss))
print("Total job activity gain + loss:", fmt_number(overall_job_activity_total))
print("Net job change gain - loss:", fmt_number(overall_net_job_change))

print("\n============================================================")
print("YEAR TOTAL TABLE")
print("============================================================")
print(year_total_table.to_string(index=False))

print("\n============================================================")
print("INDUSTRY TOTAL TABLE")
print("============================================================")
print(industry_total_table.to_string(index=False))


# ============================================================
# SAVE FULL TABLES
# ============================================================

top_name = "ALL" if TOP_N_FOR_CHART is None else str(TOP_N_FOR_CHART)

full_table_path = OUTPUT_DIR / CFG["SAVE_FULL_TABLE_NAME"].format(
    start=START_YEAR,
    end=END_YEAR
)

year_total_path = OUTPUT_DIR / CFG["SAVE_YEAR_TOTAL_TABLE_NAME"].format(
    start=START_YEAR,
    end=END_YEAR
)

industry_total_path = OUTPUT_DIR / CFG["SAVE_INDUSTRY_TOTAL_TABLE_NAME"].format(
    start=START_YEAR,
    end=END_YEAR
)

full_table.to_csv(full_table_path, index=False)
year_total_table.to_csv(year_total_path, index=False)
industry_total_table.to_csv(industry_total_path, index=False)

print("\nSaved FULL table:")
print(full_table_path)

print("\nSaved YEAR TOTAL table:")
print(year_total_path)

print("\nSaved INDUSTRY TOTAL table:")
print(industry_total_path)


# ============================================================
# CHART DATA: TOP INDUSTRIES ONLY
# ============================================================

chart_data = full_table.copy()

if TOP_N_FOR_CHART is not None:

    top_industries = (
        chart_data
        .groupby("industry_name")["job_activity_total"]
        .sum()
        .sort_values(ascending=False)
        .head(TOP_N_FOR_CHART)
        .index
        .tolist()
    )

    if INCLUDE_OTHER:
        chart_data["industry_name"] = np.where(
            chart_data["industry_name"].isin(top_industries),
            chart_data["industry_name"],
            "Other"
        )

        chart_data = (
            chart_data
            .groupby(["year", "industry_name"], dropna=False)[["job_gain", "job_loss"]]
            .sum()
            .reset_index()
        )

        chart_data["job_activity_total"] = (
            chart_data["job_gain"].abs()
            + chart_data["job_loss"].abs()
        )

        chart_data["net_job_change"] = (
            chart_data["job_gain"]
            - chart_data["job_loss"]
        )

    else:
        chart_data = chart_data[
            chart_data["industry_name"].isin(top_industries)
        ].copy()


# ============================================================
# SAVE CHART TABLE
# ============================================================

chart_table_path = OUTPUT_DIR / CFG["SAVE_CHART_TABLE_NAME"].format(
    start=START_YEAR,
    end=END_YEAR,
    top=top_name
)

chart_data.to_csv(chart_table_path, index=False)

print("\nSaved CHART table:")
print(chart_table_path)


# ============================================================
# PIVOT FOR STACKED BAR
# ============================================================

industry_order = (
    chart_data
    .groupby("industry_name")["job_activity_total"]
    .sum()
    .sort_values(ascending=False)
    .index
    .tolist()
)

all_years = list(range(START_YEAR, END_YEAR + 1))

gain_pivot = (
    chart_data
    .pivot_table(
        index="year",
        columns="industry_name",
        values="job_gain",
        aggfunc="sum",
        fill_value=0
    )
    .reindex(index=all_years, columns=industry_order, fill_value=0)
)

loss_pivot = (
    chart_data
    .pivot_table(
        index="year",
        columns="industry_name",
        values="job_loss",
        aggfunc="sum",
        fill_value=0
    )
    .reindex(index=all_years, columns=industry_order, fill_value=0)
)

print("\nYears on chart:")
print(all_years)

print("\nIndustries on chart:")

for industry in industry_order:
    print("-", industry)


# ============================================================
# PLOT CHART
# ============================================================

plt.figure(figsize=CFG["FIGSIZE"])

years = gain_pivot.index.astype(str)
y_pos = np.arange(len(years))

colors = getattr(plt.cm, CFG["COLOR_MAP"])(
    np.linspace(0, 1, len(industry_order))
)

# RIGHT SIDE = job gain
right_base = np.zeros(len(gain_pivot))

for i, industry in enumerate(industry_order):
    values = gain_pivot[industry].values

    plt.barh(
        y_pos,
        values,
        left=right_base,
        label=industry,
        color=colors[i]
    )

    right_base += values


# LEFT SIDE = job loss
left_base = np.zeros(len(loss_pivot))

for i, industry in enumerate(industry_order):
    values = loss_pivot[industry].values

    plt.barh(
        y_pos,
        -values,
        left=-left_base,
        color=colors[i]
    )

    left_base += values


plt.axvline(0, color="black", linewidth=1)

plt.yticks(y_pos, years)

chart_title_top = "ALL" if TOP_N_FOR_CHART is None else f"TOP {TOP_N_FOR_CHART}"

plt.title(
    f"{START_YEAR}-{END_YEAR} {chart_title_top} Real Industries: Job Loss vs Job Gain by Year\n"
    f"All real-industry total: Gain {fmt_number(overall_total_job_gain)} | "
    f"Loss {fmt_number(overall_total_job_loss)} | "
    f"Net {fmt_number(overall_net_job_change)}"
)

plt.xlabel("Job loss on left   |   Job gain on right")
plt.ylabel("Year")

plt.gca().xaxis.set_major_formatter(FuncFormatter(fmt_abs_number_axis))

plt.grid(axis="x", linestyle="--", alpha=0.4)

max_side = max(left_base.max(), right_base.max())

if max_side > 0:
    plt.xlim(-max_side * 1.12, max_side * 1.12)

plt.legend(
    title="Industry",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

# No note under chart
plt.tight_layout()

chart_path = OUTPUT_DIR / CFG["SAVE_CHART_NAME"].format(
    start=START_YEAR,
    end=END_YEAR,
    top=top_name
)

plt.savefig(
    chart_path,
    dpi=CFG["DPI"],
    bbox_inches="tight"
)

plt.close()


# ============================================================
# COPY TEXT FOR YOUR WRITE-UP
# This prints only.
# It does not save a note file.
# ============================================================

copy_text = f"""
COPY THIS FOR YOUR WRITE-UP
===========================

This chart shows company job gain and job loss by industry from {START_YEAR} to {END_YEAR}.
Each row is one year.
Job loss is shown on the left side, and job gain is shown on the right side.
Each color represents one industry.

The chart uses {"all real industries" if TOP_N_FOR_CHART is None else "the top " + str(TOP_N_FOR_CHART) + " real industries by total job activity"}.
Total job activity means job gain plus job loss.
Net job change means job gain minus job loss.

Aggregate rows such as Total private, Service-providing, and Goods-producing were removed because they are summary rows and would double-count the detailed industries.

Overall total count for {START_YEAR}-{END_YEAR}:
Years found: {year_count}
Unique real industries: {unique_industry_count}
Total job gain: {fmt_number(overall_total_job_gain)}
Total job loss: {fmt_number(overall_total_job_loss)}
Total job activity gain + loss: {fmt_number(overall_job_activity_total)}
Net job change gain - loss: {fmt_number(overall_net_job_change)}
"""

print("\n============================================================")
print(copy_text.strip())
print("============================================================")


# ============================================================
# DONE
# ============================================================

print("\nALL DONE.")

print("\nSaved chart here:")
print(chart_path)

print("\nSaved tables here:")
print(full_table_path)
print(chart_table_path)
print(year_total_path)
print(industry_total_path)

# job gain loss 3

In [ ]:
from pathlib import Path
import re
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# ============================================================
# TOP 12 REAL INDUSTRIES: JOB LOSS VS JOB GAIN BY YEAR
#
# FIX FOR YOUR RED-MARK AREA:
# 1) Keep the old good title at the top.
# 2) Do NOT push the chart down.
# 3) Put each industry total line-by-line on the RIGHT side,
#    under the legend:
#       Industry total: Gain ? | Loss ? | Net +?
# 4) Save PNG + CSV tables.
# ============================================================

CFG = {
    # Your company CSV file
    "COMPANY_FILE": Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv",

    # Output folder
    "OUTPUT_DIR": Path.home() / "Downloads" / "pre_formula_job_charts_industry_totals_right_lines",

    # Year range
    "START_YEAR": 2010,
    "END_YEAR": 2024,

    # Top industries to show in stacked chart
    "TOP_N": 12,

    # Loading size
    "CHUNKSIZE": 100_000,

    # CSV columns
    "YEAR_COL": "year",
    "INDUSTRY_COL": "industry_name",
    "METRIC_NAME_COL": "metric_name",
    "METRIC_CATEGORY_COL": "metric_category",
    "METRIC_CODE_COL": "metric_code",
    "VALUE_COL": "value",

    # Image settings
    "FIG_WIDTH": 16,
    "FIG_HEIGHT": 9,
    "DPI": 180,

    # Layout: right side is saved for legend + industry total lines
    "PLOT_RIGHT": 0.73,
    "PLOT_TOP": 0.91,
    "PLOT_LEFT": 0.08,
    "PLOT_BOTTOM": 0.09,

    # Right-side industry total text position
    "RIGHT_TEXT_X": 0.765,
    "RIGHT_TEXT_Y": 0.49,
    "RIGHT_TEXT_FONT_SIZE": 7.5,
}


def fmt_number(x, pos=None):
    """Show both left and right axis numbers as positive numbers."""
    try:
        return f"{abs(int(x)):,}"
    except Exception:
        return str(x)


def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def is_real_industry_name(name):
    """Remove blank / unknown / code-only industry labels."""
    s = clean_text(name)
    low = s.lower()

    if not s:
        return False
    if low in {"nan", "none", "unknown", "blank", "null", "all", "total", "total private"}:
        return False
    if re.fullmatch(r"naics\s*\d+", low, flags=re.IGNORECASE):
        return False
    if re.fullmatch(r"naics\d+", s, flags=re.IGNORECASE):
        return False
    if re.fullmatch(r"\d+", s):
        return False

    return True


def short_industry_name(name):
    """Short names so each total line fits on the right side."""
    s = str(name).strip()
    replacements = {
        "Professional and business services": "Professional/business",
        "Leisure and hospitality": "Leisure/hospitality",
        "Education and health services": "Education/health",
        "Other services (except public administration)": "Other services",
        "Transportation and warehousing": "Transportation/warehousing",
        "Natural resources and mining": "Natural resources/mining",
        "Financial activities": "Financial activities",
        "Service-providing": "Service-providing",
        "Goods-producing": "Goods-producing",
        "Retail trade": "Retail trade",
        "Wholesale trade": "Wholesale trade",
        "Manufacturing": "Manufacturing",
        "Construction": "Construction",
        "Information": "Information",
    }
    return replacements.get(s, s)


def sanitize_filename(name):
    name = re.sub(r"[^A-Za-z0-9]+", "_", str(name)).strip("_").lower()
    return name or "chart"


def load_job_gain_loss():
    company_file = CFG["COMPANY_FILE"]
    if not company_file.exists():
        raise FileNotFoundError(f"Cannot find file: {company_file}")

    needed_cols = [
        CFG["YEAR_COL"],
        CFG["INDUSTRY_COL"],
        CFG["METRIC_NAME_COL"],
        CFG["METRIC_CATEGORY_COL"],
        CFG["METRIC_CODE_COL"],
        CFG["VALUE_COL"],
    ]

    print(f"Reading: {company_file}", flush=True)
    print(f"Years: {CFG['START_YEAR']} - {CFG['END_YEAR']}", flush=True)

    parts = []
    rows_seen = 0
    rows_kept = 0
    start_time = time.time()

    reader = pd.read_csv(
        company_file,
        usecols=lambda c: c in needed_cols,
        chunksize=CFG["CHUNKSIZE"],
        low_memory=False,
    )

    for chunk_num, chunk in enumerate(reader, start=1):
        rows_seen += len(chunk)

        for col in needed_cols:
            if col not in chunk.columns:
                chunk[col] = ""

        chunk[CFG["YEAR_COL"]] = pd.to_numeric(chunk[CFG["YEAR_COL"]], errors="coerce")
        chunk[CFG["VALUE_COL"]] = pd.to_numeric(chunk[CFG["VALUE_COL"]], errors="coerce").fillna(0)

        chunk = chunk[
            (chunk[CFG["YEAR_COL"]] >= CFG["START_YEAR"]) &
            (chunk[CFG["YEAR_COL"]] <= CFG["END_YEAR"])
        ].copy()

        if chunk.empty:
            continue

        metric_text = (
            chunk[CFG["METRIC_NAME_COL"]].astype(str).str.lower().fillna("") + " " +
            chunk[CFG["METRIC_CATEGORY_COL"]].astype(str).str.lower().fillna("") + " " +
            chunk[CFG["METRIC_CODE_COL"]].astype(str).str.lower().fillna("")
        )

        has_job_word = metric_text.str.contains(
            r"\b(job|jobs|employment|employee|employees|payroll)\b",
            regex=True,
            na=False,
        )
        gain_mask = has_job_word & metric_text.str.contains(
            r"\b(gain|gains|gained|created|creation)\b",
            regex=True,
            na=False,
        )
        loss_mask = has_job_word & metric_text.str.contains(
            r"\b(loss|losses|lost|destroyed|destruction)\b",
            regex=True,
            na=False,
        )

        keep = chunk[gain_mask | loss_mask].copy()
        if keep.empty:
            continue

        keep["direction"] = np.where(gain_mask.loc[keep.index], "gain", "loss")
        keep["industry_name"] = keep[CFG["INDUSTRY_COL"]].astype(str).str.strip()
        keep = keep[keep["industry_name"].apply(is_real_industry_name)].copy()

        if keep.empty:
            continue

        keep["year"] = keep[CFG["YEAR_COL"]].astype(int)
        keep["value"] = keep[CFG["VALUE_COL"]].fillna(0)

        grouped = (
            keep.groupby(["industry_name", "year", "direction"], as_index=False)["value"]
            .sum()
        )
        parts.append(grouped)
        rows_kept += len(keep)

        if chunk_num % 5 == 0:
            elapsed = time.time() - start_time
            print(
                f"Chunk {chunk_num:,} | rows seen {rows_seen:,} | job rows kept {rows_kept:,} | {elapsed:,.1f}s",
                flush=True,
            )

    if not parts:
        raise ValueError("No job gain/loss rows found. Check metric columns and keywords.")

    long_df = pd.concat(parts, ignore_index=True)
    long_df = long_df.groupby(["industry_name", "year", "direction"], as_index=False)["value"].sum()

    wide = (
        long_df.pivot_table(
            index=["industry_name", "year"],
            columns="direction",
            values="value",
            aggfunc="sum",
            fill_value=0,
        )
        .reset_index()
    )

    if "gain" not in wide.columns:
        wide["gain"] = 0
    if "loss" not in wide.columns:
        wide["loss"] = 0

    wide["gain"] = wide["gain"].round().astype(int)
    wide["loss"] = wide["loss"].round().astype(int)
    wide["net"] = wide["gain"] - wide["loss"]
    wide = wide[["industry_name", "year", "gain", "loss", "net"]]

    print("Finished loading.", flush=True)
    print(f"Rows seen: {rows_seen:,}", flush=True)
    print(f"Job gain/loss rows kept: {rows_kept:,}", flush=True)
    print(f"Real industries found: {wide['industry_name'].nunique():,}", flush=True)

    return wide


def make_chart(data):
    out_dir = CFG["OUTPUT_DIR"]
    chart_dir = out_dir / "charts"
    table_dir = out_dir / "tables"
    chart_dir.mkdir(parents=True, exist_ok=True)
    table_dir.mkdir(parents=True, exist_ok=True)

    start_year = CFG["START_YEAR"]
    end_year = CFG["END_YEAR"]
    top_n = CFG["TOP_N"]

    totals = data.groupby("industry_name", as_index=False)[["gain", "loss"]].sum()
    totals["net"] = totals["gain"] - totals["loss"]
    totals["total_activity"] = totals["gain"] + totals["loss"]
    totals = totals.sort_values("total_activity", ascending=False)

    top_industries = totals.head(top_n)["industry_name"].tolist()
    top_totals = totals[totals["industry_name"].isin(top_industries)].copy()
    top_totals["industry_name"] = pd.Categorical(
        top_totals["industry_name"],
        categories=top_industries,
        ordered=True,
    )
    top_totals = top_totals.sort_values("industry_name")

    top_data = data[data["industry_name"].isin(top_industries)].copy()

    # Save CSV tables
    data.to_csv(table_dir / f"all_real_industry_gain_loss_{start_year}_{end_year}.csv", index=False)
    totals.to_csv(table_dir / f"all_industry_totals_{start_year}_{end_year}.csv", index=False)
    top_totals.to_csv(table_dir / f"top_{top_n}_industry_totals_{start_year}_{end_year}.csv", index=False)
    top_data.to_csv(table_dir / f"top_{top_n}_industry_gain_loss_by_year_{start_year}_{end_year}.csv", index=False)

    years = list(range(start_year, end_year + 1))

    gain_table = (
        top_data.pivot_table(index="year", columns="industry_name", values="gain", aggfunc="sum", fill_value=0)
        .reindex(index=years, columns=top_industries, fill_value=0)
    )
    loss_table = (
        top_data.pivot_table(index="year", columns="industry_name", values="loss", aggfunc="sum", fill_value=0)
        .reindex(index=years, columns=top_industries, fill_value=0)
    )

    # Total shown in the subtitle is for the plotted TOP N industries.
    total_gain = int(top_data["gain"].sum())
    total_loss = int(top_data["loss"].sum())
    total_net = total_gain - total_loss

    fig, ax = plt.subplots(figsize=(CFG["FIG_WIDTH"], CFG["FIG_HEIGHT"]))

    y_pos = np.arange(len(years))
    colors = plt.get_cmap("tab20").colors

    left_gain = np.zeros(len(years))
    left_loss = np.zeros(len(years))

    for i, industry in enumerate(top_industries):
        color = colors[i % len(colors)]
        gains = gain_table[industry].to_numpy(dtype=float)
        losses = loss_table[industry].to_numpy(dtype=float)

        ax.barh(y_pos, -losses, left=-left_loss, height=0.78, color=color, alpha=0.85)
        ax.barh(y_pos, gains, left=left_gain, height=0.78, color=color, alpha=0.85, label=industry)

        left_loss += losses
        left_gain += gains

    ax.axvline(0, color="black", linewidth=1.0)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(years)
    ax.invert_yaxis()
    ax.set_ylabel("Year")
    ax.set_xlabel("Job loss on left   |   Job gain on right")
    ax.xaxis.set_major_formatter(FuncFormatter(fmt_number))
    ax.grid(axis="x", linestyle="--", alpha=0.35)

    max_value = max(float(left_gain.max()), float(left_loss.max()), 1)
    ax.set_xlim(-max_value * 1.22, max_value * 1.22)

    # Keep the old good top title style.
    title_text = (
        f"{start_year}-{end_year} TOP {len(top_industries)} Real Industries: Job Loss vs Job Gain by Year\n"
        f"All plotted-industry total: Gain {total_gain:,} | Loss {total_loss:,} | Net {total_net:+,}"
    )
    fig.suptitle(title_text, fontsize=12, y=0.975)

    ax.legend(
        title="Industry",
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        frameon=True,
        fontsize=9,
        title_fontsize=10,
    )

    # Right-side industry totals: one industry per line.
    right_lines = ["Industry totals"]
    for _, row in top_totals.iterrows():
        industry = short_industry_name(row["industry_name"])
        gain = int(row["gain"])
        loss = int(row["loss"])
        net = gain - loss
        right_lines.append(f"{industry} total: Gain {gain:,} | Loss {loss:,} | Net {net:+,}")

    right_text = "\n".join(right_lines)
    fig.text(
        CFG["RIGHT_TEXT_X"],
        CFG["RIGHT_TEXT_Y"],
        right_text,
        ha="left",
        va="top",
        fontsize=CFG["RIGHT_TEXT_FONT_SIZE"],
        family="monospace",
        linespacing=1.25,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="lightgray", alpha=0.95),
    )

    fig.subplots_adjust(
        top=CFG["PLOT_TOP"],
        right=CFG["PLOT_RIGHT"],
        left=CFG["PLOT_LEFT"],
        bottom=CFG["PLOT_BOTTOM"],
    )

    chart_path = chart_dir / f"company_job_gain_loss_by_industry_{start_year}_{end_year}_top{len(top_industries)}_RIGHT_INDUSTRY_TOTAL_LINES.png"
    fig.savefig(chart_path, dpi=CFG["DPI"], bbox_inches="tight")
    plt.close(fig)

    # Also save the right-side total text as a .txt file, so you can check it easily.
    text_path = table_dir / f"top_{top_n}_industry_total_lines_{start_year}_{end_year}.txt"
    with open(text_path, "w", encoding="utf-8") as f:
        f.write(title_text + "\n\n" + right_text + "\n")

    print(f"Saved chart: {chart_path}", flush=True)
    print(f"Saved tables: {table_dir}", flush=True)
    print(f"Saved text preview: {text_path}", flush=True)


def main():
    data = load_job_gain_loss()
    make_chart(data)
    print("Done.", flush=True)
    print(f"Open output folder: {CFG['OUTPUT_DIR']}", flush=True)


if __name__ == "__main__":
    main()


# Job gain loss 4

In [ ]:
from pathlib import Path
import re
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# ============================================================
# ALL REAL INDUSTRIES: JOB LOSS VS JOB GAIN BY YEAR
#
# WHAT THIS VERSION FIXES:
# 1) Title says ALL Real Industries, not TOP 12.
# 2) Uses ALL real/tight industries after removing parent groups.
# 3) Removes parent/broad groups that double-count:
#       Service-providing
#       Goods-producing
# 4) Keeps the good old 2-line title at the top.
# 5) Shows each industry total line-by-line on the right side:
#       Industry total: Gain ? | Loss ? | Net +?
# 6) Saves PNG + CSV + TXT preview.
# ============================================================

CFG = {
    # Your company CSV file
    "COMPANY_FILE": Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv",

    # Output folder
    "OUTPUT_DIR": Path.home() / "Downloads" / "pre_formula_job_charts_all_real_industries_no_top12",

    # Year range
    "START_YEAR": 2010,
    "END_YEAR": 2024,

    # None means ALL real industries, no top limit
    "TOP_N": None,

    # Parent/broad categories to REMOVE because they double-count the smaller industries
    "EXCLUDE_PARENT_INDUSTRIES": [
        "Service-providing",
        "Goods-producing",
        "Total private",
        "Private sector",
        "All industries",
        "Total",
    ],

    # Loading size
    "CHUNKSIZE": 100_000,

    # CSV columns
    "YEAR_COL": "year",
    "INDUSTRY_COL": "industry_name",
    "METRIC_NAME_COL": "metric_name",
    "METRIC_CATEGORY_COL": "metric_category",
    "METRIC_CODE_COL": "metric_code",
    "VALUE_COL": "value",

    # Image settings
    "FIG_WIDTH": 16,
    "FIG_HEIGHT": 9,
    "DPI": 180,

    # Layout: right side is saved for legend + industry total lines
    "PLOT_RIGHT": 0.73,
    "PLOT_TOP": 0.91,
    "PLOT_LEFT": 0.08,
    "PLOT_BOTTOM": 0.09,

    # Right-side industry total text position
    "RIGHT_TEXT_X": 0.765,
    "RIGHT_TEXT_Y": 0.49,
}


def fmt_number(x, pos=None):
    """Show both left and right axis numbers as positive numbers."""
    try:
        return f"{abs(int(x)):,}"
    except Exception:
        return str(x)


def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def is_real_industry_name(name):
    """Remove blank / unknown / code-only / parent industry labels."""
    s = clean_text(name)
    low = s.lower()

    if not s:
        return False
    if low in {"nan", "none", "unknown", "blank", "null", "all", "total", "total private"}:
        return False
    if re.fullmatch(r"naics\s*\d+", low, flags=re.IGNORECASE):
        return False
    if re.fullmatch(r"naics\d+", s, flags=re.IGNORECASE):
        return False
    if re.fullmatch(r"\d+", s):
        return False

    excluded = {x.lower().strip() for x in CFG["EXCLUDE_PARENT_INDUSTRIES"]}
    if low in excluded:
        return False

    return True


def short_industry_name(name):
    """Short names so each total line fits on the right side."""
    s = str(name).strip()
    replacements = {
        "Professional and business services": "Professional/business",
        "Leisure and hospitality": "Leisure/hospitality",
        "Education and health services": "Education/health",
        "Other services (except public administration)": "Other services",
        "Transportation and warehousing": "Transportation/warehousing",
        "Natural resources and mining": "Natural resources/mining",
        "Financial activities": "Financial activities",
        "Retail trade": "Retail trade",
        "Wholesale trade": "Wholesale trade",
        "Manufacturing": "Manufacturing",
        "Construction": "Construction",
        "Information": "Information",
    }
    return replacements.get(s, s)


def sanitize_filename(name):
    name = re.sub(r"[^A-Za-z0-9]+", "_", str(name)).strip("_").lower()
    return name or "chart"


def load_job_gain_loss():
    company_file = CFG["COMPANY_FILE"]
    if not company_file.exists():
        raise FileNotFoundError(f"Cannot find file: {company_file}")

    needed_cols = [
        CFG["YEAR_COL"],
        CFG["INDUSTRY_COL"],
        CFG["METRIC_NAME_COL"],
        CFG["METRIC_CATEGORY_COL"],
        CFG["METRIC_CODE_COL"],
        CFG["VALUE_COL"],
    ]

    print(f"Reading: {company_file}", flush=True)
    print(f"Years: {CFG['START_YEAR']} - {CFG['END_YEAR']}", flush=True)
    print("Using ALL real industries. No TOP 12 limit.", flush=True)
    print(f"Removing parent groups: {', '.join(CFG['EXCLUDE_PARENT_INDUSTRIES'])}", flush=True)

    parts = []
    rows_seen = 0
    rows_kept = 0
    start_time = time.time()

    reader = pd.read_csv(
        company_file,
        usecols=lambda c: c in needed_cols,
        chunksize=CFG["CHUNKSIZE"],
        low_memory=False,
    )

    for chunk_num, chunk in enumerate(reader, start=1):
        rows_seen += len(chunk)

        for col in needed_cols:
            if col not in chunk.columns:
                chunk[col] = ""

        chunk[CFG["YEAR_COL"]] = pd.to_numeric(chunk[CFG["YEAR_COL"]], errors="coerce")
        chunk[CFG["VALUE_COL"]] = pd.to_numeric(chunk[CFG["VALUE_COL"]], errors="coerce").fillna(0)

        chunk = chunk[
            (chunk[CFG["YEAR_COL"]] >= CFG["START_YEAR"]) &
            (chunk[CFG["YEAR_COL"]] <= CFG["END_YEAR"])
        ].copy()

        if chunk.empty:
            continue

        metric_text = (
            chunk[CFG["METRIC_NAME_COL"]].astype(str).str.lower().fillna("") + " " +
            chunk[CFG["METRIC_CATEGORY_COL"]].astype(str).str.lower().fillna("") + " " +
            chunk[CFG["METRIC_CODE_COL"]].astype(str).str.lower().fillna("")
        )

        has_job_word = metric_text.str.contains(
            r"\b(job|jobs|employment|employee|employees|payroll)\b",
            regex=True,
            na=False,
        )
        gain_mask = has_job_word & metric_text.str.contains(
            r"\b(gain|gains|gained|created|creation)\b",
            regex=True,
            na=False,
        )
        loss_mask = has_job_word & metric_text.str.contains(
            r"\b(loss|losses|lost|destroyed|destruction)\b",
            regex=True,
            na=False,
        )

        keep = chunk[gain_mask | loss_mask].copy()
        if keep.empty:
            continue

        keep["direction"] = np.where(gain_mask.loc[keep.index], "gain", "loss")
        keep["industry_name"] = keep[CFG["INDUSTRY_COL"]].astype(str).str.strip()
        keep = keep[keep["industry_name"].apply(is_real_industry_name)].copy()

        if keep.empty:
            continue

        keep["year"] = keep[CFG["YEAR_COL"]].astype(int)
        keep["value"] = keep[CFG["VALUE_COL"]].fillna(0)

        grouped = (
            keep.groupby(["industry_name", "year", "direction"], as_index=False)["value"]
            .sum()
        )
        parts.append(grouped)
        rows_kept += len(keep)

        if chunk_num % 5 == 0:
            elapsed = time.time() - start_time
            print(
                f"Chunk {chunk_num:,} | rows seen {rows_seen:,} | job rows kept {rows_kept:,} | {elapsed:,.1f}s",
                flush=True,
            )

    if not parts:
        raise ValueError("No job gain/loss rows found. Check metric columns and keywords.")

    long_df = pd.concat(parts, ignore_index=True)
    long_df = long_df.groupby(["industry_name", "year", "direction"], as_index=False)["value"].sum()

    wide = (
        long_df.pivot_table(
            index=["industry_name", "year"],
            columns="direction",
            values="value",
            aggfunc="sum",
            fill_value=0,
        )
        .reset_index()
    )

    if "gain" not in wide.columns:
        wide["gain"] = 0
    if "loss" not in wide.columns:
        wide["loss"] = 0

    wide["gain"] = wide["gain"].round().astype(int)
    wide["loss"] = wide["loss"].round().astype(int)
    wide["net"] = wide["gain"] - wide["loss"]
    wide = wide[["industry_name", "year", "gain", "loss", "net"]]

    print("Finished loading.", flush=True)
    print(f"Rows seen: {rows_seen:,}", flush=True)
    print(f"Job gain/loss rows kept: {rows_kept:,}", flush=True)
    print(f"Real industries found after removing parent groups: {wide['industry_name'].nunique():,}", flush=True)

    return wide


def make_chart(data):
    out_dir = CFG["OUTPUT_DIR"]
    chart_dir = out_dir / "charts"
    table_dir = out_dir / "tables"
    chart_dir.mkdir(parents=True, exist_ok=True)
    table_dir.mkdir(parents=True, exist_ok=True)

    start_year = CFG["START_YEAR"]
    end_year = CFG["END_YEAR"]

    totals = data.groupby("industry_name", as_index=False)[["gain", "loss"]].sum()
    totals["net"] = totals["gain"] - totals["loss"]
    totals["total_activity"] = totals["gain"] + totals["loss"]
    totals = totals.sort_values("total_activity", ascending=False)

    # ALL real industries, no top limit
    industries = totals["industry_name"].tolist()
    plot_totals = totals.copy()
    plot_totals["industry_name"] = pd.Categorical(
        plot_totals["industry_name"],
        categories=industries,
        ordered=True,
    )
    plot_totals = plot_totals.sort_values("industry_name")

    plot_data = data[data["industry_name"].isin(industries)].copy()

    # Save CSV tables
    data.to_csv(table_dir / f"all_real_industry_gain_loss_by_year_{start_year}_{end_year}.csv", index=False)
    totals.to_csv(table_dir / f"all_real_industry_totals_{start_year}_{end_year}.csv", index=False)

    years = list(range(start_year, end_year + 1))

    gain_table = (
        plot_data.pivot_table(index="year", columns="industry_name", values="gain", aggfunc="sum", fill_value=0)
        .reindex(index=years, columns=industries, fill_value=0)
    )
    loss_table = (
        plot_data.pivot_table(index="year", columns="industry_name", values="loss", aggfunc="sum", fill_value=0)
        .reindex(index=years, columns=industries, fill_value=0)
    )

    total_gain = int(plot_data["gain"].sum())
    total_loss = int(plot_data["loss"].sum())
    total_net = total_gain - total_loss

    fig, ax = plt.subplots(figsize=(CFG["FIG_WIDTH"], CFG["FIG_HEIGHT"]))

    y_pos = np.arange(len(years))
    colors = plt.get_cmap("tab20").colors

    left_gain = np.zeros(len(years))
    left_loss = np.zeros(len(years))

    for i, industry in enumerate(industries):
        color = colors[i % len(colors)]
        gains = gain_table[industry].to_numpy(dtype=float)
        losses = loss_table[industry].to_numpy(dtype=float)

        ax.barh(y_pos, -losses, left=-left_loss, height=0.78, color=color, alpha=0.85)
        ax.barh(y_pos, gains, left=left_gain, height=0.78, color=color, alpha=0.85, label=industry)

        left_loss += losses
        left_gain += gains

    ax.axvline(0, color="black", linewidth=1.0)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(years)
    ax.invert_yaxis()
    ax.set_ylabel("Year")
    ax.set_xlabel("Job loss on left   |   Job gain on right")
    ax.xaxis.set_major_formatter(FuncFormatter(fmt_number))
    ax.grid(axis="x", linestyle="--", alpha=0.35)

    max_value = max(float(left_gain.max()), float(left_loss.max()), 1)
    ax.set_xlim(-max_value * 1.22, max_value * 1.22)

    # Good old 2-line title, but NO TOP 12.
    title_text = (
        f"{start_year}-{end_year} ALL Real Industries: Job Loss vs Job Gain by Year\n"
        f"All real-industry total: Gain {total_gain:,} | Loss {total_loss:,} | Net {total_net:+,}"
    )
    fig.suptitle(title_text, fontsize=12, y=0.975)

    ax.legend(
        title="Industry",
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        frameon=True,
        fontsize=8.5,
        title_fontsize=10,
    )

    # Right-side industry totals: one industry per line.
    right_lines = [f"Industry totals ({start_year}-{end_year})"]
    for _, row in plot_totals.iterrows():
        industry = short_industry_name(row["industry_name"])
        gain = int(row["gain"])
        loss = int(row["loss"])
        net = gain - loss
        right_lines.append(f"{industry} total: Gain {gain:,} | Loss {loss:,} | Net {net:+,}")

    right_text = "\n".join(right_lines)

    # Auto font size: helps if more than 12 industries exist.
    line_count = len(right_lines)
    if line_count <= 13:
        right_font = 7.5
    elif line_count <= 18:
        right_font = 6.5
    else:
        right_font = 5.5

    fig.text(
        CFG["RIGHT_TEXT_X"],
        CFG["RIGHT_TEXT_Y"],
        right_text,
        ha="left",
        va="top",
        fontsize=right_font,
        family="monospace",
        linespacing=1.25,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="lightgray", alpha=0.95),
    )

    fig.subplots_adjust(
        top=CFG["PLOT_TOP"],
        right=CFG["PLOT_RIGHT"],
        left=CFG["PLOT_LEFT"],
        bottom=CFG["PLOT_BOTTOM"],
    )

    chart_path = chart_dir / f"company_job_gain_loss_by_industry_{start_year}_{end_year}_ALL_REAL_INDUSTRIES_RIGHT_TOTAL_LINES.png"
    fig.savefig(chart_path, dpi=CFG["DPI"], bbox_inches="tight")
    plt.close(fig)

    # Save the right-side total text as a .txt file, so you can check it easily.
    text_path = table_dir / f"all_real_industry_total_lines_{start_year}_{end_year}.txt"
    with open(text_path, "w", encoding="utf-8") as f:
        f.write(title_text + "\n\n" + right_text + "\n")

    print(f"Saved chart: {chart_path}", flush=True)
    print(f"Saved tables: {table_dir}", flush=True)
    print(f"Saved text preview: {text_path}", flush=True)


def main():
    data = load_job_gain_loss()
    make_chart(data)
    print("Done.", flush=True)
    print(f"Open output folder: {CFG['OUTPUT_DIR']}", flush=True)


if __name__ == "__main__":
    main()


# Startup Death 1

In [ ]:
from pathlib import Path
import re
import textwrap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# ============================================================
# SETTINGS OBJECT
# Change only this part later
# ============================================================

CFG = {
    # Same Company / business startup file path
    "COMPANY_FILE": Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv",

    # Output folder
    "OUTPUT_DIR": Path.home() / "Downloads" / "pre_formula_startup_death_all_real_industries_charts",

    "CHUNKSIZE": 100_000,

    # Change year range here
    # Default matches the job chart style you were using before.
    "START_YEAR": 2010,
    "END_YEAR": 2024,

    # None = ALL industries, no Top 12
    "TOP_N": None,

    # Columns needed from the company file
    "COMPANY_COLS": [
        "year",
        "industry_name",
        "metric_name",
        "value",
    ],

    # Exclude rate / percent rows
    "RATE_WORDS": "rate|percent|percentage|pct|share|ratio",

    # Exclude job rows
    # This chart is startup/death only, NOT job gain/loss
    "JOB_WORDS": (
        "job|jobs|employment|hire|hires|hiring|"
        "layoff|layoffs|separation|separations"
    ),

    # Startup / new business metric words
    "STARTUP_WORDS": (
        "startup|start-up|birth|births|formation|formations|"
        "entry|entries|new business|new firm|new company|"
        "opening|openings"
    ),

    # Company death / closed business metric words
    "DEATH_WORDS": (
        "death|deaths|closed|closure|closures|closing|closings|"
        "exit|exits|failure|failures"
    ),

    # Remove parent / aggregate industry rows so chart is real industries only.
    # This is why Service-providing and Goods-producing will NOT show.
    "EXCLUDE_PARENT_INDUSTRIES": [
        "All industries",
        "All Industries",
        "Total",
        "Total private",
        "Total nonfarm",
        "Private",
        "Private service-providing",
        "Service-providing",
        "Goods-producing",
        "Unclassified",
        "Unknown",
        "Not classified",
    ],

    # If your file has industry_name like NAICS11, NAICS23, etc., keep them by default.
    # Change to True if you only want readable industry names and want to remove code-only names.
    "EXCLUDE_NAICS_CODE_ONLY_NAMES": False,

    # Short names for the right-side totals box
    "SHORT_NAME_MAP": {
        "Professional and business services": "Professional/business",
        "Leisure and hospitality": "Leisure/hospitality",
        "Education and health services": "Education/health",
        "Financial activities": "Financial activities",
        "Other services (except public administration)": "Other services",
        "Transportation and warehousing": "Transportation/warehousing",
        "Natural resources and mining": "Natural resources/mining",
        "Mining, quarrying, and oil and gas extraction": "Mining/oil/gas",
        "Agriculture, forestry, fishing and hunting": "Agriculture/forestry",
        "Administrative and support and waste management and remediation services": "Admin/support/waste",
        "Professional, scientific, and technical services": "Professional/scientific",
        "Management of companies and enterprises": "Management companies",
        "Real estate and rental and leasing": "Real estate/rental",
        "Health care and social assistance": "Health care/social",
        "Accommodation and food services": "Accommodation/food",
        "Arts, entertainment, and recreation": "Arts/entertainment",
    },

    "FIGSIZE": (18, 10),
    "DPI": 300,
    "COLOR_MAP": "tab20",

    "SAVE_FILE_NAME": "startup_death_ALL_real_industries_{start}_{end}.png",
    "SAVE_CHART_DATA_CSV": "startup_death_ALL_real_industries_by_year_{start}_{end}.csv",
    "SAVE_TOTALS_CSV": "startup_death_ALL_real_industries_totals_{start}_{end}.csv",

    # Right-side total box text size.
    # Use smaller number if you have many industries.
    "RIGHT_TOTAL_FONT_SIZE": 6,
    "LEGEND_FONT_SIZE": 8,

    # How many rows to print in the check table
    "CHECK_TABLE_ROWS": 50,
}


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def fmt_abs_number(x, pos):
    """Show negative left-axis values as positive numbers."""
    try:
        return f"{abs(int(x)):,}"
    except Exception:
        return str(x)


def clean_industry_name(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def short_name(name):
    return CFG["SHORT_NAME_MAP"].get(name, name)


def make_safe_filename(text):
    text = str(text).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = text.strip("_")
    return text[:120] if text else "industry"


def build_totals_text(industry_totals):
    lines = [f"Industry totals, {START_YEAR}-{END_YEAR}"]

    for industry, row in industry_totals.iterrows():
        startup = int(round(row["startup_count"]))
        death = int(round(row["death_count"]))
        net = startup - death
        label = short_name(industry)
        lines.append(
            f"{label} total: Startup {startup:,} | Death {death:,} | Net {net:+,}"
        )

    return "\n".join(lines)


# ============================================================
# VARIABLES FROM SETTINGS OBJECT
# ============================================================

COMPANY_FILE = CFG["COMPANY_FILE"]
OUTPUT_DIR = CFG["OUTPUT_DIR"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNKSIZE = CFG["CHUNKSIZE"]
START_YEAR = CFG["START_YEAR"]
END_YEAR = CFG["END_YEAR"]
TOP_N = CFG["TOP_N"]
COMPANY_COLS = CFG["COMPANY_COLS"]


# ============================================================
# CHECK FILE
# ============================================================

print("Checking Company file...")

if not COMPANY_FILE.exists():
    raise FileNotFoundError(f"Missing Company file: {COMPANY_FILE}")

print("Finished checking file.")
print("Company file:", COMPANY_FILE)

# Check columns before loading chunks
file_columns = pd.read_csv(COMPANY_FILE, nrows=0).columns.tolist()
missing_cols = [c for c in COMPANY_COLS if c not in file_columns]

if missing_cols:
    raise ValueError(
        "Missing required columns in file: " + ", ".join(missing_cols)
    )


# ============================================================
# LOAD RAW COMPANY STARTUP / DEATH DATA
# ============================================================

print(f"\nLoading raw Company startup/death data from {START_YEAR} to {END_YEAR}...")

parts = []
chunk_count = 0
matched_startup_metric_names = set()
matched_death_metric_names = set()

exclude_parent_lower = {
    str(x).strip().lower()
    for x in CFG["EXCLUDE_PARENT_INDUSTRIES"]
}

for chunk in pd.read_csv(
    COMPANY_FILE,
    usecols=COMPANY_COLS,
    chunksize=CHUNKSIZE,
    low_memory=False,
):
    chunk_count += 1
    print(f"Loading chunk {chunk_count}... rows: {len(chunk):,}")

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")
    chunk["industry_name"] = chunk["industry_name"].apply(clean_industry_name)

    chunk = chunk.dropna(subset=["year", "industry_name", "metric_name", "value"])
    chunk["year"] = chunk["year"].astype(int)

    chunk = chunk[
        (chunk["year"] >= START_YEAR) &
        (chunk["year"] <= END_YEAR)
    ].copy()

    if chunk.empty:
        continue

    # Remove parent / aggregate industries
    industry_lower = chunk["industry_name"].astype(str).str.strip().str.lower()
    real_industry_mask = ~industry_lower.isin(exclude_parent_lower)

    if CFG["EXCLUDE_NAICS_CODE_ONLY_NAMES"]:
        real_industry_mask &= ~chunk["industry_name"].astype(str).str.match(
            r"^NAICS\d+", na=False, case=False
        )

    chunk = chunk[real_industry_mask].copy()

    if chunk.empty:
        continue

    metric = chunk["metric_name"].astype(str).str.lower()

    # Exclude rate / percent / share rows
    not_rate_mask = ~metric.str.contains(
        CFG["RATE_WORDS"],
        na=False,
        regex=True,
    )

    # Exclude job rows
    not_job_mask = ~metric.str.contains(
        CFG["JOB_WORDS"],
        na=False,
        regex=True,
    )

    # STARTUP / NEW BUSINESS / NEW COMPANY
    startup_mask = metric.str.contains(
        CFG["STARTUP_WORDS"],
        na=False,
        regex=True,
    ) & not_rate_mask & not_job_mask

    # COMPANY DEATH / CLOSED BUSINESS
    death_mask = metric.str.contains(
        CFG["DEATH_WORDS"],
        na=False,
        regex=True,
    ) & not_rate_mask & not_job_mask

    matched_startup_metric_names.update(
        chunk.loc[startup_mask, "metric_name"].dropna().unique()
    )

    matched_death_metric_names.update(
        chunk.loc[death_mask, "metric_name"].dropna().unique()
    )

    chunk["startup_count"] = np.where(startup_mask, chunk["value"], 0)
    chunk["death_count"] = np.where(death_mask, chunk["value"], 0)

    part = (
        chunk
        .groupby(["year", "industry_name"], dropna=False)[["startup_count", "death_count"]]
        .sum()
        .reset_index()
    )

    parts.append(part)

print("Finished loading data.")


# ============================================================
# COMBINE DATA
# ============================================================

if not parts:
    raise ValueError(f"No Company startup/death data found from {START_YEAR} to {END_YEAR}.")

chart_data = pd.concat(parts, ignore_index=True)

chart_data = (
    chart_data
    .groupby(["year", "industry_name"], dropna=False)[["startup_count", "death_count"]]
    .sum()
    .reset_index()
)

chart_data["company_total"] = chart_data["startup_count"] + chart_data["death_count"]
chart_data = chart_data[chart_data["company_total"] > 0].copy()

if chart_data.empty:
    raise ValueError(
        f"Data loaded, but no startup/death rows matched from {START_YEAR} to {END_YEAR}."
    )

print("\nMetric names used for startup_count:")
print(sorted(matched_startup_metric_names))

print("\nMetric names used for death_count:")
print(sorted(matched_death_metric_names))


# ============================================================
# OPTIONAL TOP N
# Default is None = ALL real industries, no Top 12
# ============================================================

if TOP_N is not None:
    top_industries = (
        chart_data.groupby("industry_name")["company_total"]
        .sum()
        .sort_values(ascending=False)
        .head(TOP_N)
        .index
    )

    chart_data["industry_name"] = np.where(
        chart_data["industry_name"].isin(top_industries),
        chart_data["industry_name"],
        "Other",
    )

    chart_data = (
        chart_data
        .groupby(["year", "industry_name"], dropna=False)[["startup_count", "death_count"]]
        .sum()
        .reset_index()
    )


# ============================================================
# INDUSTRY TOTALS
# ============================================================

industry_totals = (
    chart_data
    .groupby("industry_name", dropna=False)[["startup_count", "death_count"]]
    .sum()
)
industry_totals["net"] = industry_totals["startup_count"] - industry_totals["death_count"]
industry_totals["total"] = industry_totals["startup_count"] + industry_totals["death_count"]
industry_totals = industry_totals.sort_values("total", ascending=False)

all_industries = industry_totals.index.tolist()

grand_startup = int(round(industry_totals["startup_count"].sum()))
grand_death = int(round(industry_totals["death_count"].sum()))
grand_net = grand_startup - grand_death


# ============================================================
# PIVOT FOR STACKED BAR
# ============================================================

startup_pivot = (
    chart_data
    .pivot_table(
        index="year",
        columns="industry_name",
        values="startup_count",
        aggfunc="sum",
        fill_value=0,
    )
    .sort_index()
)

death_pivot = (
    chart_data
    .pivot_table(
        index="year",
        columns="industry_name",
        values="death_count",
        aggfunc="sum",
        fill_value=0,
    )
    .sort_index()
)

# Same industry order for chart and legend: largest total first
startup_pivot = startup_pivot.reindex(columns=all_industries, fill_value=0)
death_pivot = death_pivot.reindex(columns=all_industries, fill_value=0)

print("\nYears used:")
print(startup_pivot.index.tolist())

print("\nIndustries used:")
print(all_industries)


# ============================================================
# SAVE CSV TABLES
# ============================================================

chart_csv_path = OUTPUT_DIR / CFG["SAVE_CHART_DATA_CSV"].format(
    start=START_YEAR,
    end=END_YEAR,
)

totals_csv_path = OUTPUT_DIR / CFG["SAVE_TOTALS_CSV"].format(
    start=START_YEAR,
    end=END_YEAR,
)

chart_data.sort_values(["year", "industry_name"]).to_csv(chart_csv_path, index=False)
industry_totals.to_csv(totals_csv_path)


# ============================================================
# PLOT DIVERGING STACKED HORIZONTAL BAR CHART
# Y = YEAR
# COLORS = INDUSTRY
# LEFT = COMPANY DEATH / CLOSED BUSINESS
# RIGHT = STARTUP / NEW BUSINESS
# ALL REAL INDUSTRIES, NO TOP 12 TITLE
# ============================================================

fig, ax = plt.subplots(figsize=CFG["FIGSIZE"])

# Leave a large right side for legend + industry total lines
fig.subplots_adjust(left=0.08, right=0.68, top=0.90, bottom=0.11)

years = startup_pivot.index.astype(str)
y_pos = np.arange(len(years))

# One color per industry
cmap = plt.colormaps.get_cmap(CFG["COLOR_MAP"])
if len(all_industries) <= 1:
    colors = [cmap(0.5)]
else:
    colors = [cmap(i / max(len(all_industries) - 1, 1)) for i in range(len(all_industries))]

# RIGHT SIDE = startup / new business
right_base = np.zeros(len(startup_pivot))

for i, industry in enumerate(all_industries):
    values = startup_pivot[industry].values

    ax.barh(
        y_pos,
        values,
        left=right_base,
        label=short_name(industry),
        color=colors[i],
        edgecolor="none",
    )

    right_base += values

# LEFT SIDE = death / closed business
left_base = np.zeros(len(death_pivot))

for i, industry in enumerate(all_industries):
    values = death_pivot[industry].values

    ax.barh(
        y_pos,
        -values,
        left=-left_base,
        color=colors[i],
        edgecolor="none",
    )

    left_base += values

ax.axvline(0, color="black", linewidth=1)

ax.set_yticks(y_pos)
ax.set_yticklabels(years)

ax.set_title(
    f"{START_YEAR}-{END_YEAR} ALL Real Industries: Company Death vs Startup by Year\n"
    f"All plotted-industry total: Startup {grand_startup:,} | Death {grand_death:,} | Net {grand_net:+,}",
    fontsize=13,
)

ax.set_xlabel("Company death / closed business on left   |   Startup / new business on right")
ax.set_ylabel("Year")

ax.xaxis.set_major_formatter(FuncFormatter(fmt_abs_number))
ax.grid(axis="x", linestyle="--", alpha=0.4)

# Legend on right
legend_cols = 1 if len(all_industries) <= 20 else 2
ax.legend(
    title="Industry",
    bbox_to_anchor=(1.02, 1.00),
    loc="upper left",
    fontsize=CFG["LEGEND_FONT_SIZE"],
    title_fontsize=CFG["LEGEND_FONT_SIZE"] + 1,
    ncol=legend_cols,
    frameon=True,
)

# Right-side industry total lines
# This is the part that looks like:
# Industry total: Startup ? | Death ? | Net +?
totals_text = build_totals_text(industry_totals)

fig.text(
    0.70,
    0.48,
    totals_text,
    ha="left",
    va="top",
    fontsize=CFG["RIGHT_TOTAL_FONT_SIZE"],
    family="monospace",
    bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="lightgray", alpha=0.95),
)

save_path = OUTPUT_DIR / CFG["SAVE_FILE_NAME"].format(
    start=START_YEAR,
    end=END_YEAR,
)

fig.savefig(
    save_path,
    dpi=CFG["DPI"],
    bbox_inches="tight",
)

plt.close(fig)


# ============================================================
# PRINT SMALL CHECK TABLE
# ============================================================

check_table = (
    chart_data
    .groupby(["year", "industry_name"], dropna=False)[["startup_count", "death_count"]]
    .sum()
    .reset_index()
    .sort_values(["year", "industry_name"])
)

print("\nPre-formula check table:")
print(check_table.head(CFG["CHECK_TABLE_ROWS"]))

print("\nIndustry totals table:")
print(industry_totals[["startup_count", "death_count", "net"]].head(CFG["CHECK_TABLE_ROWS"]))


# ============================================================
# DONE
# ============================================================

print("\nALL DONE.")
print("Saved chart here:")
print(save_path)
print("\nSaved by-year CSV here:")
print(chart_csv_path)
print("\nSaved industry totals CSV here:")
print(totals_csv_path)


# Read degree with 12 top low to phd

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import textwrap
import time

# ============================================================
# TOP 12 POPULAR DEGREE FIELD GROUPS
# SPLIT BY DEGREE LEVEL: LITTLE LEVEL -> GRADUATE LEVEL
#
# OUTPUT:
# 1. Top 12 table CSV
# 2. Top 12 table Excel
# 3. All field groups CSV
# 4. Stacked chart PNG
# 5. Grouped chart PNG
# 6. Total horizontal bar chart PNG
#
# MEMORY OPTIMIZATION:
# - Reads ONLY needed columns
# - Uses chunksize
# - Groups each chunk immediately
# - Does NOT load full file into memory
# ============================================================

# ============================================================
# SETTINGS
# ============================================================

BASE_DIR = Path.home() / "Downloads/MyProject_1"

DEGREE_FILE = (
    BASE_DIR
    / "degree_raw_csv_with_year_range_in_filename"
    / "degree_RAW_search_1970-2025_usable_2008-2024.csv"
)

OUTPUT_DIR = Path.home() / "Downloads" / "top12_degree_field_group_by_degree_level_2010_2024"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR = 2010
END_YEAR = 2024

TOP_N = 12
CHUNKSIZE = 100_000

YEAR_COL = "year"
FIELD_GROUP_COL = "field_group"
AWARD_LEVEL_COL = "award_level_name"
VALUE_COL = "degree_count"

NEEDED_COLUMNS = [
    YEAR_COL,
    FIELD_GROUP_COL,
    AWARD_LEVEL_COL,
    VALUE_COL,
]

# ============================================================
# DEGREE LEVEL ORDER
# little -> graduate
# ============================================================

DEGREE_LEVEL_ORDER = [
    "Certificate / Below Associate",
    "Associate",
    "Bachelor",
    "Graduate Certificate",
    "Master",
    "Doctor / Professional",
    "Other / Unmapped",
]

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def fmt_number(x, pos):
    try:
        return f"{int(x):,}"
    except Exception:
        return str(x)


def clean_text_col(s):
    return (
        s.astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "NaN": pd.NA,
            "None": pd.NA,
            "UNKNOWN": pd.NA,
            "Unknown": pd.NA,
            "unknown": pd.NA,
        })
    )


def degree_level_bucket(value):
    """
    Converts raw award_level_name into simple ordered degree levels.
    """

    if pd.isna(value):
        return "Other / Unmapped"

    s = str(value).strip().lower()

    if s == "" or s in ["nan", "none", "unknown"]:
        return "Other / Unmapped"

    if (
        "postbaccalaureate" in s
        or "post-baccalaureate" in s
        or "post bachelor" in s
        or "post-master" in s
        or "post master" in s
    ):
        return "Graduate Certificate"

    if "doctor" in s or "professional practice" in s:
        return "Doctor / Professional"

    if "master" in s:
        return "Master"

    if "bachelor" in s:
        return "Bachelor"

    if "associate" in s:
        return "Associate"

    if "certificate" in s or "award of" in s:
        return "Certificate / Below Associate"

    return "Other / Unmapped"


def print_table(title, df):
    print("\n" + "=" * 120)
    print(title)
    print("=" * 120)

    if df.empty:
        print("No data found.")
    else:
        print(df.to_string(index=False))


def format_table_for_print(df):
    out = df.copy()

    for col in out.columns:
        if col not in ["Rank", FIELD_GROUP_COL]:
            out[col] = out[col].apply(
                lambda x: f"{int(x):,}" if pd.notna(x) else "0"
            )

    return out


def wrap_labels(labels, width=36):
    return [
        "\n".join(textwrap.wrap(str(label), width=width))
        for label in labels
    ]


# ============================================================
# CHECK FILE
# ============================================================

print("BASE_DIR:", BASE_DIR)
print("DEGREE_FILE:", DEGREE_FILE)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("YEAR RANGE:", START_YEAR, "-", END_YEAR)
print("TOP_N:", TOP_N)
print("CHUNKSIZE:", CHUNKSIZE)

if not DEGREE_FILE.exists():
    raise FileNotFoundError(f"Missing file: {DEGREE_FILE}")

header = pd.read_csv(DEGREE_FILE, nrows=0)
missing_cols = [c for c in NEEDED_COLUMNS if c not in header.columns]

if missing_cols:
    print("\nColumns found in file:")
    print(list(header.columns))
    raise ValueError(f"Missing needed columns: {missing_cols}")

# ============================================================
# LOAD FILE IN CHUNKS
# ============================================================

parts = []
total_raw_rows_seen = 0
total_rows_used = 0
start_time = time.time()

print("\nLoading file with memory optimization...")

for chunk_number, chunk in enumerate(
    pd.read_csv(
        DEGREE_FILE,
        usecols=NEEDED_COLUMNS,
        chunksize=CHUNKSIZE,
        low_memory=False
    ),
    start=1
):
    total_raw_rows_seen += len(chunk)

    chunk[YEAR_COL] = pd.to_numeric(chunk[YEAR_COL], errors="coerce")
    chunk[VALUE_COL] = pd.to_numeric(chunk[VALUE_COL], errors="coerce").fillna(0)

    chunk = chunk[
        (chunk[YEAR_COL] >= START_YEAR)
        & (chunk[YEAR_COL] <= END_YEAR)
    ]

    if chunk.empty:
        elapsed = time.time() - start_time
        print(
            f"Chunk {chunk_number} done | raw rows seen: {total_raw_rows_seen:,} | "
            f"rows used: {total_rows_used:,} | elapsed: {elapsed:.2f}s"
        )
        continue

    chunk[FIELD_GROUP_COL] = clean_text_col(chunk[FIELD_GROUP_COL])
    chunk[AWARD_LEVEL_COL] = clean_text_col(chunk[AWARD_LEVEL_COL])

    chunk = chunk.dropna(subset=[FIELD_GROUP_COL, AWARD_LEVEL_COL])

    if chunk.empty:
        continue

    chunk["degree_level_group"] = chunk[AWARD_LEVEL_COL].apply(degree_level_bucket)

    chunk[FIELD_GROUP_COL] = chunk[FIELD_GROUP_COL].astype("category")
    chunk["degree_level_group"] = chunk["degree_level_group"].astype("category")

    grouped = (
        chunk.groupby(
            [FIELD_GROUP_COL, "degree_level_group"],
            as_index=False,
            observed=True
        )[VALUE_COL]
        .sum()
    )

    parts.append(grouped)
    total_rows_used += len(chunk)

    elapsed = time.time() - start_time
    print(
        f"Chunk {chunk_number} done | raw rows seen: {total_raw_rows_seen:,} | "
        f"rows used: {total_rows_used:,} | elapsed: {elapsed:.2f}s"
    )

# ============================================================
# IF NO DATA
# ============================================================

if not parts:
    print("\nNo data found.")
    print("DONE.")

else:
    # ========================================================
    # COMBINE SMALL GROUPED PARTS
    # ========================================================

    df = pd.concat(parts, ignore_index=True)

    df = (
        df.groupby(
            [FIELD_GROUP_COL, "degree_level_group"],
            as_index=False,
            observed=True
        )[VALUE_COL]
        .sum()
    )

    elapsed = time.time() - start_time

    print("\nFinished loading.")
    print("Raw rows seen:", f"{total_raw_rows_seen:,}")
    print("Rows used after year/text filter:", f"{total_rows_used:,}")
    print("Finished in:", f"{elapsed:.2f} seconds")

    # ========================================================
    # FIND TOP 12 FIELD GROUPS BY TOTAL COUNT
    # ========================================================

    field_totals = (
        df.groupby(FIELD_GROUP_COL, as_index=False, observed=True)[VALUE_COL]
        .sum()
        .sort_values(VALUE_COL, ascending=False)
    )

    top_fields = field_totals.head(TOP_N)[FIELD_GROUP_COL].tolist()

    print_table(
        title=f"TOP {TOP_N} FIELD GROUP TOTALS {START_YEAR}-{END_YEAR}",
        df=field_totals.head(TOP_N)
    )

    # ========================================================
    # MAKE TOP 12 TABLE BY DEGREE LEVEL
    # ========================================================

    top_df = df[df[FIELD_GROUP_COL].isin(top_fields)].copy()

    pivot = top_df.pivot_table(
        index=FIELD_GROUP_COL,
        columns="degree_level_group",
        values=VALUE_COL,
        aggfunc="sum",
        fill_value=0
    )

    for level in DEGREE_LEVEL_ORDER:
        if level not in pivot.columns:
            pivot[level] = 0

    pivot = pivot[DEGREE_LEVEL_ORDER]

    if pivot["Other / Unmapped"].sum() == 0:
        pivot = pivot.drop(columns=["Other / Unmapped"])

    total_col_name = f"Total {START_YEAR}-{END_YEAR}"
    pivot[total_col_name] = pivot.sum(axis=1)

    pivot = pivot.loc[top_fields]

    final_table = pivot.reset_index()
    final_table.insert(0, "Rank", range(1, len(final_table) + 1))

    count_cols = [
        c for c in final_table.columns
        if c not in ["Rank", FIELD_GROUP_COL]
    ]

    for col in count_cols:
        final_table[col] = final_table[col].round(0).astype("int64")

    print_table(
        title=f"FINAL TABLE: TOP {TOP_N} FIELD GROUPS BY DEGREE LEVEL {START_YEAR}-{END_YEAR}",
        df=format_table_for_print(final_table)
    )

    # ========================================================
    # SAVE TABLE FILES
    # ========================================================

    top12_csv = OUTPUT_DIR / f"top_{TOP_N}_field_group_by_degree_level_{START_YEAR}_{END_YEAR}.csv"
    top12_xlsx = OUTPUT_DIR / f"top_{TOP_N}_field_group_by_degree_level_{START_YEAR}_{END_YEAR}.xlsx"

    final_table.to_csv(top12_csv, index=False)

    try:
        final_table.to_excel(top12_xlsx, index=False)
        saved_xlsx = True
    except Exception as e:
        saved_xlsx = False
        print("\nCould not save Excel file.")
        print("Reason:", e)

    all_pivot = df.pivot_table(
        index=FIELD_GROUP_COL,
        columns="degree_level_group",
        values=VALUE_COL,
        aggfunc="sum",
        fill_value=0
    )

    for level in DEGREE_LEVEL_ORDER:
        if level not in all_pivot.columns:
            all_pivot[level] = 0

    all_pivot = all_pivot[DEGREE_LEVEL_ORDER]

    if all_pivot["Other / Unmapped"].sum() == 0:
        all_pivot = all_pivot.drop(columns=["Other / Unmapped"])

    all_pivot[total_col_name] = all_pivot.sum(axis=1)
    all_pivot = all_pivot.sort_values(total_col_name, ascending=False)

    all_table = all_pivot.reset_index()
    all_table.insert(0, "Rank", range(1, len(all_table) + 1))

    for col in [c for c in all_table.columns if c not in ["Rank", FIELD_GROUP_COL]]:
        all_table[col] = all_table[col].round(0).astype("int64")

    all_csv = OUTPUT_DIR / f"all_field_groups_by_degree_level_{START_YEAR}_{END_YEAR}.csv"
    all_table.to_csv(all_csv, index=False)

    totals_csv = OUTPUT_DIR / f"field_group_total_counts_{START_YEAR}_{END_YEAR}.csv"
    field_totals.to_csv(totals_csv, index=False)

    # ========================================================
    # CHART DATA
    # ========================================================

    chart_df = final_table.copy()

    degree_cols = [
        c for c in DEGREE_LEVEL_ORDER
        if c in chart_df.columns and c != "Other / Unmapped"
    ]

    if "Other / Unmapped" in chart_df.columns:
        degree_cols.append("Other / Unmapped")

    chart_index_labels = chart_df[FIELD_GROUP_COL].tolist()

    chart_pivot = chart_df.set_index(FIELD_GROUP_COL)[degree_cols]

    # ========================================================
    # CHART 1: STACKED HORIZONTAL BAR
    # Best chart for final project
    # ========================================================

    stacked_png = OUTPUT_DIR / f"top_{TOP_N}_degree_level_STACKED_BAR_{START_YEAR}_{END_YEAR}.png"

    ax = chart_pivot.sort_values(
        by=degree_cols,
        ascending=True
    ).plot(
        kind="barh",
        stacked=True,
        figsize=(18, 10),
        width=0.75,
        colormap="tab20"
    )

    ax.set_title(
        f"Top {TOP_N} Degree Field Groups by Degree Level ({START_YEAR}-{END_YEAR})",
        fontsize=18,
        pad=15
    )

    ax.set_xlabel("Degree Count", fontsize=12)
    ax.set_ylabel("Field Group", fontsize=12)
    ax.xaxis.set_major_formatter(FuncFormatter(fmt_number))

    ax.set_yticklabels(wrap_labels(chart_pivot.sort_values(by=degree_cols, ascending=True).index, width=42))

    ax.legend(
        title="Degree Level",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=9,
        title_fontsize=10,
        frameon=True
    )

    ax.grid(axis="x", alpha=0.25)

    plt.tight_layout()
    plt.savefig(stacked_png, dpi=220, bbox_inches="tight")
    plt.close()

    print("Saved stacked chart PNG:", stacked_png)

    # ========================================================
    # CHART 2: GROUPED BAR
    # Shows each degree level side-by-side
    # ========================================================

    grouped_png = OUTPUT_DIR / f"top_{TOP_N}_degree_level_GROUPED_BAR_{START_YEAR}_{END_YEAR}.png"

    ax = chart_pivot.plot(
        kind="bar",
        stacked=False,
        figsize=(22, 10),
        width=0.82,
        colormap="tab20"
    )

    ax.set_title(
        f"Top {TOP_N} Degree Field Groups by Degree Level - Grouped ({START_YEAR}-{END_YEAR})",
        fontsize=18,
        pad=15
    )

    ax.set_xlabel("Field Group", fontsize=12)
    ax.set_ylabel("Degree Count", fontsize=12)
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_number))

    ax.set_xticklabels(wrap_labels(chart_pivot.index, width=28), rotation=45, ha="right")

    ax.legend(
        title="Degree Level",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=9,
        title_fontsize=10,
        frameon=True
    )

    ax.grid(axis="y", alpha=0.25)

    plt.tight_layout()
    plt.savefig(grouped_png, dpi=220, bbox_inches="tight")
    plt.close()

    print("Saved grouped chart PNG:", grouped_png)

    # ========================================================
    # CHART 3: TOTAL FIELD GROUP BAR
    # Same idea as your old total bar chart
    # ========================================================

    total_bar_png = OUTPUT_DIR / f"top_{TOP_N}_field_group_TOTAL_BAR_{START_YEAR}_{END_YEAR}.png"

    total_series = (
        final_table
        .set_index(FIELD_GROUP_COL)[total_col_name]
        .sort_values(ascending=True)
    )

    ax = total_series.plot(
        kind="barh",
        figsize=(16, 9)
    )

    ax.set_title(
        f"Top {TOP_N} Total Degree Count by Field Group ({START_YEAR}-{END_YEAR})",
        fontsize=18,
        pad=15
    )

    ax.set_xlabel("Degree Count", fontsize=12)
    ax.set_ylabel("Field Group", fontsize=12)
    ax.xaxis.set_major_formatter(FuncFormatter(fmt_number))

    ax.set_yticklabels(wrap_labels(total_series.index, width=42))

    ax.grid(axis="x", alpha=0.25)

    for i, value in enumerate(total_series.values):
        ax.text(
            value,
            i,
            f" {value:,.0f}",
            va="center",
            fontsize=9
        )

    plt.tight_layout()
    plt.savefig(total_bar_png, dpi=220, bbox_inches="tight")
    plt.close()

    print("Saved total bar chart PNG:", total_bar_png)

    # ========================================================
    # DONE
    # ========================================================

    print("\n" + "=" * 120)
    print("SAVED FILES")
    print("=" * 120)

    print("Top 12 CSV:", top12_csv)

    if saved_xlsx:
        print("Top 12 Excel:", top12_xlsx)

    print("All field groups CSV:", all_csv)
    print("Field group total counts CSV:", totals_csv)

    print("Stacked chart PNG:", stacked_png)
    print("Grouped chart PNG:", grouped_png)
    print("Total bar chart PNG:", total_bar_png)

    print("\nDONE.")
    print("All files saved in:")
    print(OUTPUT_DIR)